# BOREAS — borehole array storage with a conducting formation

Latent-heat storage in repurposed oil-and-gas wells. **A new series, not a new
version of P2H2P/THUMS** — that model is frozen at v0.16 and still runs; its
notebook is untouched.

*v1.0 · built 2026-10-01 02:19 BRT*

---

## What changed, and why the two changes are one change

### 1. The geometry

THUMS put $n$ **hairpins** in the hole — U-tubes, down one leg and back up the
other, both exchanging with the PCM. That turns out to be self-contradictory,
and the contradiction is quantitative.

Charge water enters at $T_m + 10$ K and glides 55 K down, so the **return leg
runs about 45 K below the melting point**. Graded by depth it would refreeze
whatever the down leg had just melted. So THUMS graded the cascade along the
**tube path** instead — which works, but then two different melting points
have to coexist at the same depth, centimetres apart, in contiguous PCM with
no wall between them. Conduction across that gap is

| hairpins | leg-to-leg short circuit | % of a ~120 kW duty |
|---|---|---|
| 1 | 43 kW | 36 % |
| 2 | 62 kW | 52 % |
| 3 | **92 kW** | **77 %** |

a short circuit straight across the cascade the model exists to represent —
and a term THUMS never had.

**BOREAS separates the two jobs.** `n_ft` fin tubes all run *downward* and all
exchange; one **insulated downcomer** returns the whole flow without
exchanging. They meet in a manifold at the bottom of the same hole — no
lateral, no new drilling, the same class of object as the U-bend it replaces.

### 2. The formation

THUMS hypothesis **H6** made the borehole wall adiabatic, so $\eta_{storage}$
was identically 1 — not approximately, *exactly*, by construction. BOREAS
conducts to the rock.

### Why the second needed the first

With a hairpin, **every depth is occupied by two tube segments**, so one rock
temperature would have had two PCM segments claiming it. Only when the return
stops exchanging does $z$ become a depth. In BOREAS `L_tube = L_well`, not
twice it, and the map is one-to-one.

Three things follow for free:

- the cascade runs **hot at the bottom**, which *aligns* it with the
  geothermal gradient instead of fighting it
- the PCM becomes **pourable in layers** (path-graded PCM was never
  manufacturable)
- the downcomer's leak goes **into** the store on charge — energy misplaced,
  not energy lost

---
## 1. Setup

`CoolProp` is not in a stock Colab image. The model is written to disk by the
next cell, so this notebook needs nothing from the repository.

In [ ]:
!pip install -q CoolProp

In [ ]:
%%writefile boreas.py
"""BOREAS -- BOREhole Array Storage.

Latent-heat storage in repurposed wellbores, with a DOWNCOMER geometry and a
conducting formation. The live model, v1.0.

BOREAS is a new series, not a new version of THUMS. The P2H2P/THUMS line stays
exactly where it was at v0.16 and remains runnable; nothing here edits it. Two
things differ, and they are not independent:

  THE GEOMETRY. THUMS put n hairpins in the hole. A hairpin needs its cascade
  graded along the TUBE PATH (the return leg runs ~45 K below the melting
  point and would otherwise refreeze what the down leg melted), which puts two
  melting points at the same depth in contiguous PCM. Conduction across that
  gap is 43 kW at one hairpin and 92 kW at three, against a ~120 kW duty --
  a short circuit straight across the cascade the model exists to represent,
  and a term THUMS never had. BOREAS runs n_ft fin tubes DOWNWARD and returns
  the whole flow through ONE INSULATED DOWNCOMER. Same pressure drop, same PCM
  volume, no short circuit.

  THE FORMATION. THUMS hypothesis H6 made the borehole wall adiabatic, so
  eta_storage was identically 1 -- not approximately, exactly, by construction.
  BOREAS conducts to the rock through casing, cement and ground, with the
  undisturbed formation temperature set by an INDEPENDENT geothermal gradient.

The second needed the first. With a hairpin, every depth is occupied by two
tube segments, so one rock temperature would have had two PCM segments
claiming it. Only when the return stops exchanging does z become a depth.

Setting T_rock=None in `march_h` recovers the adiabatic wall exactly, which is
how the v0.16 reproduction test is run. One formulation (enthalpy, "Formulation C"), one sizing
framing (specify the hardware and march to cyclic steady state), no root
finding anywhere.

    plant      cycle_state_points   ->  state points and the two efficiencies
               energy_budget        ->  how much energy must move per cycle

    borehole   pcm_capacities       ->  E'_lat, C_s, C_l  per unit tube length
               pcm_state            ->  E'  ->  (A_melt, T_pcm)
               advance_segment      ->  branchwise-exact step of E'
               march_h              ->  one half-cycle over the whole exchanger

    cycle      simulate_css         ->  repeat until the state returns to itself
               simulate_css_corrected  ->  ... with one ORC correction pass
               css_report           ->  print it

Everything is driven by one frozen dataclass, `Case`. To study a parameter,
copy `CASE` with the field changed -- `CASE.with_(N_lay=12)` -- and re-run.
`validate_case` checks a case for the inconsistencies that are easy to create
and hard to see.

Superseded material is not here. The first-cycle sizing chain, the closed-form
Stefan front and the area-based melt front were removed; the reasoning that
removed them is in DESIGN_NOTES.md and in the project report, and the code is
in version control.
"""
import numpy as np
import CoolProp.CoolProp as CP
from CoolProp.CoolProp import PropsSI
from math import log, pi
import pandas as pd


# ==========================================================================
# properties
# ==========================================================================


########################################
########################################
########################################


# ==========================================================================
# cycles
# ==========================================================================


def double_stage_rankine(fluid, T_1e, T_3e, eta_t=1.0, eta_p=1.0):
  """
  Calculates parameters for a two-stage Rankine cycle with regeneration/reheating

  Args:
    fluid: The name of the working fluid.
    T_1e: Temperature of state 1e in Kelvin.
    T_3e: Temperature of state 3e in Kelvin.

  Returns:
    A dictionary containing calculated Rankine cycle parameters,
    including enthalpy, entropy, temperature, and pressure for each state,
    the Rankine efficiency, and saturation pressures.
  """
  # State 3e: Saturated vapor at the 1st stage (turbine) inlet
  p_evape = CP.PropsSI('P', 'T', T_3e, 'Q', 1, fluid) # Condenser pressure (same as 3e)
  p_3e = p_evape
  h_3e = CP.PropsSI('H', 'T', T_3e, 'Q', 1, fluid)/1000.
  s_3e = CP.PropsSI('S', 'T', T_3e, 'Q', 1, fluid)

  # State 1e: Saturated liquid at condenser outlet
  p_conde = CP.PropsSI('P', 'T', T_1e, 'Q', 0, fluid) # Evaporator pressure
  p_1e = p_conde
  h_1e = CP.PropsSI('H', 'T', T_1e, 'Q', 0, fluid)/1000.
  s_1e = CP.PropsSI('S', 'T', T_1e, 'Q', 0, fluid)

  # Intermediate pressure
#  p_inte = (p_evape * p_conde)**(1./2.)
  p_inte = (p_evape + p_conde)*(1./2.)

  # State 5e: 1st stage turbine outlet
  p_5e = p_inte
  # IRREVERSIBLE from v0.12. h_5e_s is where an isentropic machine would
  # land; the real one recovers only eta_t of that drop and therefore leaves
  # HOTTER, with more entropy. Everything downstream of 5e -- the reheat duty,
  # the regenerator split y, the evaporator inlet 10e -- moves with it, which
  # is exactly why a downstream multiplier on the work could not reproduce it.
  h_5e_s = CP.PropsSI('H', 'S', s_3e, 'P', p_5e, fluid)/1000.
  h_5e = h_3e - eta_t * (h_3e - h_5e_s)
  s_5e = CP.PropsSI('S', 'H', h_5e*1000., 'P', p_5e, fluid)
  T_5e = CP.PropsSI('T', 'H', h_5e*1000., 'P', p_5e, fluid)

  # State 6e: 2nd stage turbine inlet
  p_6e = p_5e
  T_6e = T_3e #ideal reheat
  h_6e = CP.PropsSI('H', 'T', T_6e, 'P', p_6e, fluid)/1000.
  s_6e = CP.PropsSI('S', 'T', T_6e, 'P', p_6e, fluid)

  # State 4e: 2nd stage turbine outlet
  p_4e = p_conde
  h_4e_s = CP.PropsSI('H', 'S', s_6e, 'P', p_4e, fluid)/1000.
  h_4e = h_6e - eta_t * (h_6e - h_4e_s)
  s_4e = CP.PropsSI('S', 'H', h_4e*1000., 'P', p_4e, fluid)
  T_4e = CP.PropsSI('T', 'H', h_4e*1000., 'P', p_4e, fluid)

  # State 2e: Pump 1 outlet
  p_2e = p_evape
  # a pump absorbs MORE than isentropic, so the divide goes the other way
  h_2e_s = CP.PropsSI('H', 'S', s_1e, 'P', p_2e, fluid)/1000.
  h_2e = h_1e + (h_2e_s - h_1e) / eta_p
  s_2e = CP.PropsSI('S', 'H', h_2e*1000., 'P', p_2e, fluid)
  T_2e = CP.PropsSI('T', 'H', h_2e*1000., 'P', p_2e, fluid)

  # State 7e: Pump 2 inlet
  p_7e = p_inte
  h_7e = CP.PropsSI('H', 'P', p_7e, 'Q', 0, fluid)/1000.
  s_7e = CP.PropsSI('S', 'P', p_7e, 'Q', 0, fluid)
  T_7e = CP.PropsSI('T', 'P', p_7e, 'Q', 0, fluid)

  # State 8e: Pump 2 outlet
  p_8e = p_evape
  h_8e_s = CP.PropsSI('H', 'P', p_8e, 'S', s_7e, fluid)/1000.
  h_8e = h_7e + (h_8e_s - h_7e) / eta_p
  s_8e = CP.PropsSI('S', 'H', h_8e*1000., 'P', p_8e, fluid)
  T_8e = CP.PropsSI('T', 'H', h_8e*1000., 'P', p_8e, fluid)

  # State 9e:
  p_9e = p_evape
  T_9e = T_7e
  h_9e = CP.PropsSI('H', 'T', T_9e, 'P', p_9e, fluid)/1000.
  s_9e = CP.PropsSI('S', 'T', T_9e, 'P', p_9e, fluid)

  # Energy balance in the regenerator
  y_frac = (h_5e - h_7e) / ((h_9e - h_2e) + (h_5e - h_7e))

  # State 10e:
  p_10e = p_evape
  h_10e = y_frac * h_9e + (1. - y_frac) * h_8e
  s_10e = CP.PropsSI('S', 'H', h_10e*1000, 'P', p_10e, fluid)
  T_10e = CP.PropsSI('T', 'H', h_10e*1000, 'P', p_10e, fluid)

  # cycle thermal efficiency
  numerator_eff = (h_3e - h_10e) + y_frac * (h_6e - h_5e) - y_frac * (h_4e - h_1e)
  denominator_eff = (h_3e - h_10e) + y_frac * (h_6e - h_5e)
  if abs(denominator_eff) > 1e-9:
      rank_eff = numerator_eff / denominator_eff
  else:
      rank_eff = float('inf') # Assign large value if denominator is zero or near zero

  return {
      'T_1e': T_1e, 'p_1e': p_1e, 'h_1e': h_1e, 's_1e': s_1e,
      'T_2e': T_2e, 'p_2e': p_2e, 'h_2e': h_2e, 's_2e': s_2e,
      'T_3e': T_3e, 'p_3e': p_3e, 'h_3e': h_3e, 's_3e': s_3e,
      'T_4e': T_4e, 'p_4e': p_4e, 'h_4e': h_4e, 's_4e': s_4e,
      'T_5e': T_5e, 'p_5e': p_5e, 'h_5e': h_5e, 's_5e': s_5e,
      'T_6e': T_6e, 'p_6e': p_6e, 'h_6e': h_6e, 's_6e': s_6e,
      'T_7e': T_7e, 'p_7e': p_7e, 'h_7e': h_7e, 's_7e': s_7e,
      'T_8e': T_8e, 'p_8e': p_8e, 'h_8e': h_8e, 's_8e': s_8e,
      'T_9e': T_9e, 'p_9e': p_9e, 'h_9e': h_9e, 's_9e': s_9e,
      'T_10e': T_10e, 'p_10e': p_10e, 'h_10e': h_10e, 's_10e': s_10e,
      'rank_eff': rank_eff,
      'p_evape': p_evape,
      'p_conde': p_conde,
      'p_inte': p_inte,
      'y_frac': y_frac # Added y_frac to the return dictionary
  }

# @title


def two_stage_htheatpump_2regs(refrig, T_13h, T_2h, DT_sub, eta_c=1.0):
  """
  Calculates parameters for a two-stage high-temp heat pump with a liquid separator and 2 regenerators

  Args:
    refrig: The name of the refrigerant.
    T_13h: Temperature of state 13h in Kelvin.
    T_2h: Temperature of state 2h in Kelvin.
    DT_sub: Refrigerant subcooling degree at condenser outlet.

  Returns:
    A dictionary containing calculated Rankine cycle parameters.
  """
  # State 2h: Saturated Vapor at T_2h (Condenser Outlet)
  p_condh = CP.PropsSI('P', 'T', T_2h, 'Q', 0, refrig) # Condensing pressure
  p_2h = p_condh
  h_2h = CP.PropsSI('H', 'T', T_2h, 'Q', 1, refrig)/1000.
  s_2h = CP.PropsSI('S', 'T', T_2h, 'Q', 1, refrig)

  # State 3h: Subcooled Liquid at (T_2h - DT_sub) and p_condh (Condenser Outlet with subcooling)
  T_3h = T_2h - DT_sub
  p_3h = p_condh
  h_3h = CP.PropsSI('H', 'T', T_3h, 'P', p_condh, refrig)/1000.
  s_3h = CP.PropsSI('S', 'T', T_3h, 'P', p_condh, refrig)

  # State 13h: Saturated vapor at evaporating pressure
  p_evaph = CP.PropsSI('P', 'T', T_13h, 'Q', 1, refrig) # Evaporating pressure
  p_13h = p_evaph
  h_13h = CP.PropsSI('H', 'T', T_13h, 'Q', 1, refrig)/1000.
  s_13h = CP.PropsSI('S', 'T', T_13h, 'Q', 1, refrig)


  # Intermediate pressure
  p_inth = (p_evaph * p_condh)**(1./2.)
#  p_inth = (p_evaph + p_condh)*(1./2.)

  # --- the compressions, IRREVERSIBLE from v0.12 ------------------------
  # The closure runs BACKWARDS: state 2h is fixed on the dew line at T_2h
  # (the condensing temperature the water demands), and the suction state is
  # whatever lands there. With eta_c = 1 that inverts in closed form, which
  # is what this used to do. With eta_c < 1 the same compression ends HOTTER,
  # so to still land on the dew point the suction must carry LESS superheat,
  # and the inversion is implicit -- h_2h_s depends on s_1h, which depends on
  # h_1h. Two bisections, outer stage first.
  #
  # The design intent is unchanged and worth restating: IHX-2 is sized to
  # deliver exactly the suction superheat that puts the discharge on the dew
  # point, so the condenser has no desuperheating duty. epsilon_IHX_2 remains
  # a reported OUTPUT, and it is what moves when eta_c does.
  p_1h = p_evaph
  p_11h = p_inth

  def _suction_for(h_target, p_lo, p_hi):
      """Enthalpy at p_lo whose eta_c compression to p_hi reaches h_target."""
      lo = CP.PropsSI('H', 'P', p_lo, 'Q', 1, refrig)/1000. - 40.0
      hi = h_target
      for _ in range(90):
          mid = 0.5*(lo + hi)
          s_mid = CP.PropsSI('S', 'H', mid*1000., 'P', p_lo, refrig)
          h_s = CP.PropsSI('H', 'S', s_mid, 'P', p_hi, refrig)/1000.
          if mid + (h_s - mid)/eta_c < h_target:
              lo = mid
          else:
              hi = mid
      return 0.5*(lo + hi)

  if abs(eta_c - 1.0) < 1e-12:
      s_1h = s_2h                      # closed form, the pre-v0.12 path
      T_1h = CP.PropsSI('T', 'P', p_1h, 'S', s_1h, refrig)
      h_1h = CP.PropsSI('H', 'T', T_1h, 'P', p_evaph, refrig)/1000.
      s_11h = s_1h
      T_11h = CP.PropsSI('T', 'S', s_11h, 'P', p_11h, refrig)
      h_11h = CP.PropsSI('H', 'S', s_11h, 'P', p_11h, refrig)/1000.
  else:
      h_11h = _suction_for(h_2h, p_inth, p_condh)      # HP stage inlet = 6h
      h_1h = _suction_for(h_11h, p_evaph, p_inth)      # LP stage inlet
      T_1h = CP.PropsSI('T', 'H', h_1h*1000., 'P', p_evaph, refrig)
      s_1h = CP.PropsSI('S', 'H', h_1h*1000., 'P', p_evaph, refrig)
      T_11h = CP.PropsSI('T', 'H', h_11h*1000., 'P', p_11h, refrig)
      s_11h = CP.PropsSI('S', 'H', h_11h*1000., 'P', p_11h, refrig)

  # State 5h: Low Pressure Compressor outlet
  T_5h = T_11h
  p_5h = p_11h
  h_5h = h_11h
  s_5h = s_11h

  # State 6h: High Pressure Compressor inlet
  T_6h = T_11h
  p_6h = p_11h
  h_6h = h_11h
  s_6h = s_11h

  # State 9h
  p_9h = p_inth
  T_9h = CP.PropsSI('T', 'P', p_9h, 'Q', 0, refrig) # Saturated liquid temp at p_inth
  h_9h = CP.PropsSI('H', 'P', p_9h, 'Q', 0, refrig)/1000.
  s_9h = CP.PropsSI('S', 'P', p_9h, 'Q', 0, refrig)

  # State 10h
  p_10h = p_inth
  T_10h = CP.PropsSI('T', 'P', p_10h, 'Q', 1, refrig) # Saturated vapor temp at p_inth
  h_10h = CP.PropsSI('H', 'P', p_10h, 'Q', 1, refrig)/1000.
  s_10h = CP.PropsSI('S', 'P', p_10h, 'Q', 1, refrig)

  # State 14h
  p_14h = p_evaph
  T_14h = CP.PropsSI('T', 'P', p_14h, 'Q', 0, refrig) # Saturated liquid temp at p_evaph
  h_14h = CP.PropsSI('H', 'P', p_14h, 'Q', 0, refrig)/1000.
  s_14h = CP.PropsSI('S', 'P', p_14h, 'Q', 0, refrig)

  # IHX-1 effectiveness (based on T_3h and T_10h)
  # Ensure denominator is non-zero
  denominator_epsilon1 = (T_3h - T_10h)
  if abs(denominator_epsilon1) > 1e-9:
      epsilon_IHX_1 = (T_11h - T_10h) / denominator_epsilon1
  else:
      epsilon_IHX_1 = 1.0 # Assume 100% effectiveness if temperature difference is zero

  # CP_ratio_1:
  try:
      cpf_1 = CP.PropsSI('C', 'P', p_condh, 'Q', 0, refrig)
      cpv_1 = CP.PropsSI('C', 'P', p_inth, 'Q', 1, refrig)
      # Avoid division by zero
      if abs(cpf_1) > 1e-9:
           cpr_1 = cpv_1 / cpf_1
      else:
           cpr_1 = 1.0 # Assume ratio is 1 if cpf is zero
  except ValueError:
      cpr_1 = 1.0

  # --- the IHX network, closed by an exact enthalpy balance ---------------
  # The two regenerators are closed by ENERGY, not by effectiveness-times-
  # cp-ratio correlations. Which stream passes through each one matters:
  #
  #   IHX-1   liquid 3h -> 12h  (1 kg)   heats the FLASHED VAPOUR, x_8h,
  #                                      from 10h to 11h
  #   IHX-2   liquid 12h -> 7h  (1 kg)   heats the SUCTION stream, which is
  #                                      what left the separator as LIQUID,
  #                                      (1 - x_8h), from 13h to 1h
  #
  # x_8h appears on both sides, so the three relations are solved as a fixed
  # point. The map is linear and strongly contracting here, so it converges
  # in a handful of steps.
  #
  # Until v0.9 IHX-2 used x_4h -- the vapour QUALITY at the evaporator inlet
  # -- where the flow SPLIT (1 - x_8h) belongs. A quality is not a flow
  # fraction, and the cycle did not close: IHX-2 took 9.19 kJ/kg out of the
  # liquid and put 21.76 kJ/kg into the vapour, and the condenser reported
  # 271.43 kJ/kg against 258.33 kJ/kg of work plus evaporator heat. That is
  # a 4.8 % creation of energy, and it inflated the COP.
  p_7h = p_condh
  dh_IHX1 = h_11h - h_10h            # per kg of flashed vapour
  dh_IHX2 = h_1h - h_13h             # per kg of suction (low-stage) flow
  x_8h = 0.3
  for _n_ihx in range(200):
      h_12h = h_3h - x_8h * dh_IHX1
      h_7h = h_12h - (1.0 - x_8h) * dh_IHX2
      x_new = (h_7h - h_9h) / (h_10h - h_9h)          # h_8h = h_7h
      converged = abs(x_new - x_8h) < 1e-12
      x_8h = x_new
      if converged:
          break
  else:
      raise RuntimeError(
          "two_stage_htheatpump_2regs: the IHX enthalpy balance did not "
          f"converge in 200 iterations (x_8h = {x_8h!r}). The pressure "
          "levels or the subcooling are probably inconsistent.")
  if not np.isfinite(x_8h) or not (0.0 <= x_8h <= 1.0):
      raise RuntimeError(
          f"two_stage_htheatpump_2regs: separator vapour fraction "
          f"x_8h = {x_8h:.6g} is outside [0, 1], so there is no physical "
          "flash split. Check T_2h, T_13h and DT_sub.")
  h_12h = h_3h - x_8h * dh_IHX1
  h_7h = h_12h - (1.0 - x_8h) * dh_IHX2
  h_8h = h_7h                        # isenthalpic expansion into the separator
  h_4h = h_9h                        # isenthalpic expansion into the evaporator
  T_7h = CP.PropsSI('T', 'H', h_7h * 1000., 'P', p_condh, refrig)
  T_12h = CP.PropsSI('T', 'H', h_12h * 1000., 'P', p_condh, refrig)

  # Both of these are now DIAGNOSTICS rather than inputs. State 1h is fixed
  # by the isentropic compression back from 2h, so the effectiveness IHX-2
  # would need in order to deliver it is an OUTPUT of the cycle, and worth
  # reading: a value above 1 would mean the regenerator cannot exist.
  x_4h = (h_4h - h_14h) / (h_13h - h_14h)
  epsilon_IHX_2 = (T_1h - T_13h) / (T_12h - T_13h)
  cpr_2 = (CP.PropsSI('C', 'P', p_evaph, 'Q', 1, refrig)
           / CP.PropsSI('C', 'P', p_condh, 'Q', 0, refrig))


  # State 7h (using the converged T_7h)
  # h_7h and s_7h were calculated inside the loop, re-calculate if needed outside
  try:
       h_7h = CP.PropsSI('H', 'T', T_7h, 'P', p_7h, refrig)/1000.
       s_7h = CP.PropsSI('S', 'T', T_7h, 'P', p_7h, refrig)
  except ValueError:
       h_7h, s_7h = np.nan, np.nan


  # State 12h (using the converged T_12h)
  # T_12h was calculated inside the loop, re-calculate if needed outside
  try:
       p_12h = p_condh
       h_12h = CP.PropsSI('H', 'T', T_12h, 'P', p_12h, refrig)/1000.
       s_12h = CP.PropsSI('S', 'T', T_12h, 'P', p_12h, refrig)
  except ValueError:
       h_12h, s_12h = np.nan, np.nan


  # State 8h (using the converged h_8h=h_7h and p_8h=p_inth)
  # h_8h was calculated inside the loop, re-calculate if needed outside
  try:
       p_8h = p_inth
       T_8h = CP.PropsSI('T', 'P', p_8h, 'H', h_8h*1000., refrig)
       s_8h = CP.PropsSI('S', 'P', p_8h, 'H', h_8h*1000., refrig)
  except ValueError:
       T_8h, s_8h = np.nan, np.nan


  # State 4h (using the converged h_4h=h_9h and p_4h=p_evaph)
  # h_4h was calculated inside the loop, re-calculate if needed outside
  try:
       p_4h = p_evaph
       T_4h = CP.PropsSI('T', 'P', p_4h, 'H', h_4h*1000., refrig)
       s_4h = CP.PropsSI('S', 'P', p_4h, 'H', h_4h*1000., refrig)
  except ValueError:
       T_4h, s_4h = np.nan, np.nan


  # HP COP
  # Ensure denominator is non-zero before division and check for NaN values
  # Need to use the final converged x_8h and h_5h, h_1h
  denominator_cop = (h_2h - h_1h) - x_8h * (h_5h - h_1h)
  if abs(denominator_cop) > 1e-9 and not np.isnan(h_2h) and not np.isnan(h_1h) and not np.isnan(x_8h) and not np.isnan(h_5h):
      hp_cop = (h_2h - h_3h) / denominator_cop
  else:
      hp_cop = float('inf') # Assign a large value if denominator is zero or near zero or NaN


  return {
      'T_1h': T_1h, 'p_1h': p_1h, 'h_1h': h_1h, 's_1h': s_1h,
      'T_2h': T_2h, 'p_2h': p_2h, 'h_2h': h_2h, 's_2h': s_2h,
      'T_3h': T_3h, 'p_3h': p_3h, 'h_3h': h_3h, 's_3h': s_3h,
      'T_4h': T_4h, 'p_4h': p_4h, 'h_4h': h_4h, 's_4h': s_4h,
      'T_5h': T_5h, 'p_5h': p_5h, 'h_5h': h_5h, 's_5h': s_5h,
      'T_6h': T_6h, 'p_6h': p_6h, 'h_6h': h_6h, 's_6h': s_6h,
      'T_7h': T_7h, 'p_7h': p_7h, 'h_7h': h_7h, 's_7h': s_7h,
      'T_8h': T_8h, 'p_8h': p_8h, 'h_8h': h_8h, 's_8h': s_8h,
      'T_9h': T_9h, 'p_9h': p_9h, 'h_9h': h_9h, 's_9h': s_9h,
      'T_10h': T_10h, 'p_10h': p_10h, 'h_10h': h_10h, 's_10h': s_10h,
      'T_11h': T_11h, 'p_11h': p_11h, 'h_11h': h_11h, 's_11h': s_11h,
      'T_12h': T_12h, 'p_12h': p_12h, 'h_12h': h_12h, 's_12h': s_12h,
      'T_13h': T_13h, 'p_13h': p_13h, 'h_13h': h_13h, 's_13h': s_13h,
      'hp_cop': hp_cop,
      'p_evaph': p_evaph,
      'p_condh': p_condh, # Changed from p_evaph to p_condh
      'p_inth': p_inth,
      'epsilon_IHX_1': epsilon_IHX_1,
      'epsilon_IHX_2': epsilon_IHX_2,
      'x_8h': x_8h,
  }

# @title
#import numpy as np
#import CoolProp.CoolProp as CP
#import matplotlib.pyplot as plt


# ==========================================================================
# network
# ==========================================================================


########################################
########################################
########################################


def compute_U_i(h_i, r_i, r_e, k_w, Rf_i_prime, k_m, delta, L_tube, fin_t, fin_L, num_fins):
    """
    Computes the overall heat transfer coefficient based on internal surface area,
    including thermal resistances for internal convection, wall conduction, fouling,
    and the outer finned surface with a melt layer and fin efficiency.

    Args:
        h_i: Internal convective heat transfer coefficient [W/m²·K].
        r_i: Internal radius of the tube [m].
        r_e: External radius of the tube [m].
        k_w: Casing wall thermal conductivity (also used for fin conductivity) [W/m·K]. # Updated docstring
        Rf_i_prime: Internal fouling resistance [m²·K/W].
        k_m: PCM thermal conductivity [W/m.K].
        delta: Melt layer thickness [m].
        L_tube: Length of the tube [m].
        fin_t: Thickness of each fin [m].
        fin_L: Length of each fin extending radially outwards [m].
        num_fins: The number of fins.

    Returns:
        Overall heat transfer coefficient based on internal surface area [W/m²·K].
    """
    R_conv_internal = 1.0 / h_i
    R_wall = (r_i / k_w) * log(r_e / r_i)
    R_fouling = Rf_i_prime

    # Calculate h_e (heat transfer coefficient for the outer layer)
    if delta > 1e-9 and r_e > 1e-9: # Check for non-zero delta and r_e
        log_term = np.log(1 + delta / r_e)
        if log_term > 1e-9: # Check for non-zero log term
            h_e = k_m / r_e / log_term
        else:
             # If log_term is near zero, delta/r_e is very small, meaning very thin layer.
             # This would imply a large h_e, but setting it too large might cause issues.
             # Assign a large, but finite value. Or, consider the limit as delta approaches 0.
             # As delta -> 0, log(1 + delta/r_e) ~ delta/r_e. h_e ~ k_m / r_e / (delta/r_e) = k_m/delta.
             # So the previous definition is consistent with the limit.
             # Let's revert to the previous definition for small delta, or handle it as an infinite h_e (zero resistance).
             # Given the error handling downstream, a large finite number for h_e is safer than infinity here.
             h_e = k_m / delta # Use the previous definition for very small delta
    else:
        # If delta is zero, there is no melt layer, so resistance is infinite
        h_e = 1e9 # Assign a small value, leading to large resistance

    # Calculate fin parameters for efficiency
    fin_Lc = fin_L + fin_t / 2.0
    # Ensure h_e is non-zero or negative before taking sqrt, use k_w for fin conductivity
    if h_e > 1e-9 and k_w > 1e-9 and fin_t > 1e-9:
         fin_m = np.sqrt(2 * h_e / k_w / fin_t) # Used k_w instead of KW
    else:
         fin_m = 1e9 # Assign a large value if denominator is near zero


    # Calculate fin efficiency
    fin_m_Lc = fin_m * fin_Lc
    if fin_m_Lc != 0:
        fin_eff = np.tanh(fin_m_Lc) / fin_m_Lc
    else:
        fin_eff = 1.0 # Efficiency is 100% if fin_m * fin_Lc is zero

    # Calculate fin surface area
    A_f = L_tube * (2 * fin_L + fin_t)

    # Calculate total outer surface area
    A_t = 2 * L_tube * (np.pi * r_e + num_fins * fin_L)

    # Calculate overall surface efficiency
    # Ensure A_t is not zero to avoid division error
    if A_t > 0:
        eta_o = 1 - (num_fins * A_f / A_t) * (1 - fin_eff)
    else:
        eta_o = 1.0 # If total area is zero, efficiency is irrelevant, assume 1

    # Calculate the outer perimeter of the finned tube (used in R_outer_layer calculation)
    perim_T = 2 * pi * r_e + 2 * num_fins * fin_L

    if eta_o > 1e-9 and h_e > 1e-9: # Check if eta_o and h_e are effectively non-zero
         R_outer_layer = (2 * pi * r_i) / (perim_T * eta_o * h_e)
    else:
        R_outer_layer = float('inf') # Assign infinite resistance if heat transfer is blocked

    R_total = R_conv_internal + R_wall + R_fouling + R_outer_layer

    if R_total != 0 and not np.isinf(R_total):
        U_i = 1.0 / R_total
    else:
        U_i = 0.0 # If total resistance is infinite or zero, U_i is 0.0 (no heat transfer)

    return U_i


########################################
########################################
########################################


# ==========================================================================
# front_closed_form
# ==========================================================================


#####################################
#####################################
#####################################


# ==========================================================================
# march_legacy
# ==========================================================================


########################################
########################################
########################################


# ==========================================================================
# hydraulics
# ==========================================================================


def calculate_pressure_drop(geom_par_vector, m_dot_well_tube, fluid, T_inlet, T_outlet, P_inlet):
    """
    Calculates the pressure drop and pumping power in a single tube of the hairpin heat exchanger.

    Args:
        geom_par_vector: An array or list containing [L_well, L_tube, D_well, r_i, r_e, D_i_tube, D_e_tube, fin_t, fin_L, num_fins, num_tubes].
                         We will use L_tube and D_i_tube (2*r_i) from this vector.
        m_dot_well_tube: Mass flow rate of the fluid through a single tube [kg/s].
        fluid: Name of the working fluid.
        T_inlet: Inlet temperature of the fluid [K].
        T_outlet: Outlet temperature of the fluid [K].
        P_inlet: Inlet pressure of the fluid [Pa].


    Returns:
        A tuple containing:
            - Pressure drop in the tube [Pa].
            - Pumping power [W]. # Added pumping power to return values
    """
    L_well, L_tube, D_well, r_i, r_e, D_i_tube, D_e_tube, fin_t, fin_L, num_fins, num_tubes = geom_par_vector
    L = L_tube # Length of the tube
    D_i = D_i_tube # Inner diameter of the tube
    r_i_tube = D_i / 2.0 # Inner radius

    # Average temperature and pressure for fluid properties
    T_avg = 0.5 * (T_inlet + T_outlet)
    # Assume average pressure is close to inlet pressure for property calculation for now
    # A more rigorous approach might iterate, but this is a good starting point.
    P_avg = P_inlet # Approximation


    try:
        # Get fluid properties at average conditions
        rho = CP.PropsSI("D", "T", T_avg, "P", P_avg, fluid)
        mu = CP.PropsSI("V", "T", T_avg, "P", P_avg, fluid) # Dynamic viscosity
        # You might also need Specific Heat (Cp) and Thermal Conductivity (k) for friction factor correlations,
        # but for basic pressure drop (Darcy-Weisbach), rho and mu are key.

    except ValueError:
        print(f"Warning: Could not retrieve fluid properties for {fluid} at T={T_avg:.2f}K, P={P_avg:.2f}Pa.")
        return float('nan'), float('nan') # Return NaN for both if properties fail


    # Calculate flow velocity
    A_flow = np.pi * r_i_tube**2
    if rho * A_flow > 1e-9:
        u = m_dot_well_tube / (rho * A_flow)
    else:
        return 0.0, 0.0 # No flow, no pressure drop and no pumping power

    # Calculate Reynolds number
    if mu > 1e-9 and D_i > 1e-9:
        Re = rho * u * D_i / mu
    else:
        return 0.0, 0.0

    # Calculate friction factor (Darcy friction factor)
    f = 0.0
    if Re < 2000: # Laminar flow
        f = 64.0 / Re
    else: # Turbulent flow (using Haaland correlation as an example)
        # More accurate correlations like Colebrook-White could be used if roughness is known.
        # For simplicity, using an approximation that doesn't require iteration for now.
        # If pipe roughness (epsilon) is available, a better correlation can be used.
        # Assuming smooth pipe for now (epsilon = 0).
        # The Haaland correlation: 1/sqrt(f) = -1.8 * log10((epsilon/D_i)/3.7 + (6.9/Re))
        # With epsilon=0: 1/sqrt(f) = -1.8 * log10(6.9/Re)
        # f = (1 / (-1.8 * log10(6.9/Re)))**2

        # Using a simpler turbulent correlation like Blasius for smooth pipes (Re < 100,000)
        # or a transition correlation for wider Re range.
        # Let's use a common approximation for smooth pipes for simplicity:
        if Re <= 1e5: # Blasius correlation
             f = 0.3164 / (Re**0.25)
        else: # For higher Re, use a different correlation, e.g., from Moody chart or a more general formula
             # Swamee–Jain equation (explicit approximation of Colebrook–White)
             # Requires pipe roughness (epsilon). Let's assume a typical value for smooth drawn tubing.
             # Example epsilon for drawn tubing might be 0.0015 mm = 1.5e-6 m
             # If epsilon is not a parameter, we have to stick to correlations that don't need it,
             # or make an assumption. Let's assume smooth pipe and use Blasius/extend it.
             # A more robust approach would take epsilon as an input parameter.
             # For now, extending Blasius or using a simple fit:
             # Let's use a piecewise approach based on Re.
             # For very high Re, f approaches a constant for smooth pipes.
             # A simple fit across a wider range might be better.
             # Let's stick to the Blasius for Re <= 100,000 and
             # a rough approximation for higher Re if needed, or add roughness as parameter.

             # For now, let's refine the Blasius range and consider a higher Re approximation if necessary.
             # The Blasius correlation is typically for Re up to 10^5.
             # For 10^5 < Re < 10^6, a common approximation is f = 0.0032 + 0.221 / Re^0.237.
             # For Re >= 10^6, f ~ 0.018 (fully turbulent smooth pipe).
             # Let's add a simple transition:
             if Re <= 1e6:
                  f = 0.0032 + 0.221 / Re**0.237
             else:
                  f = 0.018 # Approximation for very high Re, smooth pipe


    # Frictional pressure drop (Darcy-Weisbach equation)
    # Ensure D_i is non-zero before division
    if D_i > 1e-9:
        delta_P_friction = f * (L / D_i) * (0.5 * rho * u**2)
    else:
         delta_P_friction = 0.0


    # Minor losses (e.g., bends, entrance/exit)
    # These depend on fittings and geometry. Assuming a simple hairpin with two 180-degree bends
    # and entrance/exit losses. Need minor loss coefficients (K).
    # For simplicity, assuming K_bend for 180 deg return bend ~ 2.2 (for flanged or welded)
    # K_entrance ~ 0.5, K_exit ~ 1.0
    # Total minor loss coefficient K_total = K_entrance + 2 * K_bend + K_exit
    # K_total = 0.5 + 2 * 2.2 + 1.0 = 5.9 (example values)
    # Minor loss pressure drop: delta_P_minor = K_total * 0.5 * rho * u**2

    # Let's include minor losses as a parameter or use assumed values.
    # For this function, let's assume typical minor loss coefficients for a hairpin.
    K_entrance = 0.5 # Sharp-edged entrance
    K_bend_180 = 2.2 # 180 degree standard elbow/bend
    K_exit = 1.0 # Exit into a large reservoir/pipe

    K_total = K_entrance + 2 * K_bend_180 + K_exit # For one hairpin (two tubes and two bends)

    delta_P_minor = K_total * 0.5 * rho * u**2

    # Total pressure drop
    delta_P_total = delta_P_friction + delta_P_minor

    # Calculate pumping power [W]
    # Ensure rho is non-zero before division
    if rho > 1e-9:
        pumping_power = delta_P_total * m_dot_well_tube / rho
    else:
        pumping_power = 0.0


    return delta_P_total, pumping_power # Return both pressure drop and pumping power

# @title

# Function to evaluate the energy ratio for a given value of N_wells (Charging)


# ==========================================================================
# solvers
# ==========================================================================


#########################################
#########################################
#########################################

# Function to evaluate the energy ratio for a given value of m_dot_d_well1 (Discharging)


#########################################
#########################################
#########################################


#########################################
#########################################
#########################################


#############################################
#############################################
#############################################


# ==========================================================================
# config
# ==========================================================================
# One flat Case object. v0.1 passed sixteen positional arguments, which is how
# the PCM conductivity ended up in the slot meant for the tube wall for months
# without anything contradicting it. Named fields make that mistake visible.

from dataclasses import dataclass, replace

FT = 0.3048
IN = 0.0254


@dataclass(frozen=True)
class Case:
    # ---- PCM (Saher 2024 trimodal) ----
    T_m_C: float = 150.0          # melting temperature            [C]
    rho_s: float = 1550.0         # solid density                  [kg/m3]
    rho_l: float = 1450.0         # liquid density                 [kg/m3]
    cp_s: float = 1280.0          # solid specific heat            [J/kg/K]
    cp_l: float = 1800.0          # liquid specific heat           [J/kg/K]
    k_s: float = 0.60             # solid conductivity             [W/m/K]
    k_l: float = 0.45             # liquid conductivity            [W/m/K]
    h_m: float = 380_000.0        # latent heat of fusion          [J/kg]
    cost_per_kWh: float = 20.0

    # ---- borehole and tube geometry ----
    L_ft: float = 5000.0          # well depth                     [ft]
    D_in: float = 7.0             # borehole diameter              [in]
    r_i: float = 0.01623          # tube inner radius              [m]
    r_e: float = 0.02108          # tube outer radius              [m]
    fin_t: float = 0.0015         # fin thickness                  [m]
    fin_L: float = 0.0035         # fin radial length              [m]
    num_fins: int = 24            # fins per fin tube
    # ---- THE DOWNCOMER GEOMETRY. This is what BOREAS is for. ------------
    # THUMS put n hairpins in the hole: each a U-tube, down one leg and back
    # up the other, both legs exchanging with the PCM. That geometry is
    # self-contradictory and the contradiction is quantitative.
    #
    # Charge water enters at T_m + DT_4C_M and glides DT_3C_2C down, so the
    # RETURN leg runs ~45 K BELOW the melting point. Graded by depth it
    # refreezes what the down leg just melted. So THUMS graded the cascade
    # along the TUBE PATH instead -- which works, but requires two different
    # melting points to coexist at the same depth, centimetres apart, with
    # contiguous PCM and no wall between them. Conduction between those legs
    # is 43 kW at one hairpin and 92 kW at three, against a ~120 kW duty, and
    # it is a short circuit straight across the cascade the model exists to
    # represent. THUMS never had that term.
    #
    # BOREAS separates the two jobs. n_ft FIN TUBES all run DOWNWARD and all
    # exchange; ONE INSULATED DOWNCOMER returns the whole flow without
    # exchanging. They meet in a manifold at the bottom of the same hole --
    # no lateral, no new drilling, the same class of object as the U-bend it
    # replaces. Consequences:
    #
    #   z becomes DEPTH, one-to-one (L_tube = L_well, not twice it)
    #   the legs are PARALLEL, not series
    #   the cascade grades with depth, so the PCM is POURABLE in layers
    #   the cascade runs HOT AT THE BOTTOM, which aligns it with the
    #     geothermal gradient instead of fighting it -- see cascade_hot_bottom
    #
    # Sizing the downcomer for equal velocity keeps the pressure drop
    # identical to the hairpin's: same velocity everywhere, same total path
    # 2*L_well. The nested-coaxial alternative costs 3.3x on f_pump, which is
    # the one quantity Study 2 showed governs efficiency below ground.
    n_ft: int = 3                 # FIN TUBES per borehole (all downward)
    t_ins: float = 0.008          # downcomer insulation thickness [m]
    k_ins: float = 0.007          # downcomer insulation, VIT       [W/m/K]
    k_wall: float = 45.0          # steel conductivity             [W/m/K]
    Rf_i: float = 1e-3            # internal fouling resistance    [m2K/W]

    # ---- the formation. THUMS hypothesis H6 said the wall was adiabatic,
    # which made eta_storage identically 1 -- a number no reviewer of a
    # borehole storage paper will accept. These are what break it.
    T_surface_C: float = 18.0     # ground surface / mean annual air [C]
    grad_K_per_km: float = 45.0   # geothermal gradient              [K/km]
    k_rock: float = 2.5           # formation conductivity        [W/m/K]
    rho_cp_rock: float = 2.34e6   # formation volumetric heat cap [J/m3/K]
    k_cement: float = 1.0         # annular cement                [W/m/K]
    r_casing_o: float = 0.098     # casing outer radius             [m]
    r_bore: float = 0.110         # drilled hole radius             [m]
    t_operation_yr: float = 10.0  # age of the field when evaluated  [yr]
    # The gradient and the source temperature are specified INDEPENDENTLY and
    # checked against each other, never derived one from the other. 60 C at
    # the wellhead is a LOWER BOUND on the formation temperature -- the water
    # cools coming up -- and the producing wells may be deeper than these.
    # Writing that bound as an equality is what an earlier draft did, and it
    # threw away exactly the physics the deficit represents.
    L_producing_ft: float = 8000.0   # depth the geothermal water comes from

    # ---- THE WELL FIELD IS A CONE, NOT AN ARRAY -------------------------
    # THUMS wellheads sit on 6 ft centres with 12 ft between double rows,
    # and over 1200 wells are drilled DIRECTIONALLY from four islands with
    # drift angles to 84 deg, reaching 6500 acres. So the spacing is not a
    # number, it is a function of depth: metres apart at the wellhead, of
    # order 150 m at reservoir depth.
    #
    # That matters more than any other single parameter here. Neighbours
    # merge thermally in r^2/alpha, which at 6 ft is 36 DAYS and at 160 m is
    # eight centuries. The shallow section is one thermal body within a month
    # and barely loses at all; the deep section is a field of isolated wells.
    #
    # And the two effects run opposite ways with depth. Shallow rock is cold,
    # so the driving difference is largest exactly where the shielding is
    # strongest; deep rock is hot, so the driving difference is smallest
    # exactly where the wells stand alone. Neither a single spacing nor a
    # single resistance can represent that, which is why B is B(z).
    #
    # n_wells_field = None recovers the isolated well, which is the
    # conservative bound and the v1.0 default behaviour.
    n_wells_field: float = None   # wells sharing the formation; None = alone
    B_surface_m: float = 1.83     # wellhead spacing, 6 ft                [m]
    z_kickoff_m: float = 300.0    # vertical above the kickoff point      [m]
    fan_angle_deg: float = 45.0   # effective half-angle of the fan     [deg]

    # ---- power block and operating conditions ----
    fluid: str = "cyclopentane"   # ORC working fluid
    fluid2: str = "Water"         # borehole secondary fluid
    refrig: str = "cyclopentane"  # heat-pump refrigerant
    P: float = 1e6                # secondary-fluid pressure       [Pa]
    W_dot_el_out: float = 1000.0  # ORC net electrical output      [kW]
    # ---- machine efficiencies -------------------------------------------
    # ISENTROPIC efficiencies, applied INSIDE the cycles from v0.12: the
    # expansions, compressions and pumpings move the state points themselves.
    # Before v0.12 the state points were isentropic and 0.85 was applied as a
    # multiplier on the work downstream. That is accurate for the efficiency
    # -- 0.6 % at the design point -- but it leaves every state point ideal,
    # so the turbine exit, the regenerator split, the composite curves and
    # the evaporator pinch were all the reversible cycle's. It also makes a
    # component-wise exergy balance impossible: an isentropic machine
    # destroys nothing, so the lost work appears nowhere. See DN-21.
    #
    # Setting all three to 1.0 recovers the pre-v0.12 cycles exactly.
    eta_turb_s: float = 0.85      # ORC turbines, isentropic
    eta_pump_s: float = 0.85      # ORC pumps, isentropic
    eta_comp_s: float = 0.85      # heat-pump compressors, isentropic
    # ELECTRICAL / MECHANICAL, and correctly applied outside the working
    # fluid: these are not thermodynamic irreversibilities of the cycle.
    ElG_eff: float = 0.95         # ORC generator
    ElH_eff: float = 0.95         # compressor motor
    # RETIRED at v0.12. These were the isentropic efficiencies applied as
    # downstream multipliers; keeping them would double-count. validate_case
    # raises if a Case still names either.
    Turb_eff: float = None
    Comp_eff: float = None
    T_sink_C: float = 20.0
    # Approach at the ORC condenser, measured to the sink OUTLET. With
    # DT_sink_glide = 0 the sink is an infinite reservoir and outlet = inlet,
    # which is the model as built. Referencing it to the inlet -- as this did
    # before v0.9b -- is the same wrong-end pattern that made DT_3A_13H
    # unsafe; it was merely masked by the reservoir assumption. See DN-18.
    DT_E_sink: float = 5.0
    # Temperature rise of the sink stream through the ORC condenser. Zero is
    # the reservoir idealisation. Give it a value and the condenser acquires
    # an INTERIOR pinch at the desuperheat corner, ~93 % of the duty, which
    # crosses at about 5.4 K -- the ORC condenser is not structurally safe
    # the way the HTHP condenser is.
    DT_sink_glide: float = 0.0
    DT_2D_3E: float = 12.0
    # Minimum water-to-working-fluid gap ANYWHERE in the ORC evaporator, not
    # just at its hot end. DT_2D_3E alone cannot keep the two composite curves
    # apart, because the ORC takes most of its heat at one temperature while
    # the water glides; see `orc_pinch`. Raise this and the ORC boils lower
    # and yields less; lower it and the exchanger grows. See DN-15.
    DT_pinch_ORC: float = 5.0
    # Discharge-inlet subcooling below the COLDEST cascade layer, state 1d in
    # the plant diagram; the symmetric partner of DT_4C_M. See DN-8.
    DT_M_1D: float = 10.0
    T_source_C: float = 60.0
    DT_4C_M: float = 10.0
    # How far the SOURCE stream is cooled in the heat-pump evaporator. This is
    # the specification: it sets the source flow needed for a given duty.
    DT_3A_4A: float = 10.0
    # Minimum approach in the heat-pump evaporator, at its COLD end -- where
    # the source leaves and the refrigerant enters. The evaporating
    # temperature is DERIVED from it,
    #     T_13h = T_source_C - DT_3A_4A - DT_pinch_HPE,
    # so the approach can no longer be negative by arithmetic. Unlike the ORC
    # evaporator the cold stream here is isothermal with NO preheat kink, so
    # the gap is monotonic in Q and an end approach is genuinely sufficient:
    # no composite-curve search is needed. See DN-17.
    DT_pinch_HPE: float = 5.0
    # RETIRED at v0.9. The evaporating temperature used to be set by
    # T_13h = T_source_C - DT_3A_13H, independently of DT_3A_4A, so the
    # approach was the DIFFERENCE of two free fields and nothing checked its
    # sign. Left here only so that an old Case naming it fails loudly in
    # validate_case rather than being silently ignored.
    DT_3A_13H: float = None
    DT_2H_3C: float = 10.0
    DT_sub: float = 2.0
    # The CHARGING water glide, across the HTHP condenser. Because
    # T_3c = T_4c -- the water leaves the condenser and enters the borehole
    # with no state change between -- this is identically the borehole
    # charging glide. That equality is energy conservation on a closed loop,
    # not an assumption, and cannot be relaxed.
    DT_3C_2C: float = 55.0        # charging water glide           [K]
    # The DISCHARGING water glide, across the ORC evaporator, and likewise
    # identically the borehole discharging glide. Until v0.9b this was not a
    # field at all: both mass flows were divided by DT_3C_2C, which silently
    # asserted that the two half-cycles glide by the same amount. They need
    # not. None keeps the old behaviour exactly. See DN-18.
    DT_3D_2D: float = None        # discharging water glide        [K]
    # The cascade grading parameter. Layer spacing is DT_cascade / N_lay and
    # the top-to-bottom SPAN is DT_cascade * (N_lay - 1) / N_lay, one layer
    # short of DT_cascade itself.
    #
    # Setting this equal to the charging glide -- which None does -- is not
    # arbitrary: it is the unique choice that makes the approach between the
    # water and the layer it is melting equal to DT_4C_M at the LEADING FACE
    # of every layer, decaying to DT_4C_M - DT_cascade/N_lay at each trailing
    # face. That sawtooth is the cascade.
    #
    # Prescribing it independently is legitimate, but it is bounded BELOW,
    # not above: the approach at the far end of the well is
    #     charging : span - (glide_ch - DT_4C_M)
    #     discharge: span - (glide_dc - DT_M_1D)
    # so a span SMALLER than glide minus the approach inverts the driving
    # difference at that end. At the default the margin is only 3.889 K.
    DT_cascade: float = None      # cascade grading parameter      [K]
    t_ch: float = 10.0            # charging duration              [h]
    t_dc: float = 10.0            # discharging duration           [h]

    # ---- the geothermal resource (new at v0.13) --------------------------
    # Until v0.13 the 60 C source was effectively unlimited: the model drew
    # whatever flow the duty needed and no output recorded how much that was.
    # The exergy audit makes it a priced resource, so the flow is now reported
    # and converted into a well count.
    #
    # m_geo is NOT an input. It is DERIVED, exactly as N_wells is:
    #     m_geo = Q_src / (h(T_source) - h(T_source - DT_3A_4A))
    # and the uncertainty is carried by the per-well yield below, which is a
    # field-measured quantity, rather than by a plant-level flow that nobody
    # has ever measured. See DN-22.
    #
    # PROVISIONAL DEFAULT. 12.5 kg/s is the producer in the 4500 m / 206 C
    # repurposed-well case study, which is a hotter and deeper well than ours;
    # the published range for repurposed oil and gas wells runs from about
    # 1 kg/s up to that figure, and co-produced water from depleted fields can
    # exceed it. Treat N_geo as proportional to this number, not as a result.
    m_dot_geo_well: float = 12.5  # per producing well             [kg/s]
    # Lower bound on the reinjection temperature, T_4a = T_source - DT_3A_4A.
    # This is what bounds DT_3A_4A from ABOVE, and the second law wants to run
    # against that bound: raising DT_3A_4A cuts m_geo far faster than it cuts
    # COP. The default is deliberately slack -- it is a placeholder for a
    # formation constraint (silica scaling, injectivity) that the project does
    # not yet have a number for. validate_case warns, it does not raise.
    T_reinject_min_C: float = 25.0
    # Which exergy is charged for the geothermal stream.
    #   'resource'  everything the well lifts, relative to T_sink, with the
    #               reinjected stream booked as a named LOSS attributed to no
    #               component. The right convention for a dedicated well, which
    #               is paid for whether or not its exergy is used.
    #   'stripped'  only what the evaporator removes; reinjection is free.
    # The two give OPPOSITE guidance on DT_3A_4A: 'stripped' falls monotonically
    # and pushes the design toward the largest possible well count, 'resource'
    # has an interior optimum near 20 K. See DN-23.
    exergy_convention: str = "resource"
    loss_surplus: float = 0.05    # assumed storage loss, lambda
    N_lay: int = 9                # PCM layers along the well

    # ---- numerics ----
    n_segments: int = 100
    n_times: int = 40
    delta_max: float = 0.5
    delta_tol: float = 1e-5
    delta_maxiter: int = 20
    N_wells_bracket: tuple = (3.0, 120.0)
    ratio_bracket: tuple = (0.2, 2.0)
    tol_Q_ratio: float = 1e-3
    max_iterations: int = 40

    # ---- switches ----
    # The ONLY remaining switch. True carries PCM enthalpy (latent + sensible in
    # both phases), so a fully melted segment superheats and a fully solid one
    # subcools. False pins the PCM at T_m, which is retained not as a model but
    # as the control case for the self-levelling comparison: it isolates what
    # the sensible branches do, at a fixed well count.
    sensible_heat: bool = True

    # Which material lies between the tube and the phase front, and therefore
    # which shell the heat has to cross. See `conduction_shell` and DN-13.
    #   'directional'  melting -> liquid shell (k_l); freezing -> frozen
    #                  shell (k_s). The physical arrangement in both directions.
    #   'melt_side'    the melted thickness always, with k_l whenever any melt
    #                  exists. Correct while melting, INVERTED while freezing.
    #                  Retained only to reproduce results published before the
    #                  correction; it is not a model.
    front_geometry: str = "directional"

    # ---- derived ----
    @property
    def glide_dc_spec(self):
        """Specified discharging glide; falls back to the charging glide."""
        return self.DT_3C_2C if self.DT_3D_2D is None else self.DT_3D_2D

    @property
    def cascade_param(self):
        """Cascade grading parameter; falls back to the charging glide."""
        return self.DT_3C_2C if self.DT_cascade is None else self.DT_cascade

    @property
    def cascade_span(self):
        """Top-to-bottom melting range, one layer short of cascade_param."""
        return self.cascade_param * (self.N_lay - 1) / self.N_lay

    @property
    def T_m(self):
        return self.T_m_C + 273.15

    @property
    def L_well(self):
        return self.L_ft * FT

    @property
    def L_tube(self):
        """Developed length of one fin tube.

        EQUAL TO L_well, where THUMS had twice it. The fin tubes only go
        down; the return is the downcomer and does not exchange. This single
        line is what makes z a depth rather than a path length, and so what
        lets a depth-dependent formation temperature be attached at all.
        """
        return self.L_well

    @property
    def r_id_bore(self):
        """Downcomer bore, sized so its velocity matches the fin tubes'.

        The downcomer carries the WHOLE flow that the n_ft tubes carry
        between them, so equal velocity means equal total area. That is the
        condition under which this geometry has the same pressure drop as the
        hairpin it replaces -- same velocity everywhere, same total path.
        """
        return self.r_i * np.sqrt(float(self.n_ft))

    @property
    def r_id_outer(self):
        """Downcomer outside radius: bore + wall + insulation."""
        wall = self.r_e - self.r_i
        return self.r_id_bore + wall + self.t_ins

    @property
    def D_well(self):
        return self.D_in * IN

    @property
    def V_borehole(self):
        return np.pi * self.D_well ** 2 / 4.0 * self.L_well

    @property
    def V_well(self):
        """PCM volume in one borehole: the hole, less tubes and fins."""
        A_tube = np.pi * self.r_e ** 2
        A_fins = self.num_fins * self.fin_t * self.fin_L
        A_id = np.pi * self.r_id_outer ** 2
        occupied = self.n_ft * (A_tube + A_fins) + A_id
        return self.V_borehole - occupied * self.L_well

    @property
    def rho_latent(self):
        """Density for the latent inventory, the SAME in both directions.

        The front is carried as a solid-equivalent area, so V_melt/V_well is a
        melted MASS fraction and the energy banked on charging equals the energy
        available on discharging. Using rho_l to melt and rho_s to freeze -- the
        directional choice that is correct for the conductivity k_m -- would
        return rho_s/rho_l = 1.069 times more energy than went in.
        """
        return self.rho_s

    @property
    def r_cell(self):
        """Radius of the equivalent PCM cell owned by ONE tube leg.

        The n_ft fin tubes share the PCM-bearing cross-section equally. The
        downcomer occupies area but exchanges no heat with the PCM it
        displaces, so it is SUBTRACTED before the division rather than
        counted as a participant -- a fin tube's cell is bounded by its
        neighbours and by the downcomer's outer wall, not by a share of it.

            pi r_cell^2 = (pi R^2 - pi r_id_outer^2) / n_ft

        It is NOT the borehole wall. With three legs in a 7 in hole r_cell is
        about 40 mm while the wall is at 88.90 mm, so a melt-layer limit
        imposed at the wall would be a factor 2 too permissive and therefore
        silent at every design point.
        """
        R = self.D_well / 2.0
        A_pcm_annulus = np.pi * (R ** 2 - self.r_id_outer ** 2)
        return np.sqrt(A_pcm_annulus / (np.pi * self.n_ft))

    @property
    def delta_merge(self):
        """Melt-layer thickness at which neighbouring fronts touch.

        Hypothesis H3 (concentric annulus, no interaction between legs) is a
        statement about delta < delta_merge. At and beyond this thickness the
        remaining solid sits in the corners between legs, reached through a
        constricted path that the annular resistance ln(1+delta/r_e)/(2 pi k)
        does not represent.

        Under Formulation C this is not a bound that can be VIOLATED --
        area_from_delta(delta_merge) equals A_avail identically, and the
        enthalpy cap holds A_melt <= A_avail -- so delta <= delta_merge always.
        It is a bound the solution can SIT ON, which is what it does here.
        """
        return self.r_cell - self.r_e

    def geom_vector(self):
        """The eleven positional geometry arguments the geometry helpers expect."""
        return [self.L_well, self.L_tube, self.D_well, self.r_i, self.r_e,
                2 * self.r_i, 2 * self.r_e, self.fin_t, self.fin_L,
                self.num_fins, self.n_ft]

    def stefan_number(self, dT):
        """Ste = cp dT / h_m. The quasi-steady melt layer degrades as this grows."""
        return self.cp_l * dT / self.h_m

    def with_(self, **kw):
        return replace(self, **kw)


CASE = Case()          # the design point


# ==========================================================================
# front_energy_balance   (the v0.2 correction)
# ==========================================================================


def delta_from_area(A_melt, r_e, n_f=0, t_f=0.0, L_f=0.0):
    """Invert `area_from_delta`. Closed form; the fins only shift the quadratic.

    For delta <= L_f the fin metal grows with the front:

        pi d^2 + (2 pi r_e - n_f t_f) d - A = 0

    and for delta > L_f it is constant at n_f t_f L_f:

        pi d^2 + 2 pi r_e d - (A + n_f t_f L_f) = 0

    With n_f = 0 both collapse to  d = sqrt(r_e^2 + A/pi) - r_e,  the bare
    concentric annulus used before this correction.
    """
    A = np.maximum(np.asarray(A_melt, dtype=float), 0.0)
    # branch 1: front still inside the fins
    b = 2.0 * np.pi * r_e - n_f * t_f
    d_in = (-b + np.sqrt(b * b + 4.0 * np.pi * A)) / (2.0 * np.pi)
    # branch 2: front beyond the fin tips
    d_out = np.sqrt(r_e ** 2 + (A + n_f * t_f * L_f) / np.pi) - r_e
    return np.where(d_in <= L_f, d_in, d_out)


# ==========================================================================
# march   (segment-by-segment integration down the tube)
# ==========================================================================

_PROP_CACHE = {}


def fluid_props(fluid, T, P):
    """Density, viscosity, conductivity, cp. Cached on 0.05 K bins."""
    key = (fluid, round(T * 20.0), round(P))
    hit = _PROP_CACHE.get(key)
    if hit is not None:
        return hit
    rho = CP.PropsSI("D", "T", T, "P", P, fluid)
    mu = CP.PropsSI("V", "T", T, "P", P, fluid)
    k = CP.PropsSI("L", "T", T, "P", P, fluid)
    cp = CP.PropsSI("C", "T", T, "P", P, fluid)
    _PROP_CACHE[key] = (rho, mu, k, cp)
    return rho, mu, k, cp


def h_internal(fluid, T, P, r_i, m_dot):
    """Internal convective coefficient. Gnielinski if turbulent, else laminar.

        Re = 4 m_dot / (pi D mu)
        f  = (0.79 ln Re - 1.64)^-2
        Nu = (f/8)(Re-1000) Pr / [1 + 12.7 sqrt(f/8) (Pr^(2/3) - 1)]
    """
    rho, mu, k, cp = fluid_props(fluid, T, P)
    D = 2 * r_i
    Re = 4.0 * m_dot / (np.pi * D * mu)
    Pr = cp * mu / k
    if Re < 2300:
        Nu = 3.66                      # fully developed, isothermal wall
    else:
        f = (0.79 * np.log(Re) - 1.64) ** -2
        Nu = (f / 8.0) * (Re - 1000) * Pr / (
            1 + 12.7 * np.sqrt(f / 8.0) * (Pr ** (2.0 / 3.0) - 1))
    return Nu * k / D, cp, Re, Pr


def layer_map(T_m_lay, n_segments, N_lay):
    """Melting temperature per segment, on the v0.1 node convention."""
    z = np.linspace(0.0, 1.0, n_segments + 1)
    z_lay = np.linspace(0.0, 1.0, N_lay + 1)
    T_m = np.empty(n_segments + 1)
    for j in range(N_lay):
        T_m[(z >= z_lay[j]) & (z < z_lay[j + 1])] = T_m_lay[j]
    T_m[-1] = T_m_lay[-1]
    return T_m[1:]


# ==========================================================================
# enthalpy state   (v0.4: sensible heat in both phases)
# ==========================================================================
# Formulation B tracks A_melt and nothing else, so a segment that runs out of
# PCM has nowhere to put further heat: the march clips it and discards the
# remainder. At the design point the discarded heat on discharge was 148% of
# the heat delivered -- i.e. most of what the network asked for.
#
# The fix is to carry ENTHALPY per unit tube length instead of melted area, with
# the melted area recovered from it. One scalar per segment, measured from the
# fully-solid-at-T_m datum:
#
#     E' < 0            solid, SUBCOOLED     T = T_m + E'/C_s
#     0 <= E' <= E'_lat two-phase            T = T_m,  A_melt = E'/(rho h_m)
#     E' > E'_lat       liquid, SUPERHEATED  T = T_m + (E'-E'_lat)/C_l
#
# Three things follow, and the third is the reason to do it:
#
#   1. no clipping anywhere -- heat always has somewhere to go;
#   2. the melt fraction is bounded in [0,1] BY CONSTRUCTION, so the
#      eps_local > 1 that the latent-only model produced cannot occur;
#   3. desuperheating and subcooling during discharge are recovered, which is
#      the energy the latent-only model threw away.
#
# delta still follows from A_melt exactly as before, so the heat-transfer
# coefficient is unchanged in form -- only its driving temperature is now
# T_pcm rather than T_m.


def pcm_capacities(case):
    """Per unit tube length: latent capacity and the two sensible capacities."""
    A_avail = case.V_well / (case.n_ft * case.L_tube)   # PCM area per tube [m2]
    E_lat = case.rho_latent * case.h_m * A_avail             # J/m to melt it all
    C_s = case.rho_latent * case.cp_s * A_avail              # J/m/K, solid
    C_l = case.rho_latent * case.cp_l * A_avail              # J/m/K, liquid
    return A_avail, E_lat, C_s, C_l


def pcm_state(E, T_m_seg, case):
    """Enthalpy per unit tube length -> (melted area, PCM temperature).

    The inverse of the enthalpy curve above. `A_melt` saturates at the segment's
    own PCM content, which is what makes eps_local <= 1 structural.
    """
    A_avail, E_lat, C_s, C_l = pcm_capacities(case)
    T_m_seg = np.asarray(T_m_seg, dtype=float)
    A = np.clip(E, 0.0, E_lat) / (case.rho_latent * case.h_m)
    if not case.sensible_heat:                    # v0.3 behaviour: PCM pinned at T_m
        return A, T_m_seg.copy()
    T = np.where(E < 0.0, T_m_seg + E / C_s,
                 np.where(E > E_lat, T_m_seg + (E - E_lat) / C_l, T_m_seg))
    return A, T


def advance_segment(E, T_m, T0, K, dt, E_lat, C_s, C_l, sensible):
    """Integrate ONE segment's enthalpy over dt, exactly, branch by branch.

    The segment obeys   dE'/dt = K (T0 - T_pcm(E'))   with T0 held over the step.
    That is linear inside each branch of the enthalpy curve, so it can be solved
    in closed form rather than stepped:

      plateau  T_pcm = T_m constant           -> dE'/dt constant, E' linear in t
      solid    T_pcm = T_m + E'/C_s           -> exponential relaxation to
      liquid   T_pcm = T_m + (E'-E_lat)/C_l      E'_eq = C (T0 - T_m) [+E_lat]

    Explicit Euler is NOT usable here. On the latent plateau the segment has
    effectively infinite heat capacity, so any step is stable; the moment
    sensible heat is added the capacity becomes finite and the stability limit
    is dt < C/K, about 620 s at the design point. The time grid's last step is
    8491 s, fourteen times that, and an explicit update diverges -- it drove the
    secondary fluid to 261 K on the first attempt.

    Returns (E_new, q_avg) with q_avg = (E_new - E)/dt, so the heat taken from
    the fluid is exactly the enthalpy the PCM gained: closure stays structural.
    """
    if K <= 0.0 or dt <= 0.0:
        return E, 0.0
    E0, t_left = E, dt
    for _ in range(6):   # at most a few branch crossings
        if t_left <= 0.0:
            break
        if not sensible:                     # v0.3: PCM pinned at T_m
            E = E + K * (T0 - T_m) * t_left
            break
        if 0.0 <= E <= E_lat:                # ---- latent plateau: linear ----
            rate = K * (T0 - T_m)
            if rate == 0.0:
                break
            t_edge = ((E_lat - E) / rate) if rate > 0 else (E / -rate)
            if t_edge >= t_left:
                E = E + rate * t_left
                break
            # Land on the boundary and step just PAST it. Without the nudge the
            # next iteration re-enters the plateau at zero rate and the segment
            # sticks at E_lat for ever -- which showed up as every segment
            # reporting a melt fraction of exactly 1.000 and no superheat.
            eps = 1e-9 * max(E_lat, 1.0)
            E = (E_lat + eps) if rate > 0 else -eps
            t_left -= t_edge
        else:                                # ---- sensible: exponential ----
            if E < 0.0:
                C, E_ref, lo, hi = C_s, 0.0, -np.inf, 0.0
            else:
                C, E_ref, lo, hi = C_l, E_lat, E_lat, np.inf
            E_eq = E_ref + C * (T0 - T_m)
            E_new = E_eq + (E - E_eq) * np.exp(-K * t_left / C)
            if lo <= E_new <= hi:
                E = E_new
                break
            edge = hi if E_new > hi else lo   # crosses back onto the plateau
            num, den = E_eq - edge, E_eq - E
            if den == 0.0 or num / den <= 0.0:
                E = edge
                break
            t_edge = -C / K * np.log(num / den)
            if t_edge >= t_left or t_edge <= 0.0:
                E = E_new
                break
            E = edge
            t_left -= t_edge
    return E, (E - E0) / dt


def segment_profile(case, T_inlet, T_m_seg, m_dot, k_wall, E, n_segments=None):
    """Instantaneous fluid and conductance profile for a frozen PCM state.

    The inner loop of `march_h` without the time advance: it evaluates the
    resistance network, the segment conductance and the fluid temperature
    profile for the state `E` as it stands, and returns the instantaneous
    q' = K (T_f - T_pcm).

    It exists so that the last recorded frame of a march can carry a fluid
    profile belonging to the *end* state rather than to the state one step
    earlier. Everything it computes is already computed inside `march_h`;
    this is one extra sweep per recorded march, not one per time level.
    """
    n_segments = n_segments or len(E)
    dz = case.L_tube / n_segments
    A, T_pcm = pcm_state(E, T_m_seg, case)
    delta = delta_from_area(A, case.r_e, case.num_fins, case.fin_t, case.fin_L)
    A_avail, E_lat = pcm_capacities(case)[:2]
    d_melt, k_melt = conduction_shell(case, A, A_avail, True, E, E_lat)
    d_froz, k_froz = conduction_shell(case, A, A_avail, False, E, E_lat)

    T_prof = np.empty(n_segments + 1)
    T_prof[0] = T_inlet
    NTU_prof = np.empty(n_segments)
    U_prof = np.empty(n_segments)
    q_prime = np.empty(n_segments)

    T0 = T_inlet
    for i in range(n_segments):
        melting = T0 > T_pcm[i]
        d_path = float(d_melt[i] if melting else d_froz[i])
        k_m = float(k_melt[i] if melting else k_froz[i])
        h_i, cp_d, _, _ = h_internal(case.fluid2, T0, case.P, case.r_i, m_dot)
        U_i = compute_U_i(h_i, case.r_i, case.r_e, k_wall, case.Rf_i, k_m,
                          d_path, case.L_tube, case.fin_t,
                          case.fin_L, case.num_fins)
        NTU = float(np.clip((2 * np.pi * case.r_i * U_i * dz) / (m_dot * cp_d),
                            -50.0, 50.0))
        K = m_dot * cp_d * (1.0 - np.exp(-NTU)) / dz
        q_prime[i] = K * (T0 - T_pcm[i])
        T0 = T0 - q_prime[i] * dz / (m_dot * cp_d)
        T_prof[i + 1] = T0
        NTU_prof[i] = NTU
        U_prof[i] = U_i

    return dict(T_fluid=T_prof, NTU=NTU_prof, U_i=U_prof, q_prime=q_prime,
                A_melt=A, T_pcm=T_pcm, delta=delta, E=np.array(E, float))


def bulk_shape_factor(case):
    """Dimensionless resistance of a SINGLE-PHASE cell, mean to tube surface.

    A cell that is entirely one phase has no front. Heat entering at the tube is
    stored throughout its volume, so the temperature varies with radius and the
    lumped state T_pcm is the VOLUME MEAN. The resistance that belongs with that
    mean is not the surface-to-surface annulus value.

    Take the cell as an annulus r_e <= r <= r_o, adiabatic at r_o (its neighbour
    is identical, so the boundary is a symmetry plane), storing sensible heat at
    a uniform volumetric rate s = rho c dT/dt. Quasi-steady, the heat crossing
    radius r is what the material beyond it stores,

        -k 2 pi r dT/dr = s pi (r_o^2 - r^2),

    which integrates to

        T(r_e) - T(r) = (s/2k) [ r_o^2 ln(r/r_e) - (r^2 - r_e^2)/2 ].

    Averaging over the volume and dividing by the total Q' = s pi (r_o^2-r_e^2)
    gives R'_bulk = S/(2 pi k) with, writing beta = r_o/r_e,

        S = [ beta^4 ln(beta) - beta^4/2 + beta^2/2 - (beta^2-1)^2/4 ]
            / (beta^2 - 1)^2 .

    S depends on GEOMETRY ALONE -- k cancels -- so one shape factor serves both
    phases and only the conductivity changes between them. For the default cell,
    beta = 2.1086 and S = 0.34672, against ln(beta) = 0.74604 for the
    surface-to-surface annulus: the mean-to-surface resistance is smaller by a
    factor 2.152, the cylindrical analogue of the familiar 1/3 for a slab with
    uniform generation.

    Verified against numerical quadrature of the same integral to six decimals.
    """
    beta = case.r_cell / case.r_e
    b2 = beta * beta
    return ((b2 * b2 * np.log(beta) - b2 * b2 / 2.0 + b2 / 2.0
             - (b2 - 1.0) ** 2 / 4.0) / (b2 - 1.0) ** 2)


def bulk_equivalent_delta(case):
    """The shell thickness that reproduces the bulk resistance.

    The network in `compute_U_i` takes a thickness, not a resistance, and wraps
    it in the fin efficiency and the finned-area ratio. Feeding it an equivalent
    thickness therefore keeps the single-phase branches inside exactly the same
    machinery as the two-phase ones, rather than bolting a second path onto it:

        ln(1 + delta_eq/r_e) = S    =>    delta_eq = r_e (e^S - 1).

    Like S it is a property of the geometry alone -- the same delta_eq serves
    subcooled solid and superheated liquid, and only k changes. For the default
    cell it is 8.74 mm, against 0 and 23.37 mm, which are the two values the
    model used before this was worked out.
    """
    return case.r_e * (np.exp(bulk_shape_factor(case)) - 1.0)


def conduction_shell(case, A_melt, A_avail, melting, E=None, E_lat=None):
    """Thickness and conductivity of the shell between the tube and the front.

    The PCM-side resistance in `compute_U_i` is that of an annulus growing
    OUTWARD FROM THE TUBE WALL,

        R' = ln(1 + delta/r_e) / (2 pi k),

    so the question this function answers is: which material is that annulus
    made of, and how thick is it?

    It depends on the direction of the phase change, because the front always
    grows away from the tube -- the tube is the driven boundary in both
    half-cycles.

      MELTING (fluid hotter than the PCM). Melting begins at the tube wall and
      the front moves outward, so the shell is LIQUID and its thickness is that
      of the melted area. Heat crosses the melt to reach the remaining solid.

      FREEZING (fluid colder). Solidification also begins at the tube wall, so
      the shell is the FROZEN material and its thickness is that of the solid
      area, A_avail - A_melt. The liquid is displaced outward, beyond the front,
      and is no longer in the conduction path at all.

    All four limits come out right without special-casing:

      fully solid, melting     A_melt = 0            -> delta = 0, no resistance
      fully melted, melting    A_melt = A_avail      -> the full liquid annulus
      fully melted, freezing   A_avail - A_melt = 0  -> delta = 0, no resistance
      fully solid, freezing    A_avail - A_melt = A  -> the full frozen annulus

    The last of those is the one that matters. Until v0.7 the melted thickness
    was used in BOTH directions, which put the conduction path on the wrong
    side of the front during discharge: a segment that had frozen solid was
    given ZERO PCM-side resistance, exactly where the physical resistance is
    largest. The consequence was that the modelled exchanger IMPROVED as it
    froze, when it should degrade. See DN-13 for what that was worth.

    LIMIT OF ANY LUMPED FRONT. One thickness can describe one front. After the
    first half-cycle from a fully solid store the geometry is generally
    three-region -- at cyclic steady state the charge begins with residual
    liquid left in the OUTER part of the cell, so melting produces liquid at the
    tube, solid in the middle and liquid outside. Neither this function nor the
    retired one can represent that; this one is the better approximation, not a
    correct treatment. It is hypothesis H5, stated honestly.
    """
    if case.front_geometry not in ("directional", "melt_side"):
        raise ValueError(f"front_geometry must be 'directional' or "
                         f"'melt_side', not {case.front_geometry!r}")

    A_melt = np.asarray(A_melt, float)
    if case.front_geometry == "melt_side":          # retired, see above
        return (delta_from_area(A_melt, case.r_e, case.num_fins, case.fin_t,
                                case.fin_L),
                np.full(A_melt.shape, case.k_l))

    # ---- two-phase: the shell between the tube and the front --------------
    A_path = A_melt if melting else (A_avail - A_melt)
    delta = delta_from_area(A_path, case.r_e, case.num_fins, case.fin_t,
                            case.fin_L)
    k_path = np.full(A_melt.shape, case.k_l if melting else case.k_s)

    # ---- single phase: there is no front, so there is no side ------------
    # `E` is optional only so that callers which do not have it (none, now)
    # keep the two-phase behaviour; when it is given, the single-phase cells
    # get the bulk resistance of `bulk_shape_factor`, the SAME in both
    # directions, because a one-phase cell has no front and cannot care which
    # way the heat is going.
    if E is not None:
        E = np.asarray(E, float)
        d_bulk = bulk_equivalent_delta(case)
        solid = E < 0.0                              # subcooled solid
        liquid = E > E_lat                           # superheated liquid
        delta = np.where(solid | liquid, d_bulk, delta)
        k_path = np.where(solid, case.k_s,
                          np.where(liquid, case.k_l, k_path))
    return delta, k_path


def mixed_mean_outlet(res):
    """Flow-mixed mean outlet temperature of a march [K].

    The temperature at which the stream returns to the plant, averaged over the
    half-cycle. At constant mass flow and c_p the mixing average reduces to the
    time average, which on a logarithmic grid still has to be weighted by the
    step -- the plain mean of `T_out` is dominated by the early levels and is
    wrong by several kelvin.

    This is the quantity the ORC actually sees, and since v0.6 it is a RESULT
    rather than a prescribed state point (`cycle_state_points`).
    """
    t = np.asarray(res["t"], float)
    dt = np.diff(np.concatenate(([0.0], t)))
    return float(np.sum(np.asarray(res["T_out"], float) * dt) / np.sum(dt))


def unmirror_march(res):
    """Re-index a reversed-flow march result into charge (depth) coordinates.

    Discharge marches from the opposite end of the tube, so its segment index
    runs backwards relative to the charge. `run_cycle` and `simulate_css` hand
    it a mirrored cascade and a mirrored initial state and un-mirror the result
    when they close the loop; anything that *plots* a discharge result has to
    do the same or it draws the well upside down.

    Returns a shallow copy with every per-segment array reversed, including the
    recorded history. `z`, `t` and `t_hist` are coordinates, not fields, and
    are left alone. Scalars are left alone. The cascade in depth coordinates is
    the same for both half-cycles, which is the point: after this call a
    discharge result is plotted against `T_m_seg`, not against `T_m_seg[::-1]`.
    """
    n = len(res["E"])
    out = dict(res)
    for k in ("E", "A_melt", "T_pcm", "delta", "eps_local", "merge_proximity"):
        if k in out and out[k] is not None:
            out[k] = np.asarray(out[k])[::-1].copy()
    h = res.get("history")
    if h:
        hh = {}
        for k, v in h.items():
            v = np.asarray(v)
            # T_fluid has n+1 nodes (inlet plus one per segment); the rest have n
            hh[k] = v[:, ::-1].copy() if v.ndim == 2 and v.shape[1] in (n, n + 1) else v
        out["history"] = hh
    return out


# ==========================================================================
# the formation   (BOREAS v1.0 -- THUMS hypothesis H6 said this was adiabatic)
# ==========================================================================
# H6 made eta_storage identically 1. Not approximately 1: EXACTLY 1, by
# construction, because nothing could leave. That is an indefensible headline
# for a borehole storage paper and it is the first thing a reviewer attacks.
#
# The resistances are wildly unequal and it is worth knowing which one is the
# model. At the design point, per metre of borehole:
#
#     casing steel        0.0003 m K/W     0.1 %
#     cement              0.018            5 %
#     PCM to casing       ~0.12            25 %
#     GROUND              0.32             ~70 %
#
# Everything near the well is a rounding error. The ground resistance is the
# physics, and unlike the others it GROWS WITH TIME as the thermal front
# spreads -- which is why `t_operation_yr` is a Case parameter and why a
# single quoted loss figure is meaningless without the age that goes with it.
#
# THE SIGN IS NOT ALWAYS NEGATIVE. With a 45 K/km gradient the rock at 8000 ft
# is near 128 C, and at the Wilmington gradient of 56.5 K/km it is near 156 C
# -- hotter than the coldest cascade layers. Those segments GAIN heat from the
# formation. A model that assumes loss and takes an absolute value somewhere
# would hide that, so nothing here does.


def ground_resistance(case):
    """Quasi-steady line-source resistance of the formation, per metre.

    Eskilson's form: at time t the disturbance has spread to roughly
    2.5*sqrt(alpha t), and the resistance is the log of that over the borehole
    radius. It is quasi-steady in the sense that the DAILY cycle is far faster
    than the formation's response, so within one charge-discharge the ground
    looks like a fixed resistance to a fixed far-field temperature. The cycle
    time is 10 h and the formation time constant is years; the separation is
    four orders of magnitude, so this is a good approximation rather than a
    convenient one.

    SINGLE WELL, NO FIELD SHIELDING. This is deliberate and it is the
    conservative choice: an isolated well is the worst case, because in a
    field the neighbours hold the ground between them warm and only the
    perimeter leaks. The shielding factor is 29x to 42x, essentially
    independent of the driving temperature and only weakly dependent on rock
    conductivity, so a field result can be obtained by dividing. Nothing here
    should be read as the loss a real cluster would see.
    """
    if case.k_rock <= 0.0:
        return np.inf                      # no formation: adiabatic wall
    alpha = case.k_rock / case.rho_cp_rock
    t = case.t_operation_yr * 365.25 * 24.0 * 3600.0
    arg = 2.5 * np.sqrt(alpha * t) / case.r_bore
    R = np.log(arg) / (2.0 * np.pi * case.k_rock) if arg > 0.0 else -1.0
    if not np.isfinite(R) or R <= 0.0:
        # THE LINE SOURCE HAS BROKEN DOWN, and the branch matters.
        # An earlier version returned 0.0 here, meaning the front "has not
        # cleared the hole yet". Zero RESISTANCE is infinite CONDUCTANCE: it
        # says the rock is a perfect heat sink, which is the opposite of what
        # was meant and the most damaging possible failure of this function.
        # It was caught the first time the adiabatic limit was watched -- a
        # test that drives k_rock to zero and expects eta_storage = 1 got
        # eta_storage = 0.20 and a LARGER loss than the physical case.
        #
        # Infinity is the right fallback: outside its validity the line
        # source is not evidence of a loss, and `validate_case` warns rather
        # than letting a silent number through.
        return np.inf
    return R


def field_radius(case, depth):
    """Radius of the well cluster at a given depth [m].

    Vertical above the kickoff point, then fanning at a constant effective
    half-angle. A real directional programme builds angle over a section and
    then holds it, so the fan is not a straight cone -- but the quantity this
    feeds is a logarithm, and the difference between a cone and a build-and-
    hold is well inside the uncertainty in the spacing data itself.
    """
    if case.n_wells_field is None:
        return None
    R0 = case.B_surface_m * np.sqrt(case.n_wells_field / np.pi)
    spread = np.maximum(0.0, np.asarray(depth, float) - case.z_kickoff_m)
    return R0 + spread * np.tan(np.radians(case.fan_angle_deg))


def well_spacing(case, depth):
    """Centre-to-centre spacing at depth, scaled with the fan radius [m]."""
    Rf = field_radius(case, depth)
    if Rf is None:
        return None
    R0 = case.B_surface_m * np.sqrt(case.n_wells_field / np.pi)
    return case.B_surface_m * Rf / R0


def formation_resistance(case, depth=None):
    """Total series resistance, PCM bulk to undisturbed rock [m K / W].

    Per metre of BOREHOLE, not per metre of tube. The PCM-to-casing term is
    the crude one: n_ft parallel paths from the tube walls out to the casing,

        R_pcm = ln(R_casing_i / r_e) / (2 pi n_ft k_pcm)

    which ignores that the paths crowd together near the wall. It is an
    approximation and it is flagged as one -- but it is a quarter of a total
    that the ground dominates, so refining it would move the answer less than
    the uncertainty in k_rock does.
    """
    R_in = case.D_well / 2.0
    k_pcm = 0.5 * (case.k_s + case.k_l)
    R_pcm = np.log(R_in / case.r_e) / (2.0 * np.pi * case.n_ft * k_pcm)
    R_steel = np.log(case.r_casing_o / R_in) / (2.0 * np.pi * case.k_wall)
    R_cem = np.log(case.r_bore / case.r_casing_o) / (2.0 * np.pi * case.k_cement)
    R_g = ground_resistance(case)
    if not np.isfinite(R_g):
        return np.inf                      # adiabatic; K_loss becomes 0
    R_near = R_pcm + R_steel + R_cem

    # ---- FIELD SHIELDING -------------------------------------------------
    # A well inside a cluster cannot lose what an isolated one would: its
    # neighbours hold the rock between them warm, and once the cells have
    # merged the only heat leaving is across the cluster PERIMETER, shared
    # among N wells. Writing that as a resistance,
    #
    #     R_perimeter(z) = N sqrt(pi alpha t) / (2 pi k R_field(z))
    #
    # -- the semi-infinite-slab flux at the boundary, over the perimeter the
    # field presents, divided by the wells that share it. It carries no
    # driving temperature, so it is a true resistance and drops straight into
    # the series chain.
    #
    # The two regimes are combined by taking the LARGER resistance. That is
    # not a smooth blend and does not pretend to be: a well cannot leak more
    # than if it stood alone, and a field cannot leak more than its perimeter
    # allows, so whichever constraint binds is the one that governs. The
    # crossover is sharp in reality too -- it is the moment the cells merge.
    if depth is None or case.n_wells_field is None:
        return R_near + R_g
    alpha = case.k_rock / case.rho_cp_rock
    t = case.t_operation_yr * 365.25 * 24.0 * 3600.0
    Rf = field_radius(case, depth)
    R_per = (case.n_wells_field * np.sqrt(np.pi * alpha * t)
             / (2.0 * np.pi * case.k_rock * Rf))
    return R_near + np.maximum(R_g, R_per)


def formation_profile(case, n_segments=None, discharge=False):
    """Undisturbed rock temperature [K] at each segment, and its depth [m].

    THE COORDINATE. In BOREAS z is DEPTH, which it could not be in THUMS: the
    hairpin's two legs occupied every depth twice, so one rock temperature
    would have had two PCM segments claiming it. Here the fin tubes only go
    down and the downcomer does not exchange, so the map is one-to-one.

    On CHARGE the water enters the fin tubes at the BOTTOM -- it arrives there
    down the downcomer -- so the solver's z = 0 is the bottom of the well and
    depth decreases with z. On DISCHARGE the flow reverses and the march is
    run mirrored, so the profile is mirrored with it, exactly as T_m_seg is.
    Getting this backwards would put the hot cascade layers against the cold
    shallow rock, which is the THUMS orientation and the thing BOREAS exists
    to avoid -- so it is asserted in `validate_case`, not left to a comment.
    """
    n = n_segments or case.n_segments
    xi = (np.arange(n) + 0.5) / n                 # cell centres, 0 at inlet
    depth = case.L_well * (1.0 - xi)              # charge frame: z=0 is bottom
    T_rock = case.T_surface_C + 273.15 + case.grad_K_per_km / 1000.0 * depth
    if discharge:
        return T_rock[::-1].copy(), depth[::-1].copy()
    return T_rock, depth


def march_h(case, T_inlet, T_m_seg, m_dot, k_wall, times,
            n_segments=None, E0=None, record=False, T_rock=None,
            R_form=None):
    """Segment march on the ENTHALPY state. Replaces `march` from v0.4.

    Identical to `march` in every respect except what is integrated: the state
    is E' [J per m of tube] rather than A_melt [m2], so no heat is ever clipped
    and the PCM temperature is free to leave T_m.

    Recording convention. Every history frame is stamped, in `t_hist`, with the
    instant at which its state arrays (`E`, `A_melt`, `T_pcm`, `delta`) are
    exact. The profile arrays (`T_fluid`, `NTU`, `U_i`, `q_prime`) in the same
    frame are the ones evaluated from that state, i.e. the profile that drives
    the step *beginning* there. There are `len(t)+1` frames: one at t = 0 and
    one at the end of each accepted step, so the final state is in the record.

    Earlier builds recorded the pre-step state but stamped it with the
    post-step time, which on a logarithmic grid put the last frame a quarter of
    a charge behind its label, and appended `E` after the update while the
    other state arrays were from before it. Neither affected any marched or
    reported quantity -- `Q_cum`, `closure`, `eps_local` and the returned state
    all come from the march itself -- but every recorded plot was lagged.
    """
    n_segments = n_segments or case.n_segments
    dz = case.L_tube / n_segments
    r_i, r_e = case.r_i, case.r_e
    A_avail, E_lat, C_s, C_l = pcm_capacities(case)

    # ---- the formation sink ---------------------------------------------
    # K_loss is a conductance per unit TUBE length, so the per-borehole
    # conductance is divided among the n_ft tubes that share each depth.
    # Passing T_rock=None recovers the THUMS adiabatic wall exactly, which is
    # how the v0.16 reproduction test is run.
    if T_rock is None:
        K_loss = np.zeros(n_segments)
        T_rock_arr = np.zeros(n_segments)
    else:
        R_tot = (formation_resistance(case) if R_form is None
                 else np.asarray(R_form, float))
        R_arr = np.full(n_segments, R_tot) if np.ndim(R_tot) == 0 else R_tot
        K_loss = np.where(np.isfinite(R_arr), 1.0 / (R_arr * case.n_ft), 0.0)
        K_loss = np.nan_to_num(K_loss, posinf=0.0, neginf=0.0)
        T_rock_arr = np.asarray(T_rock, float)
    Q_loss_cum = 0.0
    # ---- A CLAUSIUS GATE ON THE LOSS TERM --------------------------------
    # The closure CANNOT catch a flipped loss sign, and this was watched:
    # negating q_loss leaves the residual at 5e-15, because q_prime and
    # Q_loss_cum both carry the error and it cancels algebraically. The
    # closure is self-consistent, which is the DN-26 trap -- a check reading
    # only its own inputs.
    #
    # This one reads the STATE instead. Heat must flow from hot to cold, so
    # q_loss and (T_pcm - T_rock) must share a sign at every segment and
    # every step. The product is accumulated and gated on being non-negative.
    # With the correct convention it is K (dT)^2 and cannot be negative; with
    # the sign flipped it is -K (dT)^2 and cannot be positive. That is the
    # independent input the closure lacks.
    clausius = 0.0

    E = np.zeros(n_segments) if E0 is None else np.array(E0, float)
    E_start = E.copy()
    t_prev, Q_cum = 0.0, 0.0
    ts, Qs, T_outs = [], [], []
    hist = {k: [] for k in ("T_fluid", "T_pcm", "A_melt", "delta", "NTU",
                            "U_i", "q_prime", "E")} if record else None
    t_hist = []

    for t in np.asarray(times, float):
        dt = t - t_prev
        if dt <= 0:
            t_prev = t
            continue

        A, T_pcm = pcm_state(E, T_m_seg, case)
        # `delta` stays the MELT thickness: it is what the H3 merge diagnostic
        # is about. The conduction path is a separate question -- which side of
        # the front the heat has to cross -- and depends on the direction.
        delta = delta_from_area(A, r_e, case.num_fins, case.fin_t, case.fin_L)
        d_melt, k_melt = conduction_shell(case, A, A_avail, True, E, E_lat)
        d_froz, k_froz = conduction_shell(case, A, A_avail, False, E, E_lat)
        T0 = T_inlet
        q_prime = np.zeros(n_segments)
        if record:
            E_before = E.copy()          # the state this frame is stamped at
            T_prof = np.empty(n_segments + 1); T_prof[0] = T_inlet
            NTU_prof = np.empty(n_segments); U_prof = np.empty(n_segments)

        for i in range(n_segments):
            # Which shell the heat crosses depends on which way the front is
            # moving; the sign of the driving difference settles it before K is
            # known, and it has the same sign as q'.
            melting = T0 > T_pcm[i]
            d_path = float(d_melt[i] if melting else d_froz[i])
            k_m = float(k_melt[i] if melting else k_froz[i])
            h_i, cp_d, _, _ = h_internal(case.fluid2, T0, case.P, r_i, m_dot)
            U_i = compute_U_i(h_i, r_i, r_e, k_wall, case.Rf_i, k_m,
                              d_path, case.L_tube, case.fin_t,
                              case.fin_L, case.num_fins)
            NTU = float(np.clip((2 * np.pi * r_i * U_i * dz) / (m_dot * cp_d),
                                -50.0, 50.0))
            # conductance of the segment to the fluid, per unit tube length:
            # q' = K (T0 - T_pcm), from the NTU effectiveness
            K = m_dot * cp_d * (1.0 - np.exp(-NTU)) / dz

            # TWO linear sinks on one node, combined EXACTLY rather than
            # split operator. For any T_pcm,
            #     K (T0 - T) + K_l (T_rock - T) = (K + K_l) (T_eq - T)
            # with T_eq the conductance-weighted mean, so `advance_segment`
            # is called unchanged and remains implicit in T_pcm. Adding the
            # loss explicitly after the step would have been a half-order
            # scheme on a term that can reverse sign down the well.
            K_l = K_loss[i]
            K_eq = K + K_l
            T_eq = (K * T0 + K_l * T_rock_arr[i]) / K_eq if K_eq > 0 else T0
            E[i], q_tot = advance_segment(E[i], T_m_seg[i], T_eq, K_eq, dt,
                                          E_lat, C_s, C_l,
                                          case.sensible_heat)
            # Recover the split. q_tot is what the PCM NET received, so the
            # effective PCM temperature over the step follows from it, and
            # the two streams separate without any further assumption.
            if K_l > 0.0:
                T_eff = T_eq - q_tot / K_eq
                q_loss = K_l * (T_eff - T_rock_arr[i])   # +ve = leaving
                Q_loss_cum += q_loss * dz * dt
                clausius += q_loss * (T_eff - T_rock_arr[i]) * dz * dt
            else:
                q_loss = 0.0
            q_prime[i] = q_tot + q_loss                  # what the TUBE gave
            # the fluid gives up exactly what the PCM took FROM IT -- the
            # formation term is not the fluid's to pay
            T0 = T0 - q_prime[i] * dz / (m_dot * cp_d)
            if record:
                T_prof[i + 1] = T0; NTU_prof[i] = NTU; U_prof[i] = U_i

        if record:
            # stamped at t_prev, where E_before / A / T_pcm / delta are exact
            t_hist.append(t_prev)
            hist["T_fluid"].append(T_prof.copy()); hist["T_pcm"].append(T_pcm.copy())
            hist["A_melt"].append(A.copy());       hist["delta"].append(delta.copy())
            hist["NTU"].append(NTU_prof.copy());   hist["U_i"].append(U_prof.copy())
            hist["q_prime"].append(q_prime.copy()); hist["E"].append(E_before)

        Q_cum += float(np.sum(q_prime) * dz) * dt
        ts.append(t); Qs.append(float(np.sum(q_prime) * dz)); T_outs.append(T0)
        t_prev = t

    if record:
        # the end state, which the loop above never gets to record
        end = segment_profile(case, T_inlet, T_m_seg, m_dot, k_wall, E,
                              n_segments=n_segments)
        t_hist.append(t_prev)
        for k in hist:
            hist[k].append(end[k])

    A, T_pcm = pcm_state(E, T_m_seg, case)
    dE = float(np.sum(E - E_start) * dz)
    delta_end = delta_from_area(A, r_e, case.num_fins, case.fin_t, case.fin_L)

    # ---- H3 proximity ----------------------------------------------------
    # delta can never EXCEED delta_merge here (see Case.delta_merge), so the
    # useful diagnostic is not a violation flag but how close to the limit the
    # solution runs, and over how much of the well. The annular conduction
    # resistance is least trustworthy where this approaches 1.
    d_merge = case.delta_merge
    prox = delta_end / d_merge if d_merge > 0 else np.full_like(delta_end, np.nan)

    return {
        "E": E, "A_melt": A, "T_pcm": T_pcm,
        "delta": delta_end,
        "Q_cum_J": Q_cum,
        "Q_loss_J": Q_loss_cum,
        # >= 0 always, by the second law. Negative means the loss term is
        # pushing heat uphill, which is the sign error the closure misses.
        "loss_clausius": clausius,
        # THE CLOSURE NOW HAS THREE TERMS. In THUMS it was
        # |Q_cum - dE| / |Q_cum|: everything the tube gave went into storage,
        # because nothing could leave. Here what the tube gives either stays
        # or goes to the rock, so the identity is Q_tube = dE + Q_loss and the
        # residual is what is left over. This is still a bookkeeping identity
        # rather than a gate -- it cannot fail without an arithmetic bug --
        # but it would have caught the sign error of dropping q_loss from
        # q_prime, which is the mistake this split invites.
        "closure": (abs(Q_cum - dE - Q_loss_cum) / abs(Q_cum)
                    if Q_cum else np.nan),
        "t": np.array(ts), "Q": np.array(Qs), "T_out": np.array(T_outs),
        "V_melt_tube": float(np.sum(A * dz)),
        "eps_local": A / A_avail,
        "delta_merge": d_merge,
        "merge_proximity": prox,
        "merge_proximity_max": float(np.max(prox)),
        "f_near_merge": float(np.mean(prox > 0.90)),
        "E_sensible_J": float(np.sum(np.where(E > E_lat, E - E_lat,
                                              np.where(E < 0, E, 0.0))) * dz),
        "z": np.linspace(0.0, case.L_tube, n_segments),
        "history": {k: np.array(v) for k, v in hist.items()} if record else None,
        "t_hist": np.array(t_hist) if record else None,
    }


# ==========================================================================
# cycles + energy budget
# ==========================================================================

def T_m_bottom(case):
    """Melting temperature of the COLDEST cascade layer [K].

    The cascade is graded so that each layer sits DT_4C_M below the charging
    fluid at its own leading face, under a linear glide. The layers are spaced
    DT_3C_2C / N_lay apart, so the span of the melting temperatures is one
    layer width SHORT of the glide:

        T_m,bottom = T_m,top - DT_3C_2C * (N_lay - 1) / N_lay

    For the default case that is 150 - 55*8/9 = 101.111 C against a 55 K glide.
    The discharge enters the exchanger at this end, which is why this is the
    reference for DT_M_1D and not T_m,top.
    """
    return case.T_m - case.cascade_span


def _orc_cold_composite(rank, fluid, n=300):
    """The ORC-side composite curve of the evaporator: (Q, T), cold end first.

    Two cold streams share the water:

      * the MAIN stream, 1 kg per kg of evaporator flow, from state 10e to
        state 3e at p_evape -- liquid preheat, then boiling at the constant
        temperature T_3e;
      * the REHEAT stream, y_frac of the flow, from 5e to 6e at p_inte,
        superheated vapour throughout.

    Both are built on real enthalpy, so the latent plateau appears as a
    vertical segment rather than being smeared into a straight line in T.
    That matters: the plateau is where this evaporator pinches.
    """
    y = rank["y_frac"]
    p_e, p_i = rank["p_evape"], rank["p_5e"]
    h10, h3 = rank["h_10e"], rank["h_3e"]
    h5, h6 = rank["h_5e"], rank["h_6e"]
    T10, T3 = rank["T_10e"], rank["T_3e"]
    T5, T6 = rank["T_5e"], rank["T_6e"]
    h_bub = CP.PropsSI("H", "P", p_e, "Q", 0, fluid) / 1000.0

    # sample in ENTHALPY, not temperature: a (T, p) call exactly on the
    # saturation line raises, and the preheat ends exactly there
    h_pre = np.linspace(h10, h_bub, n)
    T_pre = np.array([CP.PropsSI("T", "H", x * 1000.0, "P", p_e, fluid)
                      for x in h_pre])
    h_rh = np.linspace(h5, h6, n)
    T_rh = np.array([CP.PropsSI("T", "H", x * 1000.0, "P", p_i, fluid)
                     for x in h_rh])

    def Q_main(t):
        if t <= T10:
            return 0.0
        if t >= T3:
            return h3 - h10                      # preheat AND latent
        return float(np.interp(t, T_pre, h_pre)) - h10

    def Q_reheat(t):
        if t <= T5:
            return 0.0
        if t >= T6:
            return y * (h6 - h5)
        return y * (float(np.interp(t, T_rh, h_rh)) - h5)

    lo, hi = min(T10, T5), max(T3, T6)
    grid = set(np.linspace(lo, hi, 4 * n))
    grid |= {T10, T5, T6, T3, np.nextafter(T3, lo)}   # break points
    Ts = np.array(sorted(t for t in grid if lo <= t <= hi))
    Qs = np.array([Q_main(t) + Q_reheat(t) for t in Ts])
    keep = np.concatenate(([True], np.diff(Qs) > 1e-12))
    return Qs[keep], Ts[keep]


def orc_pinch(rank, T_w_hot, T_w_cold, fluid, P_water, n=600):
    """Smallest water-minus-ORC temperature difference in the evaporator [K].

    Negative when the two composite curves cross, which is a second-law
    violation: the reported ORC efficiency is then unreachable, whatever the
    component efficiencies.

    The water is the hot stream and follows REAL enthalpy at `P_water`, not a
    straight line in temperature, and the search runs over an even grid in Q
    that includes every break point. The pinch is found wherever it lies; it
    is emphatically not assumed to sit at an end. For the case as shipped in
    v0.8 it sat at 23 % of the duty, where boiling begins.
    """
    Qc, Tc = _orc_cold_composite(rank, fluid)
    Q_total = Qc[-1]
    Tw = np.linspace(T_w_cold, T_w_hot, 400)
    hw = np.array([CP.PropsSI("H", "T", t, "P", P_water, "Water") / 1000.0
                   for t in Tw])
    Qh = (hw - hw[0]) * Q_total / (hw[-1] - hw[0])    # water flow scales out
    Qq = np.unique(np.concatenate([np.linspace(0.0, Q_total, n), Qc, Qh]))
    Qq = Qq[(Qq >= 0.0) & (Qq <= Q_total)]
    return float(np.min(np.interp(Qq, Qh, Tw) - np.interp(Qq, Qc, Tc)))


def feasible_rankine(case, T_1e, T_w_hot, T_w_cold):
    """The ORC evaporating temperature, as the MINIMUM of two constraints.

        T_3e = min( T_2d - DT_2D_3E ,  T_3e^pinch )

    The first is a minimum approach at the HOT END, which is what anybody
    writes first and what this model used alone until v0.9. The second is the
    largest boiling temperature for which the composite curves keep
    `case.DT_pinch_ORC` everywhere, found by bisection. Neither subsumes the
    other, so the binding one is whichever is lower, and the returned dict
    says which it was in `T_3e_binding`.

    At the default DT_2D_3E = 12 K the PINCH binds across the whole useful
    range of glides -- the hot-end rule never becomes active -- so that field
    is inert in practice while remaining a real constraint if raised. Writing
    it as a minimum rather than as "start from the old rule and correct it"
    is the same arithmetic and a more honest statement of intent.
    """
    T_3e_hot_end = T_w_hot - case.DT_2D_3E

    def pinch_at(t):
        return orc_pinch(double_stage_rankine(case.fluid, T_1e, t,
                                      case.eta_turb_s, case.eta_pump_s),
                         T_w_hot, T_w_cold, case.fluid, case.P)

    # is the hot-end value already pinch-feasible?
    if pinch_at(T_3e_hot_end) >= case.DT_pinch_ORC:
        rank = double_stage_rankine(case.fluid, T_1e, T_3e_hot_end,
                                    case.eta_turb_s, case.eta_pump_s)
        rank["T_3e_uncorrected"] = T_3e_hot_end
        rank["T_3e_hot_end"] = T_3e_hot_end
        rank["T_3e_binding"] = "hot-end approach DT_2D_3E"
        rank["pinch"] = orc_pinch(rank, T_w_hot, T_w_cold, case.fluid, case.P)
        return rank

    lo = T_1e + 10.0                     # floor: no cycle worth the name below
    hi = T_3e_hot_end
    if pinch_at(lo) < case.DT_pinch_ORC:
        raise RuntimeError(
            f"no ORC boiling temperature above {lo - 273.15:.1f} C clears the "
            f"required pinch of {case.DT_pinch_ORC:.1f} K against water at "
            f"{T_w_hot - 273.15:.1f} -> {T_w_cold - 273.15:.1f} C "
            f"(best is {pinch_at(lo):+.2f} K). The water glide is too large "
            f"for a cycle that boils at one temperature: reduce the "
            f"discharging glide, or raise the water temperatures, or relax "
            f"DT_pinch_ORC.")
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if pinch_at(mid) >= case.DT_pinch_ORC:
            lo = mid
        else:
            hi = mid
    rank = double_stage_rankine(case.fluid, T_1e, lo,
                                case.eta_turb_s, case.eta_pump_s)
    rank["T_3e_uncorrected"] = T_3e_hot_end      # kept: reported in css_report
    rank["T_3e_hot_end"] = T_3e_hot_end
    rank["T_3e_binding"] = "evaporator pinch DT_pinch_ORC"
    rank["pinch"] = orc_pinch(rank, T_w_hot, T_w_cold, case.fluid, case.P)
    return rank


def _stream_curve(h_cold_end, h_hot_end, p, fluid, n=400):
    """(Q, T) for a single-pressure stream, from its COLD end.

    Q is duty per unit mass of that stream. Sampling is in enthalpy, so a
    latent plateau comes out flat and a (T, p) call never lands exactly on
    the saturation line.
    """
    hs = np.linspace(h_cold_end, h_hot_end, n)
    Ts = np.array([CP.PropsSI("T", "H", x * 1000.0, "P", p, fluid)
                   for x in hs])
    return hs - h_cold_end, Ts


def _water_curve(T_cold, T_hot, P, Q_total, n=400):
    """(Q, T) for the water, scaled so its total duty matches Q_total."""
    Ts = np.linspace(T_cold, T_hot, n)
    hs = np.array([CP.PropsSI("H", "T", t, "P", P, "Water") / 1000.0
                   for t in Ts])
    if abs(hs[-1] - hs[0]) < 1e-12:                 # isothermal reservoir
        return np.array([0.0, Q_total]), np.array([T_cold, T_hot])
    return (hs - hs[0]) * Q_total / (hs[-1] - hs[0]), Ts


def _UA_from_curves(Qh, Th, Qc, Tc, Q_duty_kW, n=2000):
    """UA and the effective mean approach, from two composite curves.

    Both curves are (Q, T) measured from the COLD END of the exchanger, on
    whatever duty basis; only their SHAPE is used. The conductance follows
    from the local driving difference:

        UA = int dQ / dT      ->     UA = Q_duty * <1/dT>

    and the effective mean approach is the number that reproduces it,
    dT_eff = Q_duty / UA. For a counterflow exchanger with both streams
    sensible and constant cp, dT_eff is the log-mean difference exactly;
    with a phase change on one side it is the correct generalisation, which
    the LMTD is not.

    Returns (UA [kW/K], dT_eff [K], pinch [K]). UA is infinite and dT_eff
    zero if the curves touch -- which is the honest answer, and is what the
    heat-pump evaporator returned before v0.9a gave it an approach.
    """
    Q_top = min(Qh[-1], Qc[-1])
    q = np.linspace(0.0, Q_top, n)
    dT = np.interp(q, Qh, Th) - np.interp(q, Qc, Tc)
    pinch = float(np.min(dT))
    if pinch <= 0.0:
        return float("inf"), 0.0, pinch
    # trapezoid on 1/dT, in units of the curves' own duty, then rescaled
    inv = np.trapezoid(1.0 / dT, q) if hasattr(np, "trapezoid") \
        else np.trapz(1.0 / dT, q)
    UA = Q_duty_kW * inv / Q_top
    return float(UA), float(Q_duty_kW / UA), pinch


def exchanger_UA(case, rank, hp, T, Eb):
    """Conductance UA [kW/K] required by each of the four exchangers.

    The model reports efficiencies but has never reported what they cost in
    hardware. That gap makes every approach temperature a free lunch: tighten
    `DT_pinch_ORC` to 1 K and eta_ORC rises with nothing to pay for it. UA is
    the missing axis, and it is the one that makes the approach temperatures
    genuine design variables rather than free parameters.

    Each exchanger is integrated over its real composite curves -- the same
    construction as `orc_pinch`, so a latent plateau is a flat segment and
    the water follows real enthalpy. Duties are plant-level, from the energy
    budget, so the UA values are for the whole field and add up.

    Returns a dict of dicts, one per exchanger, each with:
        duty_kW, UA_kW_per_K, dT_eff_K, pinch_K
    """
    F, R = case.fluid, case.refrig
    out = {}

    # ---- ORC evaporator: water (hot) against the two ORC streams ----------
    Q_orc = Eb["Q_dot_in_ORC"]
    Qc, Tc = _orc_cold_composite(rank, F)
    Qh, Th = _water_curve(T["T_3d"], T["T_2d"], case.P, Qc[-1])
    out["ORC evaporator"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_orc))) | {"duty_kW": Q_orc}

    # ---- ORC condenser: ORC fluid (hot) against the sink ------------------
    Q_cond_orc = Q_orc * (1.0 - rank["rank_eff"])
    Qh, Th = _stream_curve(rank["h_1e"], rank["h_4e"], rank["p_conde"], F)
    Qc, Tc = _water_curve(case.T_sink_C + 273.15,
                          case.T_sink_C + case.DT_sink_glide + 273.15,
                          case.P, Qh[-1])
    out["ORC condenser"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_cond_orc))) | {"duty_kW": Q_cond_orc}

    # ---- HTHP condenser: refrigerant (hot) against the charging water -----
    Q_cond_hp = Eb["Q_dot_out_HP"]
    Qh, Th = _stream_curve(hp["h_3h"], hp["h_2h"], hp["p_condh"], R)
    Qc, Tc = _water_curve(T["T_2c"], T["T_3c"], case.P, Qh[-1])
    out["HTHP condenser"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_cond_hp))) | {"duty_kW": Q_cond_hp}

    # ---- HTHP evaporator: source water (hot) against the refrigerant ------
    # Duty is the condenser heat less the compressor work: what the source
    # must actually give up.
    Q_evap_hp = Q_cond_hp * (1.0 - 1.0 / hp["hp_cop"])
    Qc, Tc = _stream_curve(hp["h_4h"], hp["h_13h"], hp["p_evaph"], R)
    Qh, Th = _water_curve(T["T_4a"], case.T_source_C + 273.15,
                          case.P, Qc[-1])
    out["HTHP evaporator"] = dict(zip(
        ("UA_kW_per_K", "dT_eff_K", "pinch_K"),
        _UA_from_curves(Qh, Th, Qc, Tc, Q_evap_hp))) | {"duty_kW": Q_evap_hp}

    finite = [v["UA_kW_per_K"] for v in out.values()
              if np.isfinite(v["UA_kW_per_K"])]
    out["TOTAL"] = dict(duty_kW=sum(v["duty_kW"] for v in out.values()),
                        UA_kW_per_K=sum(finite) if len(finite) == 4
                        else float("inf"),
                        dT_eff_K=float("nan"), pinch_K=min(
                            v["pinch_K"] for v in out.values()))
    return out


def ua_report(case, r):
    """Print the conductance each exchanger needs, and what it buys."""
    ua = exchanger_UA(case, r["rank"], r["hp"], r["T"], r["budget"])
    print("=" * 74)
    print("EXCHANGER CONDUCTANCE  (plant totals, all wells)")
    print("=" * 74)
    print(f"  {'exchanger':<18}{'duty':>10}{'UA':>12}"
          f"{'dT_eff':>10}{'pinch':>9}")
    print(f"  {'':<18}{'kW':>10}{'kW/K':>12}{'K':>10}{'K':>9}")
    print("  " + "-" * 60)
    for name in ("HTHP evaporator", "HTHP condenser",
                 "ORC evaporator", "ORC condenser"):
        v = ua[name]
        print(f"  {name:<18}{v['duty_kW']:>10.1f}{v['UA_kW_per_K']:>12.2f}"
              f"{v['dT_eff_K']:>10.2f}{v['pinch_K']:>9.2f}")
    print("  " + "-" * 60)
    t = ua["TOTAL"]
    print(f"  {'TOTAL':<18}{t['duty_kW']:>10.1f}{t['UA_kW_per_K']:>12.2f}"
          f"{'':>10}{t['pinch_K']:>9.2f}")
    print()
    print("  dT_eff is the single approach that would need the same UA. For a")
    print("  counterflow exchanger with both streams sensible it IS the")
    print("  log-mean difference; with a phase change on one side it is the")
    print("  correct generalisation and the LMTD is not.")
    print()
    print("  Read this beside eta_RTE. An approach temperature bought cheaply")
    print("  in efficiency is paid for here, and until v0.10 nothing said so.")
    return ua


def cycle_state_points(case, T_2d=None):
    """ORC and heat-pump state points, and the two cycle efficiencies.

    Note the expansions and compressions carry no isentropic efficiency, so
    eta_ORC and COP are idealised -- only the electrical and mechanical
    efficiencies appear.

    THE DISCHARGE CLOSURE (changed in v0.6). Only the two exchanger INLETS are
    boundary conditions on the march; both outlets are results of the heat
    transfer and cannot be prescribed. Until v0.5a the discharge was closed the
    other way round -- the outlet was pinned at T_2d = T_m,top - DT_m_2D with
    DT_m_2D = 0, and the inlet derived by subtracting the glide. That is not
    merely arbitrary, it is contradicted by the model's own solution: at the
    start of a discharge at CSS the water leaves at 157.93 C, nearly 8 K above
    the top layer's melting point, because the PCM is
    superheated. It is also silently tied to N_lay -- the implied inlet
    approach was DT_3C_2C / N_lay, one layer width, chosen by nobody.

    The inlets are now prescribed symmetrically:

        charge:    T_4c = T_m,top    + DT_4C_M     (superheat, drives melting)
        discharge: T_3d = T_m,bottom - DT_M_1D     (subcooling, drives freezing)

    and T_2d is DEMOTED to a provisional estimate, the value it would take if
    the water achieved the full glide. It is provisional because the ORC needs
    an evaporating temperature before the store has been marched: T_3e is tied
    to T_2d. Pass the realised outlet back in as `T_2d` for the correction pass
    (see `simulate_css`); the estimate is good enough that one pass suffices.

    Setting DT_M_1D = DT_3C_2C / N_lay recovers the retired closure exactly,
    which is how the change is verified to be a no-op.
    """
    T_3d_C = T_m_bottom(case) - 273.15 - case.DT_M_1D
    # provisional unless the caller supplies the realised value
    T_2d_C = (T_3d_C + case.glide_dc_spec) if T_2d is None else (T_2d - 273.15)
    T = dict(
        T_1e=case.T_sink_C + case.DT_sink_glide + case.DT_E_sink + 273.15,
        T_3e=T_2d_C - case.DT_2D_3E + 273.15,
        T_2d=T_2d_C + 273.15,
        T_3d=T_3d_C + 273.15,                     # the borehole inlet, state 1d
        T_4c=case.T_m_C + case.DT_4C_M + 273.15,
        # The source stream, and the evaporating temperature DERIVED from its
        # outlet. Until v0.9 these were two independent fields and the
        # approach between them was their difference, unchecked: the default
        # pair made it exactly zero, and reversing them made it negative with
        # no symptom anywhere, because T_4a was written here and never read
        # again. See DN-17.
        T_4a=case.T_source_C - case.DT_3A_4A + 273.15,
        T_13h=case.T_source_C - case.DT_3A_4A - case.DT_pinch_HPE + 273.15,
    )
    T["T_3c"] = T["T_4c"]
    T["T_2c"] = T["T_3c"] - case.DT_3C_2C
    T["T_2h"] = T["T_3c"] + case.DT_2H_3C

    # T["T_3e"] above is only the OLD hot-end rule, kept so the uncorrected
    # value stays visible. What the water can actually deliver is decided by
    # the whole evaporator, so the boiling temperature comes back from the
    # pinch check and is written into T -- otherwise the reported state points
    # and the efficiency would describe different cycles. This runs on the
    # correction pass too, with the realised T_2d.
    rank = feasible_rankine(case, T["T_1e"], T["T_2d"], T["T_3d"])
    T["T_3e_hot_end_rule"] = T["T_3e"]
    T["T_3e"] = rank["T_3e"]
    hp = two_stage_htheatpump_2regs(case.refrig, T["T_13h"], T["T_2h"],
                                    case.DT_sub, case.eta_comp_s)
    for name, v in (("rank_eff", rank.get("rank_eff")),
                    ("hp_cop", hp.get("hp_cop"))):
        if v is None or not np.isfinite(v):
            raise RuntimeError(f"{name} is not finite ({v})")
    return rank, hp, T


def water_h(case, T_K):
    """Specific enthalpy of the heat-transfer water, kJ/kg."""
    return CP.PropsSI("H", "T", T_K, "P", case.P, case.fluid2) / 1000.0


def water_s(case, T_K):
    """Specific entropy of the heat-transfer water, kJ/kg-K."""
    return CP.PropsSI("S", "T", T_K, "P", case.P, case.fluid2) / 1000.0


def stream_exergy_rate(case, m_dot, T_hot, T_cold):
    """Exergy given up by a water stream cooling from T_hot to T_cold, kW.

        m_dot [ (h_hot - h_cold) - T_0 (s_hot - s_cold) ],   T_0 = T_sink

    Positive when T_hot > T_cold >= T_0. Used for the process water and for
    the geothermal stream, which are the same fluid at the same pressure.
    Passing T_cold = T_0 gives the stream's exergy relative to the dead
    state, which is what the 'resource' convention charges for.
    """
    T_0 = case.T_sink_C + 273.15
    return m_dot * ((water_h(case, T_hot) - water_h(case, T_cold))
                    - T_0 * (water_s(case, T_hot) - water_s(case, T_cold)))


def energy_budget(case, rank_eff, hp_cop, T):
    """Work backwards from the specified electrical output to the charging duty."""
    # rank_eff and hp_cop now ALREADY carry the machine losses, so only
    # the electrical efficiencies remain here. Dividing by Turb_eff as
    # well would double-count them.
    W_dot_T = case.W_dot_el_out / case.ElG_eff
    Q_dot_in_ORC = W_dot_T / rank_eff
    D_E_in_ORC = Q_dot_in_ORC * case.t_dc * 3600.0                 # kJ
    D_E_out_HP = D_E_in_ORC * (1.0 + case.loss_surplus)
    Q_dot_out_HP = D_E_out_HP / case.t_ch / 3600.0
    W_dot_C_HP = Q_dot_out_HP / hp_cop
    W_dot_el_in = W_dot_C_HP / case.ElH_eff

    # v0.13: the ENTHALPY difference, not c_p at the mean temperature times
    # the glide. Over the 105 -> 160 C charging rise c_p varies about 3 %, so
    # the linearised form gives a flow rate 0.096 % inconsistent with the
    # model's own CoolProp data. That is invisible in the energy balance --
    # both sides use the same wrong flow -- but it fabricates 7.17 kW of
    # exergy and stopped `exergy_audit` closing. The identical formula was
    # written a SECOND time in `run_cycle`, where it is live; see the note
    # there. DN-24.
    m_dot_w_ch = Q_dot_out_HP / (water_h(case, T["T_3c"])
                                 - water_h(case, T["T_2c"]))
    return dict(Q_dot_in_ORC=Q_dot_in_ORC, D_E_in_ORC=D_E_in_ORC,
                D_E_out_HP=D_E_out_HP, Q_dot_out_HP=Q_dot_out_HP,
                W_dot_el_in=W_dot_el_in, m_dot_w_ch=m_dot_w_ch)


def melting_temperatures(case):
    """Layer melting temperatures for charging and for discharging."""
    dTm = case.cascade_param / case.N_lay
    i = np.arange(case.N_lay, dtype=float)
    return ((case.T_m - i * dTm).tolist(),
            (case.T_m - case.cascade_param + (i + 1.0) * dTm).tolist())


# ==========================================================================
# sizing
# ==========================================================================


# ==========================================================================
# run   (charge, size, discharge, KPIs)
# ==========================================================================


# ==========================================================================
# cyclic steady state   (v0.5)
# ==========================================================================
# run_cycle() above answers "how many wells does one charge from a COLD START
# need?". That is a first-cycle question, and the cycle does not return to the
# state it started from, so the answer is not what a plant repeating a 24 h
# schedule would see.
#
# WHY SIZING CANNOT SIMPLY BE MOVED TO CSS. At cyclic steady state the state
# returns to itself, so the enthalpy change over a cycle is zero. This model has
# no loss path -- H2 no axial conduction, H6 adiabatic wall, H9 no azimuthal
# exchange -- so charge energy EQUALS discharge energy identically:
#
#       eta_storage(CSS) == 1     exactly, for every N and every flow.
#
# Two consequences follow, and both are structural rather than numerical.
#
#   1. The assumed loss surplus lambda is UNATTAINABLE at CSS. The budget
#      requires D_E_out_HP = (1+lambda) D_E_in_ORC, so the ratio of CSS charge
#      to requirement pins at 1/(1+lambda) for EVERY N. There is no root.
#      On the first cycle the surplus is not lost but PARKED in the store as
#      residual melt; at CSS there is nowhere left to park it.
#
#   2. Even with lambda = 0 the rate criterion goes DEGENERATE. Charge and
#      discharge are then the same number, so one equation is left for two
#      unknowns: it fixes the FLOW RATIO and says nothing about N.
#
# THE REFORMULATION. Stop asking the energy balance to determine N. Specify the
# hardware, march to CSS, and report what comes out:
#
#       N        from latent heat alone, D_E_out_HP / (rho V_well h_m)
#       m_ch     pinned by the charging glide
#       m_dc     pinned by the discharging glide
#
# Nothing is solved -- there is no root-find anywhere in this routine -- so the
# question of convergence does not arise. What was an unsatisfiable constraint
# becomes a reported output: the deviation of the delivered energy, and hence of
# the net electrical output, from its target.
#
# This also dissolves an older objection. size_well_field's docstring argues
# that the discharge flow cannot be pinned to its glide because the delivered
# energy then saturates below the requirement for any well count, leaving no
# root. Under this framing that saturation is not a failure; it is the answer.


def simulate_css(case, N=None, n_cycles=80, tol=1e-9, record=False, T_2d=None):
    """March a specified well field to cyclic steady state.

    One pass. `T_2d` overrides the provisional exchanger outlet used to set the
    ORC evaporating temperature; `simulate_css_corrected` is the wrapper that
    supplies the realised value. Called directly, this is the uncorrected
    result.

    No root-finding. `N` defaults to the latent-heat-only well count and both
    flows are pinned by their glides, so the routine is a pure simulation.

    `lambda` is forced to zero regardless of `case.loss_surplus`, because a
    non-zero surplus cannot be absorbed at CSS in a model with no loss path
    (see the note above). The override is reported in the returned dict as
    `lambda_overridden` so it can never happen silently.

    Returns the converged half-cycle states, the convergence history, and the
    deviation of the delivered energy from the target.
    """
    lambda_overridden = case.loss_surplus != 0.0
    case = case.with_(loss_surplus=0.0) if lambda_overridden else case

    rank, hp, T = cycle_state_points(case, T_2d=T_2d)
    Eb = energy_budget(case, rank["rank_eff"], hp["hp_cop"], T)
    n = case.n_segments
    T_m_lay, T_m_lay_dc = melting_temperatures(case)
    T_m_seg = layer_map(T_m_lay, n, case.N_lay)
    T_m_seg_dc = T_m_seg[::-1].copy()     # flow reverses; see run_cycle
    # The rock profile is mirrored with exactly the same rule as T_m_seg, for
    # exactly the same reason: on discharge the flow enters at the top.
    T_rock_ch, depth_ch = formation_profile(case, n, discharge=False)
    T_rock_dc, depth_dc = formation_profile(case, n, discharge=True)
    # The resistance is now a PROFILE, because the spacing is. Built on the
    # same two frames as T_rock so a mirroring slip cannot put the shielded
    # shallow resistance against the deep rock.
    R_form_ch = formation_resistance(case, depth_ch)
    R_form_dc = formation_resistance(case, depth_dc)
    times_ch = np.logspace(0.0, np.log10(case.t_ch * 3600.0), case.n_times)
    times_dc = np.logspace(0.0, np.log10(case.t_dc * 3600.0), case.n_times)

    # ---- hardware and flows: all SPECIFIED, none solved -------------------
    if N is None:
        N = Eb["D_E_out_HP"] * 1000.0 / (case.rho_latent * case.V_well * case.h_m)
    # v0.13: enthalpy differences, not c_p at the mean temperature times the
    # glide. This is the LIVE copy of the formula fixed in `energy_budget`;
    # the two were written independently and both were wrong the same way,
    # which is why patching only one of them produced an encouraging no-op
    # and left the defect in place. See DN-24.
    m1_ch = (Eb["Q_dot_out_HP"] / (water_h(case, T["T_3c"])
                                   - water_h(case, T["T_2c"]))
             / (N * case.n_ft))
    # NOTE the discharge flow stays pinned by the SPECIFIED glide, as it has
    # been since v0.6, and not by the realised T_2d that the correction pass
    # knows about. Using the realised outlet here is tempting -- it cuts the
    # reported deviation from +4.46 % to +1.45 % and lifts eta_RTE by 0.28 %
    # -- but that is partly circular: feeding last pass's outlet into this
    # pass's flow definition drives the delivered energy toward the target by
    # construction, without iterating to a fixed point. It would also change
    # the model's SPECIFICATION ("the flow is pinned by the glide"), which is
    # a different decision from fixing a linearisation. Measured and left
    # alone. See DN-25.
    m1_dc = (Eb["Q_dot_in_ORC"]
             / (water_h(case, T["T_3d"] + case.glide_dc_spec)
                - water_h(case, T["T_3d"]))
             / (N * case.n_ft))

    # ---- march until the state repeats -----------------------------------
    def cycle(E0, rec=False):
        ch = march_h(case, T["T_4c"], T_m_seg, m1_ch, case.k_wall, times_ch,
                     n_segments=n, E0=E0, record=rec, T_rock=T_rock_ch,
                     R_form=R_form_ch)
        dc = march_h(case, T["T_3d"], T_m_seg_dc, m1_dc, case.k_wall, times_dc,
                     n_segments=n, E0=ch["E"][::-1], record=rec,
                     T_rock=T_rock_dc, R_form=R_form_dc)
        return ch, dc

    E = np.zeros(n)
    history = []
    for k in range(n_cycles):
        ch, dc = cycle(E)
        Q_ch = ch["Q_cum_J"] * case.n_ft * N / 1000.0          # kJ, field
        Q_dc = abs(dc["Q_cum_J"]) * case.n_ft * N / 1000.0
        drift = float(np.max(np.abs(dc["E"][::-1] - E)))
        history.append(dict(cycle=k + 1, Q_charge_kJ=Q_ch, Q_discharge_kJ=Q_dc,
                            drift=drift,
                            eps_end_charge=float(ch["eps_local"].mean()),
                            eps_end_discharge=float(dc["eps_local"].mean())))
        E = dc["E"][::-1]                 # back to charge-march indexing
        if drift < tol:
            break

    if record:
        # One extra cycle from the converged state, this time recording. The
        # state repeats to `tol`, so this reproduces the last cycle; doing it
        # here rather than inside the loop means the profile sweep that closes
        # each recorded march is paid once instead of `cycles` times.
        ch, dc = cycle(E, rec=True)

    # The discharge marches from the far end, so its segment index runs
    # backwards. Everything returned to the caller is in CHARGE (depth)
    # indexing, which is why there is one cascade, `T_m_seg`, and not two.
    dc = unmirror_march(dc)

    req = Eb["D_E_in_ORC"]
    deviation = Q_dc / req - 1.0

    # ---- the outlet temperatures, which are results and not inputs ---------
    T_2d_realised = mixed_mean_outlet(dc)      # what the ORC evaporator sees
    T_2c_realised = mixed_mean_outlet(ch)      # what returns to the HP condenser
    glide_dc = T_2d_realised - T["T_3d"]
    glide_ch = T["T_4c"] - T_2c_realised
    return dict(
        N_wells=N, m1_ch=m1_ch, m1_dc=m1_dc, flow_ratio=m1_dc / m1_ch,
        cycles=len(history), converged=history[-1]["drift"] < tol, tol=tol,
        history=history, charge=ch, discharge=dc, E_css=E,
        Q_charge_kJ=Q_ch, Q_discharge_kJ=Q_dc, required_kJ=req,
        deviation=deviation,
        W_el_out_implied=case.W_dot_el_out * (1.0 + deviation),
        # ---- THE NUMBER THAT WAS 1.0000 BY CONSTRUCTION ------------------
        # In THUMS eta_storage was Q_dc/Q_ch with an adiabatic wall, so it
        # was identically 1 and carried no information whatever: it could not
        # move, it could not fail, and reporting it was theatre. Here it is a
        # measurement. Q_loss is signed and SUMMED, not accumulated in
        # magnitude -- the deep segments can sit in rock hotter than they are
        # and run the other way, and that is a result, not an error to be
        # absorbed by an abs().
        Q_loss_ch_kJ=ch["Q_loss_J"] * case.n_ft * N / 1000.0,
        Q_loss_dc_kJ=dc["Q_loss_J"] * case.n_ft * N / 1000.0,
        Q_loss_kJ=(ch["Q_loss_J"] + dc["Q_loss_J"]) * case.n_ft * N / 1000.0,
        R_formation=float(np.mean(np.atleast_1d(
            formation_resistance(case, depth_ch)))),
        B_top=(None if case.n_wells_field is None
               else float(np.atleast_1d(well_spacing(case, 0.0))[0])),
        B_bottom=(None if case.n_wells_field is None
                  else float(np.atleast_1d(
                      well_spacing(case, case.L_well))[0])),
        R_ground=ground_resistance(case),
        eta_storage=Q_dc / Q_ch, lambda_overridden=lambda_overridden,
        T_2d_assumed=T["T_2d"], T_2d_realised=T_2d_realised,
        T_2c_assumed=T["T_2c"], T_2c_realised=T_2c_realised,
        glide_dc=glide_dc, glide_ch=glide_ch,
        glide_ratio_dc=glide_dc / case.glide_dc_spec,
        T_2d_was_provisional=T_2d is None,
        T=T, budget=Eb, T_m_seg=T_m_seg, rank=rank, hp=hp)
        # no `T_m_seg_dc`: the discharge is returned in depth indexing, where
        # the cascade is the same physical object as on charge. The mirrored
        # copy is an internal detail of the march and does not leave here.


def geothermal_resource(case, r):
    """The 60 C source as a FINITE, priced resource.

    Until v0.13 the source was effectively unlimited: `DT_3A_4A` said how far
    the stream was cooled, the flow followed from the duty, and no output ever
    recorded how much flow that was. Nothing in the model objected to a design
    that quietly demanded five times the geothermal field.

    The flow is DERIVED, not prescribed -- the same choice already made for
    `N_wells`. Prescribing a plant-level flow would put the uncertainty in a
    quantity nobody has measured; deriving it puts the uncertainty in the
    per-well yield, which is field data. See DN-22.

    Note that the exergy of the source is homogeneous of degree one in the
    flow, and m_geo = Q_src / (c_p dT), so BOTH conventions below reduce to
    the duty, the glide and the two temperatures. Deriving the flow rather
    than prescribing it costs the second-law analysis nothing.
    """
    hp, T = r["hp"], r["T"]
    T_0 = case.T_sink_C + 273.15
    T_src = case.T_source_C + 273.15
    T_rei = T["T_4a"]                          # T_source - DT_3A_4A

    x = hp["x_8h"]
    m_ref = r["budget"]["Q_dot_out_HP"] / (hp["h_2h"] - hp["h_3h"])
    Q_src = m_ref * (1.0 - x) * (hp["h_13h"] - hp["h_4h"])          # kW
    m_geo = Q_src / (water_h(case, T_src) - water_h(case, T_rei))   # kg/s

    ex_full = stream_exergy_rate(case, m_geo, T_src, T_0)
    ex_rei = stream_exergy_rate(case, m_geo, T_rei, T_0)
    ex_stripped = stream_exergy_rate(case, m_geo, T_src, T_rei)

    N_geo = m_geo / case.m_dot_geo_well
    return dict(
        Q_src_kW=Q_src, m_geo=m_geo, m_ref=m_ref,
        T_reinject_C=T_rei - 273.15,
        Ex_full_kW=ex_full, Ex_reinject_kW=ex_rei, Ex_stripped_kW=ex_stripped,
        utilisation=ex_stripped / ex_full if ex_full else float("nan"),
        N_geo=N_geo, N_geo_per_MWe=N_geo / (case.W_dot_el_out / 1000.0),
        # which of the two the audit will charge for
        Ex_charged_kW=(ex_full if case.exergy_convention == "resource"
                       else ex_stripped),
        Ex_loss_kW=(ex_rei if case.exergy_convention == "resource" else 0.0),
        convention=case.exergy_convention)


def borehole_pumping(case, r):
    """Field pumping power for both half-cycles, kW, at the realised flows.

    ONE copy, called by `performance_indices` and by `exergy_audit`. It was
    briefly written twice -- which is the DN-24 trap exactly, a formula
    duplicated so that a later fix can land in one copy and miss the other.
    """
    gv = case.geom_vector()
    N, T = r["N_wells"], r["T"]
    _, pp_dc = calculate_pressure_drop(gv, r["m1_dc"], case.fluid2,
                                       r["T_2d_realised"], T["T_3d"], case.P)
    _, pp_ch = calculate_pressure_drop(gv, r["m1_ch"], case.fluid2,
                                       T["T_4c"], r["T_2c_realised"], case.P)
    f = case.n_ft * N / 1000.0

    # ---- THE DOWNCOMER IS ALSO A PIPE -----------------------------------
    # Easy to forget, because it is the passage that does no heat transfer --
    # and forgetting it would flatter BOREAS against THUMS on precisely the
    # quantity Study 2 showed decides efficiency below ground. The fin tubes
    # now run L_well where the hairpin ran 2*L_well, so leaving the downcomer
    # out would halve the pumping rather than keep it equal.
    #
    # It carries the WHOLE well's flow (n_ft * m1) through a bore sized by
    # r_id_bore = r_i * sqrt(n_ft), i.e. n_ft times the area. Same velocity,
    # same length, so to the extent friction factors match it contributes the
    # same drop as one fin tube's worth -- which is the sense in which this
    # geometry has the hairpin's pressure drop, not half of it.
    gv_id = list(gv)
    gv_id[3] = case.r_id_bore                    # r_i
    gv_id[5] = 2.0 * case.r_id_bore              # D_i_tube, what the drop uses
    gv_id[8], gv_id[9] = 0.0, 0                  # no fins inside the downcomer
    _, pid_ch = calculate_pressure_drop(gv_id, r["m1_ch"] * case.n_ft,
                                        case.fluid2, T["T_4c"],
                                        T["T_4c"], case.P)
    _, pid_dc = calculate_pressure_drop(gv_id, r["m1_dc"] * case.n_ft,
                                        case.fluid2, T["T_3d"],
                                        T["T_3d"], case.P)
    f_id = N / 1000.0                            # ONE downcomer per well
    return pp_ch * f + pid_ch * f_id, pp_dc * f + pid_dc * f_id


def downcomer_leak(case, r):
    """Heat crossing the downcomer insulation, per half-cycle [kJ, field].

    THE SIGN REVERSES BETWEEN HALF-CYCLES, and the two are not equal.

    On CHARGE the downcomer carries the 160 C supply down past PCM grading
    101 C at the top to 150 C at the bottom, so heat leaks OUT of the pipe and
    INTO the store. The energy is not lost -- it is deposited in the wrong
    layer, at a large temperature mismatch. An exergy penalty, not an energy
    one. This is the advantage the geometry has over a nested coaxial, where
    the insulated passage carries the COLD return and its leak un-stores heat
    and hands it back to the heat pump.

    On DISCHARGE the downcomer carries the hot outlet back up past the same
    PCM, and the leak re-deposits heat that was just extracted. That one is a
    true capacity loss, and it is the larger of the two because the driving
    difference is larger.

    Quasi-steady and evaluated against the CASCADE temperature rather than the
    realised PCM temperature: the downcomer is a boundary condition on the
    store, not a participant in its march, and resolving it properly would
    mean coupling it segment by segment into `march_h`. Reported separately so
    it is never silently folded into the formation loss, which is a different
    mechanism with a different sign convention.
    """
    T, n = r["T"], case.n_segments
    T_m_seg = r["T_m_seg"]
    R_ins = (np.log(case.r_id_outer / case.r_id_bore)
             / (2.0 * np.pi * case.k_ins))
    dz = case.L_well / n

    # charge: supply descends at T_4c; discharge: return ascends at T_2d
    q_ch = float(np.sum((T["T_4c"] - T_m_seg) / R_ins) * dz)
    q_dc = float(np.sum((r["T_2d_realised"] - T_m_seg) / R_ins) * dz)
    N = r["N_wells"]
    return (q_ch * N * case.t_ch * 3600.0 / 1e3,      # kJ, into the store
            q_dc * N * case.t_dc * 3600.0 / 1e3)


def exergy_audit(case, r):
    """Exergy destruction by component over one cycle at CSS, in kWh.

    THE DEAD STATE is T_0 = T_sink. It is the lowest temperature in the
    system, so every exergy is positive, and the heat dumped at the ORC
    condenser is genuinely unrecoverable. It is NOT the heat-pump evaporator
    source: that is the 60 C geothermal stream, a second and quite separate
    reservoir, which is why `geothermal_resource` has to price it.

    THE CLOSURE, and why this function is a gate rather than a report:

        W_el_in + Ex_geo  =  W_el_out + Ex_reinject + sum(I_j)

    The left and right sides are computed from DIFFERENT things. The boundary
    terms come from four stream states. Every I_j comes from the entropy
    generated inside one component, from its own state points and flow
    fractions. If the cycle state points, the flow splits and the duties are
    mutually consistent, the two agree to machine precision; if any one of
    them is wrong, they do not. That is what caught the c_p linearisation in
    the water flow rates (DN-24), which was invisible to every energy balance
    in the model because both sides of those balances used the same wrong
    flow.

    WHAT IS A DESTRUCTION AND WHAT IS A LOSS. The reinjected geothermal
    stream leaves the boundary intact -- its exergy is not destroyed, it is
    discarded. It is reported on its own line and attributed to NO component,
    because no component is responsible for it; the decision to size the
    field this way is. Under the 'stripped' convention it is not charged at
    all and the line is zero.

    THE BOREHOLE is one bucket here, obtained from the difference of the two
    water-stream exergies over a full cycle. That is exact at CSS -- the PCM
    returns to its own initial state, so its net exergy change is zero, and no
    PCM entropy function is needed. It does not say WHERE in the well or WHEN
    the destruction happens; that needs s(E') and is the next step.
    """
    T_0 = case.T_sink_C + 273.15
    hp, rank, T, Eb = r["hp"], r["rank"], r["T"], r["budget"]
    t_ch, t_dc = case.t_ch, case.t_dc                       # hours
    geo = geothermal_resource(case, r)
    m_ref = geo["m_ref"]
    x = hp["x_8h"]

    # ---- heat pump, per kg of condenser flow ------------------------------
    s = lambda k: hp["s_" + k] / 1000.0                     # kJ/kg-K
    h = lambda k: hp["h_" + k]
    gen = {}
    gen["HTHP compressor HP"] = 1.0 * (s("2h") - s("6h"))
    gen["HTHP compressor LP"] = (1.0 - x) * (s("5h") - s("1h"))
    gen["HTHP throttle 7h-8h"] = 1.0 * (s("8h") - s("7h"))
    gen["HTHP throttle 9h-4h"] = (1.0 - x) * (s("4h") - s("9h"))
    gen["HTHP flash separator"] = (x * s("10h") + (1.0 - x) * s("9h")
                                   - s("8h"))
    gen["HTHP IHX-1"] = 1.0 * (s("12h") - s("3h")) + x * (s("11h") - s("10h"))
    gen["HTHP IHX-2"] = (1.0 * (s("7h") - s("12h"))
                         + (1.0 - x) * (s("1h") - s("13h")))
    m_w_ch = Eb["m_dot_w_ch"]
    gen["HTHP condenser"] = (1.0 * (s("3h") - s("2h"))
                             + m_w_ch * (water_s(case, T["T_3c"])
                                         - water_s(case, T["T_2c"])) / m_ref)
    gen["HTHP evaporator"] = ((1.0 - x) * (s("13h") - s("4h"))
                              + geo["m_geo"]
                              * (water_s(case, T["T_4a"])
                                 - water_s(case, case.T_source_C + 273.15))
                              / m_ref)
    I = {k: T_0 * m_ref * v * t_ch for k, v in gen.items()}     # kWh/cycle

    W_comp = m_ref * ((h("2h") - h("6h")) + (1.0 - x) * (h("5h") - h("1h")))
    W_el_in = Eb["W_dot_el_in"]
    I["HTHP motor"] = (W_el_in - W_comp) * t_ch

    # ---- ORC, per kg through the first turbine stage ----------------------
    y = rank["y_frac"]
    se = lambda k: rank["s_" + k] / 1000.0
    he = lambda k: rank["h_" + k]
    m_orc = Eb["Q_dot_in_ORC"] / ((he("3e") - he("10e"))
                                  + y * (he("6e") - he("5e")))
    ge = {}
    ge["ORC turbine 1"] = 1.0 * (se("5e") - se("3e"))
    ge["ORC turbine 2"] = y * (se("4e") - se("6e"))
    ge["ORC pump 1"] = y * (se("2e") - se("1e"))
    ge["ORC pump 2"] = (1.0 - y) * (se("8e") - se("7e"))
    ge["ORC regenerator"] = ((1.0 - y) * (se("7e") - se("5e"))
                             + y * (se("9e") - se("2e")))
    ge["ORC mixer"] = se("10e") - (y * se("9e") + (1.0 - y) * se("8e"))
    m_w_dc = Eb["Q_dot_in_ORC"] / (water_h(case, T["T_2d"])
                                   - water_h(case, T["T_3d"]))
    ge["ORC evaporator"] = ((1.0 * (se("3e") - se("10e"))
                             + y * (se("6e") - se("5e")))
                            + m_w_dc * (water_s(case, T["T_3d"])
                                        - water_s(case, T["T_2d"])) / m_orc)
    # the sink is AT the dead state, so the rejected heat carries no exergy
    # and the whole of it is destroyed in the condenser
    Q_rej = m_orc * y * (he("4e") - he("1e"))
    ge["ORC condenser"] = y * (se("1e") - se("4e")) + Q_rej / T_0 / m_orc
    for k, v in ge.items():
        I[k] = T_0 * m_orc * v * t_dc

    W_turb = m_orc * ((he("3e") - he("5e")) + y * (he("6e") - he("4e"))
                      - y * (he("2e") - he("1e"))
                      - (1.0 - y) * (he("8e") - he("7e")))
    I["ORC generator"] = (W_turb - case.W_dot_el_out) * t_dc

    # ---- THE TWO GATES THAT CAN ACTUALLY FAIL -----------------------------
    # Each machine bucket is summed from entropy generated INSIDE its
    # components and checked against a boundary balance built from stream
    # states only. These are independent computations and they are what
    # caught DN-24.
    I_hp = sum(v for k, v in I.items() if k.startswith("HTHP"))
    I_orc = sum(v for k, v in I.items() if k.startswith("ORC"))
    dEx_ch = stream_exergy_rate(case, m_w_ch, T["T_3c"], T["T_2c"]) * t_ch
    dEx_dc = stream_exergy_rate(case, m_w_dc, T["T_2d"], T["T_3d"]) * t_dc
    bnd_hp = (W_comp * t_ch + geo["Ex_stripped_kW"] * t_ch - dEx_ch
              + (W_el_in - W_comp) * t_ch)
    bnd_orc = dEx_dc - W_turb * t_dc + (W_turb - case.W_dot_el_out) * t_dc
    gates = {"HTHP": (I_hp, bnd_hp), "ORC": (I_orc, bnd_orc)}

    # ---- borehole: BY DIFFERENCE, and therefore not a gate ----------------
    # T_3c = T_4c and T_2d/T_3d are shared with the surface exchangers, so
    # this term is algebraically the residual of the water loop: it cancels
    # the two exchanger terms exactly and the GLOBAL sum below is an
    # identity, not a test. An earlier draft printed that identity as a
    # closure check reading 0.00e+00 and called it a pass. It cannot fail,
    # so it cannot pass. The independent value comes from the resolved
    # integral over z and t once s(E') exists; until then this number is
    # correct but unverified.
    I["borehole"] = dEx_ch - dEx_dc

    # The water loop does NOT in fact close at CSS: the well returns water at
    # T_2c_realised, not at the T_2c the condenser assumes, and likewise on
    # discharge. Reported rather than absorbed silently.
    mism = float("nan")
    if r.get("T_2c_realised") is not None:
        mism = (stream_exergy_rate(case, m_w_ch, T["T_3c"],
                                   r["T_2c_realised"]) * t_ch - dEx_ch
                - (stream_exergy_rate(case, m_w_dc, r["T_2d_realised"],
                                      T["T_3d"]) * t_dc - dEx_dc))

    # ---- REALISED BASIS, and the same boundary as eta_RTE ----------------
    # Until v0.15 this function evaluated the whole audit on the TARGET
    # duties out of `energy_budget`, and charged no pumping. `eta_RTE` in
    # `performance_indices` does neither: it uses the energy the field
    # actually moved, and it charges the borehole parasitics. So psi and
    # eta_RTE were computed on different bases and different boundaries, and
    # quoting "0.317 becomes 0.246" compared two things that were not
    # like for like. The manuscript caught this before the code did.
    #
    # At CSS the storage efficiency is identically 1 and lambda is forced to
    # zero, so Q_charge = Q_discharge and D_E_out_HP = D_E_in_ORC: both half
    # cycles scale by the SAME factor, and the rebase is one multiplication.
    scale = r["Q_discharge_kJ"] / Eb["D_E_in_ORC"]
    for key in I:
        I[key] *= scale
    W_in = W_el_in * t_ch * scale
    W_out = case.W_dot_el_out * t_dc * scale
    Ex_geo = geo["Ex_charged_kW"] * t_ch * scale
    Ex_loss = geo["Ex_loss_kW"] * t_ch * scale

    # The borehole parasitics are work in, dissipated as friction in the
    # well, so they are an input AND a destruction. Charged with eta_RTE's
    # sign convention: the charge pump adds to the input, the discharge pump
    # subtracts from the output. They are computed at the realised flows
    # already, so they are not scaled.
    pump_ch, pump_dc = borehole_pumping(case, r)
    P_ch, P_dc = pump_ch * t_ch, pump_dc * t_dc
    I["borehole pumping"] = P_ch + P_dc
    W_in += P_ch
    W_out -= P_dc

    boundary = W_in + Ex_geo - W_out - Ex_loss
    total = sum(I.values())

    # A THIRD GATE, and the only one that spans two functions. eta_RTE is
    # computed here from the scaled budget plus the parasitics, and in
    # `performance_indices` from the energy the field actually moved. Two
    # routes, one answer. It is what would have caught the basis mismatch
    # that the manuscript found before the code did, and it can only hold
    # while psi and eta_RTE share a boundary -- which is the point.
    # Recomputed the way `performance_indices` does it -- from the energy the
    # field moved and the same parasitics -- rather than from the budget.
    Q_in_ORC = r["Q_discharge_kJ"] / (t_dc * 3600.0)
    Q_out_HP = r["Q_charge_kJ"] / (t_ch * 3600.0)
    rk_eff = case.W_dot_el_out / case.ElG_eff / Eb["Q_dot_in_ORC"]
    cop_r = Eb["Q_dot_out_HP"] / (Eb["W_dot_el_in"] * case.ElH_eff)
    W_out_pi = (Q_in_ORC * rk_eff * case.ElG_eff - pump_dc) * t_dc
    W_in_pi = (Q_out_HP / cop_r / case.ElH_eff + pump_ch) * t_ch
    gates["eta_RTE"] = (W_out / W_in * W_in_pi, W_out_pi)

    return dict(
        T_0=T_0, components_kWh=I, total_kWh=total, boundary_kWh=boundary,
        scale=scale, pumping_ch_kWh=P_ch, pumping_dc_kWh=P_dc,
        gates={k: dict(sum_kWh=a, boundary_kWh=b, residual_kWh=a - b,
                       residual_rel=abs(a - b) / abs(b) if b else float("nan"))
               for k, (a, b) in gates.items()},
        gates_pass=all(abs(a - b) <= 1e-9 * abs(b) for a, b in gates.values()),
        global_is_identity=True,
        loop_mismatch_kWh=mism,
        W_el_in_kWh=W_in, W_el_out_kWh=W_out,
        Ex_geo_kWh=Ex_geo, Ex_reinject_kWh=Ex_loss,
        eta_RTE=W_out / W_in,
        psi=W_out / (W_in + Ex_geo),
        psi_stripped=W_out / (W_in + geo["Ex_stripped_kW"] * t_ch * scale),
        shares={k: v / total for k, v in I.items()},
        geo=geo)


def exergy_report(case, r, a=None):
    """Print the destruction table, largest first, and the closure check."""
    a = exergy_audit(case, r) if a is None else a
    g = a["geo"]
    print(f"EXERGY AUDIT   dead state T_0 = {a['T_0'] - 273.15:.1f} C "
          f"(the sink), convention '{g['convention']}'")
    print(f"  IN   electrical           {a['W_el_in_kWh']:10.1f} kWh")
    print(f"  IN   geothermal           {a['Ex_geo_kWh']:10.1f} kWh   "
          f"({g['m_geo']:.1f} kg/s, {g['N_geo']:.2f} producers)")
    print(f"  OUT  electrical           {a['W_el_out_kWh']:10.1f} kWh")
    print(f"  OUT  reinjected (LOSS)    {a['Ex_reinject_kWh']:10.1f} kWh   "
          f"at {g['T_reinject_C']:.1f} C, "
          f"{100 * (1 - g['utilisation']):.1f} % of the resource discarded")
    print(f"  {'-' * 56}")
    for k, v in sorted(a["components_kWh"].items(), key=lambda kv: -kv[1]):
        print(f"    {k:<26}{v:10.1f} kWh{100 * a['shares'][k]:7.1f} %")
    print(f"  {'-' * 56}")
    print(f"    {'SUM of components':<26}{a['total_kWh']:10.1f} kWh")
    print(f"    {'boundary balance':<26}{a['boundary_kWh']:10.1f} kWh"
          "   (identity -- see below)")
    print("\n  GATES. Only these can fail. Each sums the entropy generated "
          "inside\n  one machine's components and checks it against a "
          "balance built from\n  stream states alone.")
    for k, g in a["gates"].items():
        ok = g["residual_rel"] <= 1e-9
        print(f"    {k:<8}{g['sum_kWh']:11.2f} vs {g['boundary_kWh']:11.2f} "
              f"kWh   rel {g['residual_rel']:.2e}   "
              f"{'OK' if ok else '*** FAIL ***'}")
    print("    borehole  by difference -- the water-loop residual. It cancels"
          "\n              the two exchanger terms exactly, so the global sum"
          "\n              above is an IDENTITY and cannot fail. Verified only"
          "\n              when the resolved s(E') integral lands.")
    if a["loop_mismatch_kWh"] == a["loop_mismatch_kWh"]:
        print(f"    loop      {a['loop_mismatch_kWh']:+.1f} kWh between the "
              "outlets the surface assumes\n              and the outlets the "
              "well actually delivers at CSS")
    print(f"\n  eta_RTE {a['eta_RTE']:.4f}    psi (resource) {a['psi']:.4f}"
          f"    psi (stripped) {a['psi_stripped']:.4f}")
    return a


def performance_indices(case, r):
    """The performance indicators of the factorial study, evaluated at CSS.

    These were defined for the first-cycle framing, where the field delivered
    the target energy by construction. At cyclic steady state it delivers
    whatever it delivers, so every one of them is computed from the ENERGY THE
    FIELD ACTUALLY MOVES rather than from the target. That matters: charging
    with the target input while crediting the realised output would flatter the
    round-trip efficiency by exactly the deviation.

        eta_T       thermal -> net electric conversion on the discharge side,
                    W_el_out / Q_in_ORC. Identically eta_ORC * eta_turb * eta_gen,
                    so it is a property of the power block alone.
        eta_T_eff   the same with the discharge pumping parasitic charged
                    against it. This is the "effective discharge efficiency".
        eta_RTE     round-trip: net electricity out over electricity in, both
                    including their pumping parasitics.
        eps_RTE     eta_RTE weighted by the fraction of the store that actually
                    CYCLES, eps(end of charge) - eps(end of discharge).
        dE_therm    thermal energy delivered per well per cycle   [kWh]
        dE_elec     net electric energy delivered per well,
                    dE_therm * eta_T_eff                          [kWh]

    A note on eps_RTE, because it changed meaning and the change matters when
    reading a factorial table. The first-cycle version weighted by eps_m, the
    melted fraction at end of charge. Under the enthalpy formulation that
    SATURATES -- once a segment has melted all its PCM the further energy goes
    into superheat, which eps_m cannot see -- and at this design point it is
    1.0000 exactly, so eps_RTE was identical to eta_RTE and carried no
    information at all. The cycled fraction does not saturate: a store that
    fills completely but only half empties returns 0.5, which is the useful
    statement.

    Note also that eta_RTE is INDEPENDENT of the deviation, exactly. A field
    that delivers 1 % more than target also drew 1 % more in, because the charge
    flow is pinned by its glide in the same way. The deviation is a statement
    about plant size, not about efficiency.
    """
    T, Eb = r["T"], r["budget"]
    N = r["N_wells"]
    t_ch_s, t_dc_s = case.t_ch * 3600.0, case.t_dc * 3600.0

    # ---- what the field actually moved, per cycle ------------------------
    Q_dot_in_ORC = r["Q_discharge_kJ"] / t_dc_s          # kW, field
    Q_dot_out_HP = r["Q_charge_kJ"] / t_ch_s             # kW, field
    rank_eff = Eb["Q_dot_in_ORC"] and (case.W_dot_el_out / case.ElG_eff
                                       / Eb["Q_dot_in_ORC"])
    W_el_out = Q_dot_in_ORC * rank_eff * case.ElG_eff
    cop = Eb["Q_dot_out_HP"] / (Eb["W_dot_el_in"] * case.ElH_eff)
    W_el_in = Q_dot_out_HP / cop / case.ElH_eff

    # ---- parasitics, at the realised temperatures ------------------------
    pumping_ch, pumping_dc = borehole_pumping(case, r)

    # ---- the indicators --------------------------------------------------
    eta_T = W_el_out / Q_dot_in_ORC
    eta_T_eff = eta_T - pumping_dc / Q_dot_in_ORC
    eta_RTE = ((W_el_out - pumping_dc) * case.t_dc
               / ((W_el_in + pumping_ch) * case.t_ch))
    eps_cycled = float(r["charge"]["eps_local"].mean()
                       - r["discharge"]["eps_local"].mean())

    # ---- what BOREAS measures that THUMS could not ----------------------
    # eta_storage was 1.0000 by construction under H6. It is now a number
    # that can move, and the sign of the loss is NOT fixed: deep segments can
    # sit in rock hotter than they are and gain heat, so a negative loss is a
    # result rather than a bug and nothing here takes an absolute value.
    Q_loss = r.get("Q_loss_kJ", 0.0)
    dc_ch, dc_dc = downcomer_leak(case, r)
    f_loss = Q_loss / r["Q_charge_kJ"] if r["Q_charge_kJ"] else float("nan")
    f_bypass = ((dc_ch + dc_dc) / r["Q_charge_kJ"]
                if r["Q_charge_kJ"] else float("nan"))
    dE_therm = r["Q_discharge_kJ"] / 3600.0 / N          # kWh per well
    out = dict(
        eta_T=eta_T, eta_T_eff=eta_T_eff,
        eta_RTE=eta_RTE, eps_RTE=eta_RTE * eps_cycled,
        eps_cycled=eps_cycled,
        eta_storage=r["eta_storage"],
        Q_loss_kJ=Q_loss, f_formation_loss=f_loss,
        Q_bypass_ch_kJ=dc_ch, Q_bypass_dc_kJ=dc_dc, f_bypass=f_bypass,
        R_formation=r.get("R_formation", float("nan")),
        R_ground=r.get("R_ground", float("nan")),
        dE_therm_kWh=dE_therm, dE_elec_kWh=dE_therm * eta_T_eff,
        pumping_ch_kW=pumping_ch, pumping_dc_kW=pumping_dc,
        f_pump=(pumping_ch + pumping_dc) / W_el_out,
        W_el_out_kW=W_el_out, W_el_in_kW=W_el_in,
        rho_E_kWh_m3=dE_therm / case.V_well,
        # ---- POWER density, which is not energy density -----------------
        # rho_E says how much energy a cubic metre of store holds; it says
        # nothing about how fast it can be delivered, and the two are traded
        # against each other by the discharge window. A store with twice the
        # energy density and half the discharge rate is a different machine.
        # Per unit DEPTH as well, because drilling is priced by the metre.
        rho_P_kW_m3=(W_el_out / N) / case.V_well,
        rho_P_kW_per_m=(W_el_out / N) / case.L_well,
        t_discharge_h=case.t_dc)

    # ---- what the efficiencies above cost in hardware --------------------
    # Without these, every approach temperature is a free lunch: tighten
    # DT_pinch_ORC from 10 K to 1 K and eta_RTE rises 11 % with nothing to
    # pay, while the evaporator conductance more than doubles. Reported per
    # unit of delivered electricity as well, which is the form that lets two
    # designs of different size be compared.
    if "rank" in r and "hp" in r:
        ua = exchanger_UA(case, r["rank"], r["hp"], r["T"], r["budget"])
        out["UA_total_kW_K"] = ua["TOTAL"]["UA_kW_per_K"]
        out["UA_per_MWe"] = ua["TOTAL"]["UA_kW_per_K"] / (W_el_out / 1000.0)
        out["pinch_min_K"] = ua["TOTAL"]["pinch_K"]
        for key, tag in (("HTHP evaporator", "hpe"), ("HTHP condenser", "hpc"),
                         ("ORC evaporator", "orce"), ("ORC condenser", "orcc")):
            out[f"UA_{tag}_kW_K"] = ua[key]["UA_kW_per_K"]
            out[f"dTeff_{tag}_K"] = ua[key]["dT_eff_K"]
            # The four pinches individually, not just their minimum. Which
            # exchanger is tightest is design information; `pinch_min_K`
            # throws it away, and a design can move the binding one without
            # the minimum changing at all.
            out[f"pinch_{tag}_K"] = ua[key]["pinch_K"]

        # ---- second law, and the resource the first law books as free ----
        # eta_RTE treats the 60 C geothermal stream as costless. It is not:
        # a dedicated producer is drilled and paid for whether or not its
        # exergy is used. psi prices it, and comes out well BELOW eta_RTE.
        # The well count is the number the operator actually cares about,
        # and until v0.13 the model reported only half of it.
        ex = exergy_audit(case, r)
        out["psi"] = ex["psi"]
        out["psi_stripped"] = ex["psi_stripped"]
        out["I_total_kWh"] = ex["total_kWh"]
        out["Ex_reinject_kWh"] = ex["Ex_reinject_kWh"]
        out["resource_utilisation"] = ex["geo"]["utilisation"]
        out["m_geo_kg_s"] = ex["geo"]["m_geo"]
        out["N_geo"] = ex["geo"]["N_geo"]
        out["T_reinject_C"] = ex["geo"]["T_reinject_C"]
        N_st = r.get("N_wells")
        if N_st is not None:
            out["N_wells_total"] = N_st + ex["geo"]["N_geo"]
            out["wells_per_MWe"] = ((N_st + ex["geo"]["N_geo"])
                                    / (W_el_out / 1000.0))
        for name, v in ex["components_kWh"].items():
            out["I_" + name.replace(" ", "_").replace("-", "")] = v
        out["exergy_gates_pass"] = ex["gates_pass"]

        # ---- per-well productivity, and one scalar for the whole audit ----
        # rho_P says power per cubic metre of store; this says power per WELL,
        # which is the number an operator compares against a drilling cost.
        if N_st is not None:
            out["kW_per_well"] = W_el_out / N_st
        out["I_per_MWh"] = ex["total_kWh"] / (ex["W_el_out_kWh"] / 1000.0)

    # ---- REGIME LABELS, not performance -------------------------------
    # These do not say whether a design is good. They say whether two runs
    # are the same plant, and a screen that averages across them will report
    # main effects taken over a regime change. A run whose ORC switched from
    # pinch-bound to hot-end-bound is a different question, not a worse
    # answer. Carried in the KPI dict so a factorial or a Morris screen
    # cannot lose them.
    out["deviation"] = r["deviation"]
    out["merge_proximity_max"] = float(max(r["charge"]["merge_proximity_max"],
                                           r["discharge"]["merge_proximity_max"]))
    tb = r["rank"].get("T_3e_binding", "")
    out["T_3e_binding"] = tb
    out["regime_pinch_bound"] = 1.0 if "pinch" in tb else 0.0
    msgs = validate_case(case, verbose=False)
    out["feasible"] = 0.0 if any(k == "error" for k, _ in msgs) else 1.0
    out["n_warnings"] = float(sum(1 for k, _ in msgs if k == "warn"))
    return out


def kpi_report(case, r, k=None):
    """Print the factorial-study indicators."""
    k = k or performance_indices(case, r)
    print("PERFORMANCE INDICATORS   (at cyclic steady state, per cycle)")
    print("=" * 66)
    print(f"  eta_T       discharge thermal -> electric   {k['eta_T']:9.4f}")
    print(f"  eta_T,eff   the same, less discharge pumping{k['eta_T_eff']:9.4f}"
          f"   <- effective")
    print(f"  eta_RTE     round trip, both parasitics     {k['eta_RTE']:9.4f}")
    print(f"  eps_RTE     eta_RTE x cycled fraction       {k['eps_RTE']:9.4f}")
    print()
    print(f"  dE_therm    thermal energy per well      {k['dE_therm_kWh']:12.1f} kWh")
    print(f"  dE_elec     net electric energy per well {k['dE_elec_kWh']:12.1f} kWh")
    print(f"  rho_E       thermal energy density       {k['rho_E_kWh_m3']:12.2f} kWh/m3")
    print(f"  rho_P       electric POWER density      {k['rho_P_kW_m3']:12.4f} kW/m3"
          f"   ({k['t_discharge_h']:.0f} h window)")
    print(f"              per metre of well           {k['rho_P_kW_per_m']:12.4f} kW/m")
    print("      rho_E and rho_P are traded against each other by the")
    print("      discharge window; a store with twice the energy density and")
    print("      half the rate is a different machine, not a better one.")
    print()
    print(f"  cycled fraction of the store   {k['eps_cycled']:8.4f}"
          f"   (melted {r['charge']['eps_local'].mean():.4f},"
          f" residual {r['discharge']['eps_local'].mean():.4f})")
    print(f"  pumping, charge / discharge    {k['pumping_ch_kW']:8.2f}"
          f" / {k['pumping_dc_kW']:.2f} kW"
          f"   = {100*k['f_pump']:.2f} % of gross output")
    if "UA_total_kW_K" in k:
        print()
        print(f"  UA total, all four exchangers  "
              f"{k['UA_total_kW_K']:8.1f} kW/K"
              f"   = {k['UA_per_MWe']:.1f} per MWe")
        print(f"      HTHP evap {k['UA_hpe_kW_K']:7.1f}  "
              f"HTHP cond {k['UA_hpc_kW_K']:7.1f}  "
              f"ORC evap {k['UA_orce_kW_K']:7.1f}  "
              f"ORC cond {k['UA_orcc_kW_K']:7.1f}")
        print(f"  the four pinches, individually (K)")
        print(f"      HTHP evap {k['pinch_hpe_K']:7.2f}  "
              f"HTHP cond {k['pinch_hpc_K']:7.2f}  "
              f"ORC evap {k['pinch_orce_K']:7.2f}  "
              f"ORC cond {k['pinch_orcc_K']:7.2f}")
        print(f"      minimum   {k['pinch_min_K']:7.2f}  <- which exchanger is"
              f" tightest is design information")
        print( "                         that the minimum alone throws away.")
        print("      Read UA beside eta_RTE. Every approach temperature is")
        print("      bought here; `ua_report` breaks it down.")
    print()
    print("  eta_RTE is independent of the deviation: a field that delivers")
    print("  1 % above target also drew 1 % more in. The deviation is a")
    print("  statement about plant SIZE, not about efficiency.")


def simulate_css_corrected(case, N=None, record=False, **kw):
    """CSS with a single ORC correction pass. The v0.6 default path.

    The ORC needs an evaporating temperature before the store has been marched,
    because T_3e is tied to T_2d. Since v0.6 T_2d is a result, so the first pass
    uses the provisional estimate -- the outlet the water would reach if it
    achieved the full glide -- and the second re-evaluates the ORC, the energy
    budget, the well count and both flows at the outlet the first pass actually
    produced.

    ONE pass is enough, and that is a property of the problem rather than a
    hopeful choice: the outlet is pinned by the store, not by the plant. Over a
    sweep of the discharge subcooling from 6 to 15 K -- which moves the inlet by
    9 K and the deviation by twelve percentage points -- the realised outlet
    moves by less than 0.3 K. `dT_2d_pass` is returned so the assumption is
    visible rather than assumed; if it is ever large, iterate.

    This keeps the plant level decoupled from the store in the sense that
    matters for Figure "procedure": there is no outer loop, only a second
    evaluation.
    """
    first = simulate_css(case, N=N, record=False, **kw)
    second = simulate_css(case, N=N, record=record,
                          T_2d=first["T_2d_realised"], **kw)
    second["pass1"] = first
    second["dT_2d_pass"] = second["T_2d_realised"] - first["T_2d_realised"]
    second["corrected"] = True
    return second


def css_report(case, r):
    """Print the cyclic-steady-state result."""
    print("CYCLIC STEADY STATE")
    print("=" * 66)
    if r["lambda_overridden"]:
        print(f"  NOTE  lambda forced to 0, from case.loss_surplus = "
              f"{case.loss_surplus:.3f}. A non-zero loss surplus cannot be")
        print("        absorbed at CSS: the state returns to itself and this")
        print("        model has no loss path, so charge == discharge exactly.")
        print("        The ZERO is what drove this run; the 0.050 in the Case")
        print("        is inert here and affects only energy_budget called on")
        print("        its own.")
        print()
    if r["T"].get("T_3e_hot_end_rule") is not None:
        old = r["T"]["T_3e_hot_end_rule"] - 273.15
        new = r["T"]["T_3e"] - 273.15
        if new < old - 1e-6:
            print(f"  NOTE  ORC boiling temperature set by the EVAPORATOR "
                  f"PINCH, not by DT_2D_3E:")
            print(f"        hot-end rule would give {old:7.2f} C; the pinch "
                  f"allows {new:7.2f} C.")
            print()
    print(f"  N_wells          {r['N_wells']:10.4f}   from latent heat alone, not solved")
    print(f"  m_dot charge     {r['m1_ch']:10.4f} kg/s per leg-pair, pinned by the glide")
    print(f"  m_dot discharge  {r['m1_dc']:10.4f} kg/s per leg-pair, pinned by the glide")
    print(f"  flow ratio       {r['flow_ratio']:10.4f}   a consequence, not a solve")
    print()
    if r["converged"]:
        print(f"  converged in {r['cycles']} cycles"
              f"   (drift {r['history'][-1]['drift']:.1e})")
    else:
        # Loud, because the result is still printed and looks perfectly normal.
        # Slow convergence is not rare: it scales with the well count, and a
        # case with low latent heat or many wells can need three times the
        # default. Re-run with a larger n_cycles before believing anything
        # below this line.
        print(f"  *** NOT CONVERGED *** stopped at the n_cycles limit of"
              f" {r['cycles']} with drift {r['history'][-1]['drift']:.2e},"
              f" tolerance {r['tol']:.0e}")
        print("      Re-run with a larger n_cycles. Everything below assumes")
        print("      the state has returned to itself, and it has not.")
    print(f"  eta_storage      {r['eta_storage']:10.6f}   must be 1: adiabatic, closed cycle")
    print()
    print(f"  required   D_E_in_ORC  {r['required_kJ']:12.5e} kJ")
    print(f"  delivered  at CSS      {r['Q_discharge_kJ']:12.5e} kJ")
    print(f"  DEVIATION              {100*r['deviation']:+12.3f} %")
    print()
    print(f"  implied net output     {r['W_el_out_implied']/1000:10.4f} MWe"
          f"   (target {case.W_dot_el_out/1000:.4f})")
    print()
    print("  the deviation IS the glide shortfall")
    print(f"    realised discharge glide / assumed   {r['glide_ratio_dc']:10.6f}")
    print(f"    delivered energy / required          "
          f"{r['Q_discharge_kJ']/r['required_kJ']:10.6f}")
    print("    The flow is pinned by the ASSUMED glide, so the energy")
    print("    delivered is just the ratio of realised to assumed glide.")
    print()
    print("  exchanger outlets are RESULTS, not inputs (v0.6)")
    print(f"    discharge, to the ORC       assumed {r['T_2d_assumed']-273.15:8.3f} C"
          f"   realised {r['T_2d_realised']-273.15:8.3f} C")
    print(f"    charge, to the HP condenser assumed {r['T_2c_assumed']-273.15:8.3f} C"
          f"   realised {r['T_2c_realised']-273.15:8.3f} C")
    if r.get("corrected"):
        print(f"    ORC corrected at the realised outlet; second pass moved it"
              f" {r['dT_2d_pass']:+.4f} K")
    elif r["T_2d_was_provisional"]:
        print("    UNCORRECTED: the ORC still runs on the provisional outlet.")
        print("    Use simulate_css_corrected for the reported result.")
    ch, dc = r["charge"], r["discharge"]
    print()
    print("  melted fraction at CSS")
    print(f"    end of charge     mean {ch['eps_local'].mean():.4f}"
          f"   min {ch['eps_local'].min():.4f}   max {ch['eps_local'].max():.4f}")
    print(f"    end of discharge  mean {dc['eps_local'].mean():.4f}"
          f"   min {dc['eps_local'].min():.4f}   max {dc['eps_local'].max():.4f}")
    if dc["eps_local"].mean() > 0.05 and ch["eps_local"].mean() > 0.99:
        print()
        print("    The store melts completely but does not refreeze completely,")
        print("    so the shortfall is a DISCHARGE RATE limit, not an inventory")
        print("    limit. More PCM per well is not the lever.")


def validate_case(case, verbose=True):
    """Check a Case for the inconsistencies that are easy to create.

    This is not a check that the model is right -- that lives in the
    verification notebook. It is a check that YOUR PARAMETERS are consistent,
    and it fires on the traps that have actually caught people:

      * `n_segments` not a multiple of `N_lay`, which makes `layer_map` place
        a layer boundary mid-segment;
      * a discharge inlet that is not below the coldest layer, or a charge
        inlet not above the hottest, so the half-cycle cannot run;
      * a glide so large that the coldest layer falls below the sink, or so
        small that the cascade collapses;
      * a time grid too coarse for the fastest segment time constant.

    Returns a list of (level, message). `level` is 'error' for a case that
    cannot be run and 'warn' for one that can but probably should not be.
    """
    msgs = []
    def err(m): msgs.append(('error', m))
    def warn(m): msgs.append(('warn', m))

    # --- the cascade -----------------------------------------------------
    if case.N_lay < 1:
        err(f'N_lay = {case.N_lay}; must be at least 1')
    if case.n_segments % case.N_lay:
        warn(f'n_segments = {case.n_segments} is not a multiple of '
             f'N_lay = {case.N_lay}: layer_map puts a layer boundary inside a '
             f'segment, so one segment carries the wrong melting temperature '
             f'by up to {case.DT_3C_2C / case.N_lay:.3f} K. '
             f'Nearest clean values: '
             f'{case.N_lay * (case.n_segments // case.N_lay)} or '
             f'{case.N_lay * (case.n_segments // case.N_lay + 1)}')

    T_bot = T_m_bottom(case) - 273.15
    spacing = case.DT_3C_2C / case.N_lay

    # --- the two inlets --------------------------------------------------
    T_in_ch = case.T_m_C + case.DT_4C_M
    T_in_dc = T_bot - case.DT_M_1D
    if case.DT_4C_M <= 0:
        err(f'DT_4C_M = {case.DT_4C_M} K; the charge inlet must be ABOVE the '
            f'hottest layer or nothing melts')
    if case.DT_M_1D <= 0:
        err(f'DT_M_1D = {case.DT_M_1D} K; the discharge inlet must be BELOW '
            f'the coldest layer or nothing freezes')
    if 0 < case.DT_4C_M < spacing:
        warn(f'DT_4C_M = {case.DT_4C_M:.3f} K is less than one layer width '
             f'({spacing:.3f} K), so the charging fluid drops below a layer\'s '
             f'melting point before leaving it')
    if 0 < case.DT_M_1D < spacing:
        warn(f'DT_M_1D = {case.DT_M_1D:.3f} K is less than one layer width '
             f'({spacing:.3f} K); the discharge pinches at each layer exit')

    # --- glide and the sink ----------------------------------------------
    if case.DT_3C_2C <= 0:
        err(f'DT_3C_2C = {case.DT_3C_2C} K; the glide sets both mass flows')
    if T_in_dc <= case.T_sink_C:
        err(f'discharge inlet {T_in_dc:.1f} C is at or below the sink '
            f'{case.T_sink_C:.1f} C')
    if T_in_ch <= T_bot:
        err(f'charge inlet {T_in_ch:.1f} C is below the coldest layer '
            f'{T_bot:.1f} C')

    # --- properties ------------------------------------------------------
    for f in ('h_m', 'cp_s', 'cp_l', 'k_s', 'k_l', 'rho_s', 'rho_l'):
        v = getattr(case, f, None)
        if v is not None and v <= 0:
            err(f'{f} = {v}; must be positive')

    # --- the time grid ---------------------------------------------------
    if case.n_times < 8:
        warn(f'n_times = {case.n_times}; the logarithmic grid needs enough '
             f'levels that the last step does not span most of the window')

    # --- the working fluid is liquid water, and must stay that way --------
    T_sat_w = CP.PropsSI('T', 'P', case.P, 'Q', 0, 'Water') - 273.15
    if T_in_ch >= T_sat_w:
        err(f'the charging inlet T_4c = {T_in_ch:.2f} C is at or above the '
            f'saturation temperature of water at P = {case.P/1e5:.2f} bar, '
            f'which is {T_sat_w:.2f} C. The model assumes single-phase '
            f'liquid throughout (H7) and carries no boiling correlation, so '
            f'it would report nonsense rather than fail. Raise case.P or '
            f'lower T_m_C + DT_4C_M.')
    elif T_sat_w - T_in_ch < 10.0:
        warn(f'the charging inlet is {T_sat_w - T_in_ch:.2f} K below the '
             f'saturation temperature of water at {case.P/1e5:.2f} bar; '
             f'there is very little margin against boiling')

    # --- approaches that are zero by construction -------------------------
    # None of these is an error. Each is a place where the model quietly
    # assumes an infinite exchanger, and a parametric study that leans on it
    # will report an efficiency no hardware can reach.
    for old, new in (('Turb_eff', 'eta_turb_s'), ('Comp_eff', 'eta_comp_s')):
        if getattr(case, old, None) is not None:
            err(f'{old} = {getattr(case, old)} was RETIRED at v0.12. It was '
                f'the isentropic efficiency applied as a multiplier on the '
                f'work AFTER an isentropic cycle; it is now applied inside '
                f'the cycle, to the state points, as {new}. Setting both '
                f'would double-count. Use {new} and leave {old} at None.')
    for f in ('eta_turb_s', 'eta_pump_s', 'eta_comp_s', 'ElG_eff', 'ElH_eff'):
        v = getattr(case, f)
        if not (0.0 < v <= 1.0):
            err(f'{f} = {v}; must be in (0, 1]')
    if getattr(case, 'DT_3A_13H', None) is not None:
        err(f'DT_3A_13H = {case.DT_3A_13H} was RETIRED at v0.9 and is now '
            f'ignored. It used to set the evaporating temperature '
            f'independently of DT_3A_4A, so the approach in the heat-pump '
            f'evaporator was the difference of two free fields and nothing '
            f'checked its sign: the defaults made it exactly zero, and '
            f'reversing them made it negative with no symptom. The '
            f'evaporating temperature is now derived, '
            f'T_13h = T_source_C - DT_3A_4A - DT_pinch_HPE. Set '
            f'DT_pinch_HPE instead, and leave DT_3A_13H at None.')
    # ---- the geothermal resource, new at v0.13 --------------------------
    if case.exergy_convention not in ('resource', 'stripped'):
        err(f"exergy_convention = {case.exergy_convention!r}; must be "
            f"'resource' (charge everything the well lifts, book the "
            f"reinjected stream as a loss) or 'stripped' (charge only what "
            f"the evaporator removes). They give OPPOSITE guidance on "
            f"DT_3A_4A, so the choice is not cosmetic. See DN-23.")
    if case.m_dot_geo_well <= 0:
        err(f'm_dot_geo_well = {case.m_dot_geo_well} kg/s; N_geo is directly '
            f'proportional to it and it must be positive.')
    T_rei = case.T_source_C - case.DT_3A_4A
    if T_rei < case.T_reinject_min_C:
        warn(f'reinjection at {T_rei:.1f} C is below T_reinject_min_C = '
             f'{case.T_reinject_min_C:.1f} C. DT_3A_4A = {case.DT_3A_4A} K '
             f'draws the source down further than the formation constraint '
             f'allows. This is a WARNING and not an error only because the '
             f'default limit is a placeholder: the project has no measured '
             f'value for it yet. The second law pushes DT_3A_4A UP -- it '
             f'cuts m_geo far faster than it cuts COP -- so this bound is '
             f'the one that will bind in an optimisation.')
    if T_rei <= case.T_sink_C:
        err(f'reinjection at {T_rei:.1f} C is at or below the sink '
            f'({case.T_sink_C} C), which is the dead state. The source would '
            f'carry no exergy worth taking and the audit would go negative.')
    if case.DT_pinch_HPE <= 0:
        err(f'DT_pinch_HPE = {case.DT_pinch_HPE} K; the refrigerant must '
            f'evaporate BELOW the source outlet or the evaporator runs heat '
            f'uphill. Zero means infinite area.')
    elif case.DT_pinch_HPE < 2.0:
        warn(f'DT_pinch_HPE = {case.DT_pinch_HPE:.2f} K is a very close '
             f'approach in the heat-pump evaporator; the area needed grows '
             f'roughly as 1/DT as this goes to zero')
    T_evap = case.T_source_C - case.DT_3A_4A - case.DT_pinch_HPE
    if T_evap <= case.T_sink_C:
        err(f'the derived evaporating temperature {T_evap:.2f} C is at or '
            f'below the sink at {case.T_sink_C:.2f} C; there is no lift left '
            f'to speak of. Reduce DT_3A_4A or DT_pinch_HPE.')

    # --- how hard the far end of the well is driven -----------------------
    # NOT an error, and an earlier version of this check wrongly made it one.
    # The margin below is computed on a LINEAR GLIDE against melting
    # temperatures, i.e. it assumes the PCM sits at T_m. Under Formulation C
    # it does not: the store superheats and subcools, so water arriving on
    # the "wrong" side of T_m still exchanges heat, and `conduction_shell`
    # takes the direction from sign(T_j - T_pcm) segment by segment. The
    # model handles a negative margin correctly and reports the consequence
    # through eps and the deviation. DT_M_1D = DT_3C_2C/N_lay -- the retired
    # v0.5 closure, margin exactly zero -- runs and is one of the frozen
    # verification fixtures. See DN-20.
    span = case.cascade_span
    g_ch, g_dc = case.DT_3C_2C, case.glide_dc_spec
    for gl, appr, name, half, far in (
            (g_ch, case.DT_4C_M, 'DT_4C_M', 'charging', 'coldest'),
            (g_dc, case.DT_M_1D, 'DT_M_1D', 'discharging', 'hottest')):
        margin = span - (gl - appr)
        if margin <= 0.0:
            warn(f'under a linear glide the {half} water would reach the '
                 f'{far} layer {-margin:.3f} K on the wrong side of its '
                 f'melting point (span {span:.3f} K, glide {gl:.3f} K, '
                 f'{name} {appr:.3f} K). This is NOT infeasible -- the PCM '
                 f'leaves T_m under Formulation C and the march takes the '
                 f'heat-flow direction per segment -- but that end of the '
                 f'store is being driven by the sensible branch, or briefly '
                 f'in reverse, rather than melting or freezing as intended. '
                 f'Expect a lower cycled fraction. Raise DT_cascade or '
                 f'{name}, or narrow the glide, to restore the margin.')
        elif margin < 2.0:
            warn(f'the {half} margin at the {far} layer is only '
                 f'{margin:.3f} K (span {span:.3f} K against a glide of '
                 f'{gl:.3f} K less {name} = {appr:.3f} K); that end of the '
                 f'store is driven very softly')

    if case.DT_cascade is not None:
        msgs.append(('info',
                     f'cascade prescribed independently: DT_cascade = '
                     f'{case.DT_cascade:.3f} K, span {span:.3f} K, against a '
                     f'charging glide of {g_ch:.3f} K. The leading-face '
                     f'approach is no longer uniform along the well.'))
    if case.DT_3D_2D is not None:
        msgs.append(('info',
                     f'discharging glide prescribed independently: '
                     f'DT_3D_2D = {case.DT_3D_2D:.3f} K against a charging '
                     f'glide of {g_ch:.3f} K'))
    if case.DT_sink_glide > 0.0:
        warn(f'DT_sink_glide = {case.DT_sink_glide:.2f} K. The ORC condenser '
             f'then has an INTERIOR pinch at the desuperheat corner, around '
             f'93 % of its duty, and the model does not check it. It crosses '
             f'near {case.DT_E_sink + 0.4:.1f} K of sink glide.')
    warn('the ORC regenerator has a zero approach by construction '
         '(T_9e = T_7e in double_stage_rankine), so its duty is an upper '
         'bound rather than a design value. This is structural, not a '
         'setting you can change from Case.')

    # --- the ORC evaporator has to obey the second law --------------------
    try:
        rank, _hp, Tst = cycle_state_points(case)
        got = rank.get('pinch')
        old = rank.get('T_3e_uncorrected')
        if old is not None and rank['T_3e'] < old - 1e-6:
            rank_old = double_stage_rankine(case.fluid, Tst['T_1e'], old)
            warn(f'the ORC pinch forced the boiling temperature DOWN, from '
                 f'{old - 273.15:.2f} C to {rank["T_3e"] - 273.15:.2f} C, and '
                 f'with it eta_ORC from {rank_old["rank_eff"]:.4f} to '
                 f'{rank["rank_eff"]:.4f}. The hot-end rule DT_2D_3E = '
                 f'{case.DT_2D_3E:.1f} K is not what sets the evaporating '
                 f'temperature here; the pinch at DT_pinch_ORC = '
                 f'{case.DT_pinch_ORC:.1f} K is. This is the honest cost of a '
                 f'{case.DT_3C_2C:.0f} K water glide against a fluid that '
                 f'boils at one temperature.')
        elif got is not None:
            msgs.append(('info', f'ORC evaporator pinch {got:+.2f} K, '
                                 f'requirement {case.DT_pinch_ORC:.1f} K'))
    except Exception as e:                       # never block on a diagnostic
        warn(f'could not evaluate the ORC pinch: {e}')

    if verbose:
        if not msgs:
            print('validate_case: OK')
            print(f'  cascade   {case.T_m_C:.3f} C down to {T_bot:.3f} C '
                  f'in {case.N_lay} layers of {spacing:.3f} K')
            print(f'  inlets    charge {T_in_ch:.3f} C, '
                  f'discharge {T_in_dc:.3f} C')
        for level, m in msgs:
            print(f'{level.upper():5s} {m}')
    # ======================================================================
    # BOREAS v1.0 additions
    # ======================================================================

    # --- the gradient and the source are INDEPENDENT, and checked ---------
    # An earlier draft derived the gradient from T_source_C by setting the
    # formation temperature at producing depth EQUAL to the wellhead
    # temperature. That is a lower bound written as an equality: the water
    # cools on the way up, so the rock is at least that hot and generally
    # hotter, and the producing wells may be deeper than the retrofits. The
    # deficit is real physics and deriving one from the other throws it away.
    # So both are specified, and only the INEQUALITY is enforced.
    T_at_prod = (case.T_surface_C
                 + case.grad_K_per_km / 1000.0 * case.L_producing_ft * 0.3048)
    if case.T_source_C > T_at_prod + 1e-9:
        err(f'T_source_C = {case.T_source_C:.1f} C but the gradient only '
            f'reaches {T_at_prod:.1f} C at {case.L_producing_ft:.0f} ft; '
            f'the source cannot be hotter than the rock it comes from. '
            f'Deepen L_producing_ft or raise grad_K_per_km.')
    elif case.T_source_C > T_at_prod - 5.0:
        warn(f'T_source_C = {case.T_source_C:.1f} C is within 5 K of the '
             f'formation temperature at producing depth ({T_at_prod:.1f} C), '
             f'which leaves no allowance for cooling up the production string')

    # --- the cascade must run HOT AT THE BOTTOM --------------------------
    # This is the whole point of the downcomer and it is the one thing a
    # careless edit to melting_temperatures() would silently reverse. Hot at
    # the top is the THUMS orientation: it puts the hottest PCM against the
    # coldest shallow rock and the coldest PCM against rock that, at 8000 ft
    # and a Wilmington gradient, is hotter than it is -- so the deep layers
    # never freeze and that capacity is dead. Watched, not commented.
    try:
        T_rock_arr, depth = formation_profile(case, case.n_segments)
        T_m_lay, _ = melting_temperatures(case)
        T_m_seg = layer_map(T_m_lay, case.n_segments, case.N_lay)
        if depth[0] < depth[-1]:
            err('formation_profile is not in charge-frame indexing: '
                'z = 0 must be the BOTTOM of the well')
        if T_m_seg[0] < T_m_seg[-1]:
            err('the cascade runs hot-at-top. BOREAS requires hot-at-bottom '
                'so the cascade aligns with the geothermal gradient; '
                'check melting_temperatures()')
        stuck = float(np.mean(T_m_seg < T_rock_arr))
        if stuck > 0.02:
            warn(f'{100 * stuck:.0f} % of the well sits in rock HOTTER than '
                 f'its melting point and cannot freeze on discharge; '
                 f'that capacity is dead. Shorten the well, raise T_m_C, '
                 f'or lower grad_K_per_km.')
    except Exception as exc:              # pragma: no cover
        warn(f'could not check the cascade orientation: {exc}')

    # --- the geometry has to close ---------------------------------------
    R = case.D_well / 2.0
    a = case.r_e + case.fin_L                     # finned tube outer radius
    b = case.r_id_outer
    # Area is NECESSARY but not sufficient; 0.85 is a generous packing bound
    # for a handful of unequal circles and is used only to catch the gross
    # case. The arrangement is the one in the schematic -- the downcomer
    # OFFSET, not centred -- which packs better than a concentric annulus.
    if np.pi * (b ** 2 + case.n_ft * a ** 2) > 0.85 * np.pi * R ** 2:
        err(f'the downcomer ({2000 * b:.0f} mm OD) and {case.n_ft} finned '
            f'tubes ({2000 * a:.0f} mm over fins) cannot pack into a '
            f'{1000 * case.D_well:.0f} mm hole')
    elif b + 2.0 * a > R + 1e-6:
        warn(f'the downcomer must be run OFF-CENTRE: {2000 * b:.0f} mm OD '
             f'plus two finned tubes is {2000 * (b / 2 + a):.0f} mm against a '
             f'{1000 * R:.0f} mm radius, so a concentric arrangement does not '
             f'close. The schematic arrangement does; centralisers must hold '
             f'the offset.')
    if case.V_well <= 0:
        err('no PCM volume left: the tubes and downcomer fill the hole')
    f_pcm = case.V_well / case.V_borehole
    if f_pcm < 0.15:
        warn(f'PCM is only {100 * f_pcm:.0f} % of the borehole; the hardware '
             f'is crowding out the storage')

    # --- the formation model ---------------------------------------------
    if not np.isfinite(ground_resistance(case)):
        warn(f'the line-source ground resistance is outside its validity at '
             f'k_rock = {case.k_rock} W/m/K, t = {case.t_operation_yr} yr; '
             f'the formation has been treated as ADIABATIC for this run, so '
             f'eta_storage will come back as exactly 1')
    if case.t_operation_yr > 50.0:
        warn('t_operation_yr beyond 50 yr: the line source is being '
             'extrapolated well past any calibration')

    return msgs

In [ ]:
import importlib, sys
for _m in ('boreas',):
    sys.modules.pop(_m, None)     # %%writefile changed the file on disk
import boreas as B
import numpy as np, matplotlib.pyplot as plt, pandas as pd
print(B.__doc__.strip().splitlines()[0])
c = B.Case()
print(f'\ndefault case: {c.n_ft} fin tubes + 1 downcomer, '
      f'{c.L_ft:.0f} ft, T_m = {c.T_m_C:.0f} C')

---
## 2. The hole, in cross-section

`n_ft` finned tubes and one insulated downcomer, sized so its velocity matches
theirs. That sizing is the whole reason this geometry costs nothing in
pumping: the downcomer carries the flow of all `n_ft` tubes through `n_ft`
times the area, so the velocity is the same everywhere and the total path is
still $2L_{well}$ — exactly what the hairpin had.

The downcomer runs **off-centre**, as in the schematic. A concentric
arrangement does not close for three 1¼″ tubes in a 7″ hole; an offset one
packs comfortably. `validate_case` says so rather than leaving it to a comment.

In [ ]:
def pack(case):
    # Place the downcomer tangent to the wall, then fit the fin tubes round it.
    # SOLVED, not eyeballed: a hand-placed sketch had a fin tube overlapping
    # the downcomer by 30 mm, which is the kind of thing a figure will happily
    # show you forever. The constraint is only that no two circles intersect
    # and all sit inside the hole -- and it is checked below the plot, because
    # a schematic nobody verifies is a drawing, not a result.
    R = case.D_well / 2
    a, b = case.r_e + case.fin_L, case.r_id_outer
    d_id, d_ft = R - b, R - a                     # both tangent to the wall
    # smallest angle off the downcomer that clears it
    cosmin = (d_id**2 + d_ft**2 - (a + b)**2) / (2 * d_id * d_ft)
    th_min = np.arccos(np.clip(cosmin, -1, 1))
    span = 2 * np.pi - 2 * th_min
    n = case.n_ft
    if n == 1:
        th = np.array([np.pi])
    else:
        th = th_min + np.linspace(0, span, n)
    return (d_id, 0.0), [(d_ft * np.cos(t), d_ft * np.sin(t)) for t in th], a, b, R


(xid, yid), fts, a, b, R = pack(c)
fig, ax = plt.subplots(figsize=(6, 6))
ax.add_patch(plt.Circle((0, 0), R * 1000, fc='#2ecc71', ec='k', lw=2, zorder=1))
ax.add_patch(plt.Circle((xid * 1000, yid * 1000), b * 1000, fc='w',
                        ec='#1a5276', lw=6, zorder=2))
ax.text(xid * 1000, yid * 1000, 'ID', ha='center', va='center', fontsize=12, zorder=3)
for (x, y) in fts:
    ax.add_patch(plt.Circle((x * 1000, y * 1000), a * 1000, fc='w',
                            ec='#1a5276', lw=5, zorder=2))
    ax.text(x * 1000, y * 1000, 'FT', ha='center', va='center', fontsize=10, zorder=3)
L = R * 1100
ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect('equal'); ax.axis('off')
ax.set_title(f'{c.n_ft} fin tubes + 1 insulated downcomer\n'
             f'{2000*R:.0f} mm hole, PCM = {100*c.V_well/c.V_borehole:.0f} % of it',
             fontsize=12)
plt.show()

# the figure is only worth looking at if it is a real packing
gaps = [np.hypot(x - xid, y - yid) - a - b for (x, y) in fts]
print(f'  smallest downcomer-to-tube gap  {1000*min(gaps):+6.1f} mm')
if len(fts) > 1:
    pp = [np.hypot(fts[i][0]-fts[j][0], fts[i][1]-fts[j][1]) - 2*a
          for i in range(len(fts)) for j in range(i+1, len(fts))]
    print(f'  smallest tube-to-tube gap       {1000*min(pp):+6.1f} mm')
print(f'  fin tube      {2000*c.r_e:.1f} mm OD, {1000*c.fin_L:.1f} mm fins, '
      f'{c.num_fins} of them')
print(f'  downcomer     {2000*c.r_id_bore:.1f} mm bore, {2000*c.r_id_outer:.0f} mm over '
      f'{1000*c.t_ins:.0f} mm insulation (k = {c.k_ins} W/m/K)')
print(f'  r_cell        {1000*c.r_cell:.1f} mm   (melt fronts meet here)')
print(f'  L_tube        {c.L_tube:.0f} m = L_well  <-- not 2x, and that is the point')

---
## 3. The formation

The thermal circuit from the PCM outward is a series chain, and the terms are
wildly unequal:

$$R_{tot} = \underbrace{R_{PCM}}_{\sim 25\%} + \underbrace{R_{casing}}_{0.1\%}
+ \underbrace{R_{cement}}_{5\%} + \underbrace{R_{ground}(t)}_{\sim 70\%}$$

The casing is thermally invisible and the cement is a rounding error. **The
ground resistance is the model**, and unlike the others it *grows with time*
as the thermal front spreads — which is why `t_operation_yr` is a parameter
and why a quoted loss figure is meaningless without the age beside it.

### The gradient is independent, and only checked

`T_source_C` is a **lower bound** on the formation temperature — the water
cools coming up the production string, and the producing wells may be deeper
than the retrofits. An earlier draft derived the gradient from the source
temperature, which writes a bound as an equality and throws away exactly the
physics the deficit represents. Here both are specified and `validate_case`
enforces only the inequality.

In [ ]:
print(f'  R_PCM->casing   {np.log((c.D_well/2)/c.r_e)/(2*np.pi*c.n_ft*0.5*(c.k_s+c.k_l)):8.4f} m K/W')
print(f'  R_cement        {np.log(c.r_bore/c.r_casing_o)/(2*np.pi*c.k_cement):8.4f}')
print(f'  R_ground        {B.ground_resistance(c):8.4f}   <-- at t = {c.t_operation_yr:.0f} yr')
print(f'  R_total         {B.formation_resistance(c):8.4f}')

T_rock, depth = B.formation_profile(c)
T_m_lay, _ = B.melting_temperatures(c)
T_m_seg = B.layer_map(T_m_lay, c.n_segments, c.N_lay)
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(T_m_seg - 273.15, depth, lw=2.5, label='PCM cascade')
ax.plot(T_rock - 273.15, depth, lw=2.5, ls='--', label='undisturbed rock')
ax.fill_betweenx(depth, T_rock - 273.15, T_m_seg - 273.15,
                 where=(T_m_seg > T_rock), alpha=.18, color='#c0392b', label='loses')
ax.fill_betweenx(depth, T_rock - 273.15, T_m_seg - 273.15,
                 where=(T_m_seg <= T_rock), alpha=.3, color='#2471a3', label='GAINS')
ax.invert_yaxis(); ax.set_xlabel('temperature, C'); ax.set_ylabel('depth, m')
ax.set_title(f'cascade hot at the BOTTOM, aligned with the gradient\n'
             f'{c.grad_K_per_km:.0f} K/km from {c.T_surface_C:.0f} C', fontsize=11)
ax.legend(); ax.grid(alpha=.3); plt.show()

### Why hot-at-the-bottom matters

The rock is coldest at the surface and hottest at depth. THUMS ran the cascade
**hot at the top**, so the hottest PCM faced the coldest rock and the coldest
PCM faced rock that — at 8000 ft and a Wilmington gradient — is *hotter than
it is*. Those deep layers never freeze, and that capacity is dead.

The net loss is identical either way (the mean of $T_{PCM}$ is the same), but
the **distribution** is not:

| at 8000 ft, 56.5 K/km | ΔT at top | ΔT at bottom | cannot freeze |
|---|---|---|---|
| hot-top (THUMS) | +132 K | −55 K | **29 %** |
| hot-bottom (BOREAS) | +83 K | −6 K | **7 %** |

This is not a free choice: the downcomer *makes* it hot-at-bottom, because
charge water reaches the bottom before it has given up any heat.

---
## 4. $B(z)$ — the field is a cone, not an array

THUMS wellheads sit on **6 ft centres**, 12 ft between double rows, and over
1200 wells are drilled *directionally* from four islands with drift angles to
84° across 6500 acres. So the spacing is not a number — it is a function of
depth.

The governing quantity is how long neighbours take to merge thermally,
$r^2/\alpha$:

| spacing | merge time |
|---|---|
| 1.83 m (6 ft) | **36 days** |
| 8 m | 1.9 yr |
| 162 m (reservoir) | 779 yr |

The shallow section is **one thermal body within a month**; the deep section is
a field of isolated wells. In the same well.

**And the two effects oppose each other.** Shallow rock is cold, so the driving
difference is largest exactly where shielding is strongest; deep rock is hot,
so it is smallest exactly where the wells stand alone. That partial
cancellation is specific to directionally-drilled island fields — it does not
arise in purpose-drilled BTES, where spacing is uniform by design.

A shielded well's loss is the cluster perimeter flux shared among $N$ wells,

$$R_{perimeter}(z) = \frac{N\sqrt{\pi\alpha t}}{2\pi k\,R_{field}(z)}$$

which carries no driving temperature and so is a true resistance. The two
regimes combine by taking the **larger** resistance — not a blend: a well
cannot leak more than if it stood alone, a field cannot leak more than its
perimeter allows, and whichever binds governs.

In [ ]:
cf = B.Case(n_wells_field=1000.0)
z = np.linspace(0, cf.L_well, 200)
Bz = B.well_spacing(cf, z)
Rf = B.formation_resistance(cf, z)
R_iso = B.formation_resistance(B.Case())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
a1.plot(Bz, z, lw=2.5, color='#1a5276'); a1.invert_yaxis()
a1.set_xlabel('spacing B(z), m'); a1.set_ylabel('depth, m'); a1.grid(alpha=.3)
a1.set_title(f'{cf.B_surface_m:.2f} m at the wellhead, '
             f'{Bz[-1]:.0f} m at {cf.L_well:.0f} m', fontsize=11)
a2.plot(Rf, z, lw=2.5, label='field, B(z)')
a2.axvline(R_iso, color='#c0392b', ls='--', lw=2, label='isolated well')
a2.invert_yaxis(); a2.set_xscale('log')
a2.set_xlabel('formation resistance, m K/W'); a2.set_ylabel('depth, m')
a2.legend(); a2.grid(alpha=.3)
a2.set_title('shielding weakens with depth but never stops binding', fontsize=11)
plt.tight_layout(); plt.show()
print(f'  shielding factor: {Rf[0]/R_iso:6.0f}x at the top, '
      f'{Rf[-1]/R_iso:5.1f}x at the bottom')

---
## 5. Run it

`n_wells_field=None` is the **isolated well** — the conservative bound, and
the v1.0 default. It is not a pessimistic estimate of the real loss; it is a
different configuration, and one nobody would build.

About a minute per case.

In [ ]:
rows = {}
for lab, kw in [('isolated well', {}),
                ('array, 1.83 m', dict(n_wells_field=1000., fan_angle_deg=0.)),
                ('directional fan', dict(n_wells_field=1000.))]:
    cc = B.Case(n_segments=99, **kw)
    r = B.simulate_css_corrected(cc)
    k = B.performance_indices(cc, r)
    rows[lab] = {'loss %': 100 * k['f_formation_loss'],
                 'eta_storage': r['eta_storage'],
                 'eta_RTE': k['eta_RTE'],
                 'eps_cycled': k['eps_cycled'],
                 'wells/MWe': k['wells_per_MWe']}
    print(f'  {lab} done')
display(pd.DataFrame(rows).T.round(4))

Two orders of magnitude in the loss, from one geometric parameter.

**This also settles the insulation question.** For an isolated well there is a
shallow optimum near 5 mm of casing insulation, worth about +19 % on delivered
energy before the PCM-volume penalty turns the curve over. In a field the loss
is already under 1 %, so insulation trades roughly 11 % of the PCM for
nothing.

That an optimum exists *at all* is a symptom that the isolated-well scope is
unphysical: a model whose advice is "insulate" is reporting on a configuration
that does not exist.

---
## 6. The gates, watched to fail

A check that has never been seen to fail is not evidence. Both of these were
corrupted deliberately, and the second exists *because* the first was silent.

**The closure cannot see a flipped loss sign.** Negating `q_loss` leaves the
residual at 5e-15 — `q_prime` and `Q_loss_cum` both carry the error and it
cancels algebraically. The closure reads only its own inputs.

So the loss term carries a **Clausius gate**: accumulate
$q_{loss}\,(T_{PCM}-T_{rock})$, which is $K(\Delta T)^2$ under the correct
convention and $-K(\Delta T)^2$ under a flipped one.

In [ ]:
src = open('boreas.py').read()
r_ok = B.simulate_css_corrected(B.Case(n_segments=99))
print(f'{"(uncorrupted)":<28} closure {r_ok["charge"]["closure"]:.2e}   '
      f'clausius {r_ok["charge"]["loss_clausius"]:+.3e}')
for lab, bad in [
        ('drop q_loss from q_prime',
         src.replace('q_prime[i] = q_tot + q_loss', 'q_prime[i] = q_tot')),
        ('flip the loss sign',
         src.replace('q_loss = K_l * (T_eff - T_rock_arr[i])',
                     'q_loss = K_l * (T_rock_arr[i] - T_eff)'))]:
    assert bad != src, lab
    ns = {}
    exec(compile(bad, 'corrupt', 'exec'), ns)
    rb = ns['simulate_css_corrected'](ns['Case'](n_segments=99))
    cl, cz = rb['charge']['closure'], rb['charge']['loss_clausius']
    print(f'{lab:<28} closure {cl:.2e} {"FIRES" if cl > 1e-6 else "silent":>6}   '
          f'clausius {cz:+.3e} {"FIRES" if cz < 0 else "silent"}')

### The adiabatic limit

Driving `k_rock` to zero must recover THUMS hypothesis H6 exactly: no loss,
$\eta_{storage} = 1$.

This gate found a real bug on its first run. `ground_resistance` returned
`0.0` when the line source fell outside its validity, meaning *"the front has
not cleared the hole"*. But zero **resistance** is infinite **conductance** —
it claims the rock is a perfect heat sink. The test returned
$\eta_{storage}=0.20$ and a loss **larger** than the physical case.

In [ ]:
r0 = B.simulate_css_corrected(B.Case(n_segments=99, k_rock=1e-9))
print(f'  k_rock -> 0 : Q_loss {r0["Q_loss_kJ"]:.3e} kJ, '
      f'eta_storage {r0["eta_storage"]:.9f}')
print(f'  {"PASS" if r0["Q_loss_kJ"] == 0.0 else "FAIL"}'
      f'   (residual is the CSS drift tolerance, not a loss)')

---
## 7. Change something

Every parameter is a field of `Case`. The ones BOREAS adds:

| | |
|---|---|
| `n_ft` | fin tubes per hole (the downcomer is always one more) |
| `t_ins`, `k_ins` | downcomer insulation |
| `T_surface_C`, `grad_K_per_km` | the geothermal gradient |
| `k_rock`, `rho_cp_rock`, `k_cement` | formation and annulus |
| `t_operation_yr` | how old the field is when you evaluate it |
| `n_wells_field` | `None` = isolated; a number = shielded |
| `B_surface_m`, `z_kickoff_m`, `fan_angle_deg` | the directional fan |

Always run `validate_case` first — it is fast, and it catches the gradient
contradicting the source temperature, a cascade that has been flipped
hot-to-top, a geometry that will not pack, and a well whose deep layers cannot
freeze.

In [ ]:
mine = B.Case(
    n_segments=99,
    n_ft=5,                    # more, smaller tubes -> more PCM, more pumping
    grad_K_per_km=56.5,        # central Wilmington, 1949 survey
    t_operation_yr=10.0,
    n_wells_field=1000.0,
)
for lvl, m in B.validate_case(mine, verbose=False):
    print(f'  [{lvl}] {m}')

r = B.simulate_css_corrected(mine)
B.css_report(mine, r)
B.kpi_report(mine, r)

---
### What this model will not tell you

- **Anything about the surface machines.** They are frozen here. Round-trip
  efficiency is set by the heat-pump *lift* — $(T_h - T_0)/(T_h - T_{src})$ —
  and so by the geothermal source temperature, not by anything below ground.
- **A multi-year loss curve.** `t_operation_yr` evaluates the ground
  resistance at one age. The decay from year 1 to year 30 comes from running
  it several times, not from an outer loop.
- **An optimum.** This is a model, not a search.